# 3강 실습 · Jev를 닮은 오픈 모델, 판단 한 번 뜯어보기

**Jev**(TypeSafe)는 글을 쓰지 않는 AI예요. *상황(state) + 질문 + 보기*를 주면 문장 대신 **보기마다 확률**을 돌려줘요.
3강에서는 이걸 따라 만든 오픈소스들을 봤죠. 이 노트북에서는 그중 **SemIf**(구 openjev · Qwen3.5-4B)를 무료 Colab T4에서 직접 돌리고, 판단 한 번이 어떤 계산으로 나오는지 한 단계씩 열어 봐요.

> Jev 내부 방식은 공개되지 않았어요. 여기서 뜯어보는 건 **Jev를 흉내 낸 오픈 모델(SemIf)** 의 방식이에요.

### 목차와 시간
| 장 | 하는 일 | 기본 실행 | T4 예상 시간 |
|---|---|:---:|---:|
| 0 | 준비: 도우미·미리 잰 숫자 불러오기 | ✅ | 수 초 |
| 1 | 원리 한눈에 — 영상 ① | 영상은 이미 들어 있어요 | – |
| 2 | GPU 확인 · SemIf 설치 · `hf download`로 모델 받기 | ✅ | 3~7분 (약 9.3GB) |
| 3 | 모델 올리기 (T4는 float16) | ✅ | 1~2분 |
| 4 | 한 케이스 뜯어보기 — 영상 ③ + 7단계 위젯 | ✅ | 20~60초 |
| 5 | 강의 5케이스 판정 + 강사 A4000과 비교 | ✅ | 수 초 |
| 6 | 왜 빠른가 — 영상 ② + 미리 잰 속도 + 가벼운 실측 | ✅ | 10~30초 |
| 7 | 정확도와 보정 — API vs 로컬 40건 (미리 잰 값) | ✅ | 즉시 |
| 8 | 내 질문 만들어 보기 | ✅ | 수 초 |
| 9 | (선택) TypeSafe Jev API와 나란히 | 키가 있을 때만 | 수 초 |
| 10 | (선택) Jev-Omni — 80GB GPU 전용 | 파일 목록·검증 기록만 | 수 초 |
| 11 | 정리 | | |

T4 시간은 **예상치**예요(강사 Mac에서 같은 코드를 돌린 시간과 Colab 다운로드 속도로 어림했어요). 대부분은 2장의 모델 다운로드예요.

### 무거운 실험은 미리 잰 결과로 봐요
예전 노트북의 마지막 예제(14B 모델 비교 + llama.cpp를 CUDA로 직접 빌드)는 T4에서 한참 걸렸어요. 이 노트북은 그때 **이미 잰 결과(JSON)** 를 불러와 그림만 그려요. 직접 다시 재고 싶은 분을 위한 안내는 6장 끝에 있어요.

### 시작하기
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** (무료 등급으로 충분해요)
2. 위에서부터 차례로 실행해요(**런타임 → 모두 실행**도 돼요).
3. GPU가 없으면 자동으로 **'미리 잰 숫자'(precomputed) 모드**가 돼요. 모델은 건너뛰고 그림만 그려서 1분 안에 끝나요.

> 🎬 **영상 3개는 셀 출력으로 미리 넣어 뒀어요.** 실행하지 않아도 보여요. 영상 셀을 다시 실행하면(모두 실행 포함) 공개 저장소 [jkf87/jev-handson](https://github.com/jkf87/jev-handson)에서 영상을 받아 와서 다시 보여 줘요. 인터넷이 안 되면 안내 문구가 나오는데, 노트북을 새로 열면 저장된 영상이 다시 보여요.

## 0. 준비: 도우미 불러오기

아래 세 셀(①②③)은 이 노트북이 쓰는 **도구와 숫자**를 메모리에 올려요. GPU도 인터넷도 필요 없고 몇 초면 끝나요. Colab에서는 코드가 접혀 있어요(펼쳐 읽어도 돼요).

- **① 그림 도우미**: 영상 다시 보기, 7단계 위젯, 차트 함수. 차트는 인터넷(CDN) 없이 브라우저에서 바로 그려지는 SVG예요.
- **② 판정 뜯어보기 도우미**: SemIf 원본 함수 `score()`로 판정하면서, 위젯에 보여 줄 중간값(토큰·층별 점수·보기 순서 바꾸기)을 함께 모아요.
- **③ 미리 잰 숫자**: 강사 A4000·Mac, 예전 Colab T4(14B) 실험에서 이미 잰 결과예요. 묶음마다 `source`에 원본 파일 경로가 있어요.

먼저 **① 그림 도우미**예요. 위젯 HTML 원본(`WIDGET_TEMPLATE`)과 차트 함수가 들어 있어요.

In [ ]:
#@title 🔧 ① 그림 도우미 (영상 · 위젯 · 차트)
# 그림 도우미: 인터넷(CDN) 없이 브라우저에서 바로 그려지는 SVG/HTML만 써요. 한글은 브라우저 글꼴로 나와요.
import base64
import html
import json
import math
import uuid
from pathlib import Path

from IPython.display import HTML, display

INK, GREEN, ORANGE, LIGHT, LINE = "#19272D", "#466A62", "#AA5238", "#F0F5F2", "#D6DED9"
MUTED, FAINT, GRAY = "#5E6E68", "#8A9691", "#B7C2BD"
FONT = "'Noto Sans KR','Noto Sans CJK KR','Apple SD Gothic Neo','Malgun Gothic','Nanum Gothic',system-ui,sans-serif"  # HTML style="..." 안에 들어가니 작은따옴표만 써요
QWEN_VOCAB = 248320  # Qwen/Qwen3.5-4B @851bf6e config.json의 vocab_size
CASE_TITLES = {
    "route-gpu": "GPU 서버 요청 → 어느 세션?",
    "route-vague": "목적지 없는 요청 → 어느 세션?",
    "classify-fail": "로그 없는 실패 → 원인은?",
    "route-mail": "메일 확인 요청 → 어느 세션?",
    "gate-danger": "요약만 부탁했는데 전체 삭제? → 진행?",
    "my-test": "내 질문: 환불 메일 → 어느 팀?",
}
PRESET_KO = {
    "code_security": "코드 보안 · 28필드",
    "fintech_fraud": "결제 사기 · 28필드",
    "support_triage": "고객 문의 분류 · 28필드",
    "high_cardinality_255": "관세 분류(후보 255개) · 4필드",
}
ENGINE_KO = {
    "Jev API (TypeSafe)": "Jev API (클라우드)",
    "decider-2b (A4000 CUDA)": "decider-2b · A4000",
    "decider-2b (Mac MPS)": "decider-2b · Mac",
    "OpenJev Qwen3.5-4B (A4000)": "OpenJev(SemIf) 4B · A4000",
    "Qwen3.5-4B 생성(JSON) (Mac Ollama)": "Qwen3.5-4B JSON 생성 · Mac",
}


def esc(s):
    return html.escape(str(s))


def _card(inner, title=None, sub=None, note=None, table=None, width=760):
    head = ""
    if title:
        head += f'<div style="font-size:16px;font-weight:700;margin-bottom:2px">{esc(title)}</div>'
    if sub:
        head += f'<div style="font-size:12.5px;color:{MUTED};margin-bottom:8px">{sub}</div>'
    foot = f'<div style="font-size:12.5px;color:{MUTED};margin-top:8px;line-height:1.6">{note}</div>' if note else ""
    tbl = ""
    if table:
        tbl = (f'<details style="margin-top:8px;font-size:12.5px"><summary style="cursor:pointer;color:{MUTED}">숫자 표로 보기</summary>'
               f'{table}</details>')
    return (f'<div style="font-family:{FONT};color:{INK};background:#fff;border:1px solid {LINE};border-radius:12px;'
            f'padding:14px 16px;max-width:{width}px;box-sizing:border-box;line-height:1.5">{head}{inner}{foot}{tbl}</div>')


def _table(headers, rows, num_cols=()):
    th = "".join(f'<th style="text-align:{"right" if i in num_cols else "left"};border-bottom:1px solid {LINE};padding:4px 8px">{esc(h)}</th>'
                 for i, h in enumerate(headers))
    body = ""
    for r in rows:
        body += "<tr>" + "".join(
            f'<td style="text-align:{"right" if i in num_cols else "left"};border-bottom:1px solid {LINE};padding:3px 8px;font-variant-numeric:tabular-nums">{esc(v)}</td>'
            for i, v in enumerate(r)) + "</tr>"
    return f'<table style="border-collapse:collapse;margin-top:6px">{"<tr>" + th + "</tr>"}{body}</table>'


def _legend(items):
    parts = []
    for color, label, shape in items:
        if shape == "x":
            sw = f'<span style="color:{color};font-weight:700;margin-right:4px">✕</span>'
        elif shape == "line":
            sw = f'<span style="display:inline-block;width:14px;height:2px;background:{color};vertical-align:3px;margin-right:4px"></span>'
        else:
            sw = f'<span style="display:inline-block;width:11px;height:11px;border-radius:{50 if shape == "dot" else 3}%;background:{color};margin-right:4px;vertical-align:-1px"></span>'
        parts.append(f'<span style="margin-right:14px">{sw}{esc(label)}</span>')
    return f'<div style="font-size:12.5px;color:{MUTED};margin:2px 0 6px">{"".join(parts)}</div>'


def _svg(w, h, body, label):
    return (f'<svg viewBox="0 0 {w} {h}" width="100%" style="max-width:{w}px;display:block" role="img" '
            f'aria-label="{esc(label)}" xmlns="http://www.w3.org/2000/svg" font-family="{FONT}">{body}</svg>')


def _bar(x, y, w, h, color, title=None):
    """끝만 4px 둥근 막대(바닥 쪽은 각지게)."""
    w = max(w, 0)
    r = min(4, w / 2, h / 2)
    if w < 5:
        d = f"M{x:.1f},{y:.1f} h{w:.1f} v{h:.1f} h{-w:.1f} z"
    else:
        d = (f"M{x:.1f},{y:.1f} h{w - r:.1f} a{r},{r} 0 0 1 {r},{r} v{h - 2 * r:.1f} "
             f"a{r},{r} 0 0 1 {-r},{r} h{-(w - r):.1f} z")
    t = f"<title>{esc(title)}</title>" if title else ""
    return f'<path d="{d}" fill="{color}">{t}</path>'


def _text(x, y, s, size=12, color=INK, anchor="start", weight=400):
    return (f'<text x="{x:.1f}" y="{y:.1f}" font-size="{size}" fill="{color}" text-anchor="{anchor}" '
            f'font-weight="{weight}">{esc(s)}</text>')


# ─────────────────────────── 영상 ───────────────────────────
CLIP_DIRS = [Path("assets"), Path("colab/assets"), Path("/content/assets")]
# 셀을 다시 실행했는데 옆에 assets/가 없으면(Colab에서 노트북만 연 경우) 공개 저장소에서 받아 와요
CLIP_REPO_RAW = "https://raw.githubusercontent.com/jkf87/jev-handson/main/"
CLIP_REPO_DIR = "03_유사프로젝트/colab/assets/"


def _clip_bytes(name):
    for d in CLIP_DIRS:
        p = d / name
        if p.exists():
            return p.read_bytes()
    try:
        import urllib.parse
        import urllib.request
        req = urllib.request.Request(CLIP_REPO_RAW + urllib.parse.quote(CLIP_REPO_DIR + name),
                                     headers={"User-Agent": "jev-lecture-colab/1.0"})
        with urllib.request.urlopen(req, timeout=20) as r:
            data = r.read()
        if data[4:8] == b"ftyp":   # MP4가 맞는지(파일 머리 확인)
            return data
    except Exception:
        pass
    return None


def show_clip(name, caption=""):
    """영상 다시 보기: 옆의 assets/ → 공개 저장소(jkf87/jev-handson) 순서로 찾아요. 둘 다 없으면 안내만 해요."""
    data = _clip_bytes(name)
    if data:
        b64 = base64.b64encode(data).decode()
        display(HTML(f'<figure style="margin:0;max-width:960px"><video controls playsinline preload="metadata" '
                     f'style="width:100%;border:1px solid {LINE};border-radius:10px" src="data:video/mp4;base64,{b64}"></video>'
                     f'<figcaption style="font-family:{FONT};font-size:12.5px;color:{MUTED}">{esc(caption)}</figcaption></figure>'))
        return
    display(HTML(f'<div style="font-family:{FONT};font-size:13px;color:{MUTED};border:1px dashed {LINE};border-radius:8px;padding:10px 12px;max-width:760px">'
                 f'영상 <code>{esc(name)}</code>을 찾지 못했어요(옆에 assets/ 폴더가 없고, 공개 저장소에서도 받지 못했어요). '
                 f'노트북을 새로 열면 저장된 출력으로 다시 보여요. 영상 파일은 강의 자료의 <code>03_유사프로젝트/colab/assets/</code>에 있어요.</div>'))


# ─────────────────────────── 위젯 ───────────────────────────
WIDGET_TEMPLATE = None  # 노트북에서는 build_notebook.py가 위젯 HTML 원본을 여기에 채워 넣어요


def _case_for_widget(r, ref=None, ref_label=None):
    title = CASE_TITLES.get(r["id"], r["id"])
    if r["id"] not in CASE_TITLES or r["id"] == "my-test":  # 내 질문은 질문 글에서 제목을 만들어요
        q = r["row"]["question"]
        title = "내 질문: " + (q if len(q) <= 22 else q[:21] + "…")
    c = {
        "id": r["id"], "title": title, "row": r["row"], "prompt": r["prompt"],
        "tokens": r["tokens"], "slots": r["slots"], "option_ids": r["option_ids"],
        "logits": [round(v, 4) for v in r["option_logits"]],
        "top": [[t[0], t[1], t[2]] for t in r["top_tokens"]], "mass": r["allowed_mass"],
        "lens": r["lens"], "layer_types": r["layer_types"],
        "perms": [{"order": p["order"], "logits": p["logits"]} for p in r.get("perms", [])],
        "ms": r["forward_seconds"] * 1000, "sha": r["prompt_sha256"],
    }
    if ref:
        c["ref"] = {"label": ref_label, "probs": dict(zip(ref["option_ids"], ref["probabilities"])), "sha": ref.get("prompt_sha256")}
    return c


def show_explainer(results, source_label, device_label, refs=None, ref_label="A4000 기록", default_case="route-vague"):
    """explain_case() 결과 목록을 받아 7단계 위젯을 그려요. refs = {케이스 id: A4000 기록}."""
    display(HTML(explainer_html(results, source_label, device_label, refs, ref_label, default_case)))


def explainer_html(results, source_label, device_label, refs=None, ref_label="A4000 기록", default_case="route-vague"):
    refs = refs or {}
    cases = [_case_for_widget(r, refs.get(r["id"]), ref_label) for r in results]
    data = {"cases": cases, "roles": ROLE_NAMES_KO, "sourceLabel": source_label, "deviceLabel": device_label,
            "defaultCase": default_case if any(c["id"] == default_case for c in cases) else cases[0]["id"],
            "vocabSize": QWEN_VOCAB}
    payload = json.dumps(data, ensure_ascii=False).replace("<", "\\u003c").replace("\u2028", "\\u2028").replace("\u2029", "\\u2029")
    static = "자바스크립트가 꺼진 화면이에요. 요약: " + " · ".join(
        f"{c['id']} → {c['option_ids'][max(range(len(c['logits'])), key=c['logits'].__getitem__)]}" for c in cases)
    uid = "jx" + uuid.uuid4().hex[:10]
    return WIDGET_TEMPLATE.replace("__UID__", uid).replace("__STATIC__", esc(static)).replace("__DATA__", payload)


ROLE_NAMES_KO = {
    "template": "채팅 틀(특수 토큰)", "system": "지시문(system)", "json": "JSON 키·기호", "evidence": "상황(state)",
    "criterion": "질문(question)", "letter": "보기 글자", "option": "보기 설명", "gen": "답이 올 자리",
}


# ─────────────────────────── 5장: 강의 5케이스 ───────────────────────────
def chart_cases(now, ref, now_label, ref_label="A4000 기록 (bfloat16)"):
    """now/ref = {케이스 id: {option_ids, probabilities}}. 보기마다 두 막대(이번 실행 / A4000)."""
    W, lab, plot = 760, 230, 430
    rows_svg, y, table = [], 8, []
    for cid, r in now.items():
        a = ref.get(cid)
        if a is None:
            continue
        pick = r["option_ids"][max(range(len(r["probabilities"])), key=r["probabilities"].__getitem__)]
        apick = a["option_ids"][max(range(len(a["probabilities"])), key=a["probabilities"].__getitem__)]
        mark = "✓ 같은 선택" if pick == apick else "✗ 선택이 달라요"
        rows_svg.append(_text(0, y + 12, f"{cid}", 13, INK, weight=700))
        rows_svg.append(_text(W - 4, y + 12, mark, 12, GREEN if pick == apick else ORANGE, "end", 700))
        y += 20
        amap = dict(zip(a["option_ids"], a["probabilities"]))
        for oid, p in zip(r["option_ids"], r["probabilities"]):
            q = amap.get(oid, 0.0)
            rows_svg.append(_text(lab - 8, y + 13, oid, 12.5, INK, "end", 700 if oid == pick else 400))
            rows_svg.append(_bar(lab, y, plot * p, 9, GREEN, f"{cid} · {oid} · {now_label} {p:.3f}"))
            rows_svg.append(_bar(lab, y + 11, plot * q, 9, GRAY, f"{cid} · {oid} · {ref_label} {q:.3f}"))
            rows_svg.append(_text(lab + plot * max(p, q) + 6, y + 14, f"{p:.3f} / {q:.3f}", 11.5, MUTED))
            table.append([cid, oid, f"{p:.4f}", f"{q:.4f}", f"{p - q:+.4f}"])
            y += 26
        y += 10
    for t in (0, 0.5, 1.0):
        rows_svg.insert(0, f'<line x1="{lab + plot * t}" x2="{lab + plot * t}" y1="0" y2="{y - 6}" stroke="{LINE}" stroke-width="1"/>')
        rows_svg.append(_text(lab + plot * t, y + 8, f"{t:.1f}", 11, MUTED, "middle"))
    svg = _svg(W, y + 14, "".join(rows_svg), "강의 5케이스 보기 확률 비교")
    legend = _legend([(GREEN, now_label, "sq"), (GRAY, ref_label, "sq")])
    display(HTML(_card(legend + svg, "강의 5케이스: 이번 실행 vs 강사 A4000",
                       "막대 = 보기별 확률(합 1). 위 막대가 이번 실행, 아래 막대가 A4000 기록이에요. 오른쪽 숫자 = 이번 / A4000.",
                       table=_table(["케이스", "보기", "이번", "A4000", "차이"], table, (2, 3, 4)))))


# ─────────────────────────── 6장: 속도 ───────────────────────────
def _logx(v, lo, hi, x0, w):
    return x0 + w * (math.log10(v) - math.log10(lo)) / (math.log10(hi) - math.log10(lo))


def chart_parallel(pd):
    """llama.cpp parallel-decision: 같은 GGUF로 Mac·A4000에서 병렬 판단 vs 일반 JSON 생성 (로그 눈금)."""
    W, lab, x0, pw, lo, hi = 760, 230, 240, 440, 50, 20000
    body, y = [], 6
    ticks = [(100, "0.1초"), (300, "0.3초"), (1000, "1초"), (3000, "3초"), (10000, "10초")]
    rows = [r for r in pd["rows"] if r["project"] == "llama"]
    order = ["support_triage", "code_security", "fintech_fraud", "high_cardinality_255"]
    for dev, dev_ko in (("a4000", "RTX A4000 (CUDA)"), ("mac", "Mac M5 (Metal)")):
        body.append(_text(0, y + 12, dev_ko, 13, INK, weight=700))
        y += 20
        for preset in order:
            r = next(x for x in rows if x["device"] == dev and x["preset"] == preset)
            a, b = _logx(r["parallel_ms"], lo, hi, x0, pw), _logx(r["json_ms"], lo, hi, x0, pw)
            cy = y + 10
            body.append(_text(lab - 8, cy + 4, PRESET_KO[preset], 12.5, INK, "end"))
            body.append(f'<line x1="{a:.1f}" x2="{b:.1f}" y1="{cy}" y2="{cy}" stroke="{GRAY}" stroke-width="2"/>')
            body.append(f'<circle cx="{a:.1f}" cy="{cy}" r="5.5" fill="{GREEN}" stroke="#fff" stroke-width="2"><title>병렬 판단 {r["parallel_ms"]:.0f}ms</title></circle>')
            body.append(f'<circle cx="{b:.1f}" cy="{cy}" r="5.5" fill="{ORANGE}" stroke="#fff" stroke-width="2"><title>JSON 생성 {r["json_ms"]:.0f}ms</title></circle>')
            body.append(_text(b + 10, cy + 4, f"{r['speedup']:.0f}배", 12.5, INK, weight=700))
            y += 24
        y += 8
    for v, t in ticks:
        x = _logx(v, lo, hi, x0, pw)
        body.insert(0, f'<line x1="{x:.1f}" x2="{x:.1f}" y1="0" y2="{y}" stroke="{LINE}" stroke-width="1"/>')
        body.append(_text(x, y + 14, t, 11, MUTED, "middle"))
    svg = _svg(W, y + 22, "".join(body), "병렬 판단과 JSON 생성 지연 비교")
    tbl = _table(["장치", "구현", "프리셋", "필드", "병렬(ms)", "JSON(ms)", "배율", "JSON 형식 통과"],
                 [[r["device"], r["project"], r["preset"], r["fields"], f"{r['parallel_ms']:.0f}", f"{r['json_ms']:.0f}",
                   f"{r['speedup']:.1f}×", "통과" if r["json_schema_valid"] else f"실패(빠진 필드 {r['json_missing_fields']}개)"]
                  for r in pd["rows"]], (3, 4, 5, 6))
    display(HTML(_card(_legend([(GREEN, "병렬 판단 (보기 점수 읽기)", "dot"), (ORANGE, "일반 JSON 생성", "dot")]) + svg,
                       "병렬 판단 vs JSON 생성 — 같은 1.5B 모델, 필드 28개",
                       "llama.cpp parallel-decision · Qwen2.5-1.5B GGUF Q4_K_M(두 장치 같은 파일) · 가로축은 로그 눈금 · 첫 호출 뒤 3회 중앙값",
                       note="그림의 llama.cpp 8줄에서 일반 JSON 생성은 <b>8번 모두 형식 검사를 통과하지 못했어요</b>(필드 누락·없는 값). "
                            "표에는 RLCD 줄도 있어요. RLCD는 장치마다 모델 빌드가 달라(Mac MLX 4bit / A4000 PyTorch BF16) 그림에서 뺐고, 배율은 6~79배예요.<br>"
                            "<b>빠르다고 다 맞는 건 아니에요</b> — 같은 A4000, 정답이 있는 작은 문의 8건: " + " · ".join(
                                f"{a['model']} 병렬 판단 {a['parallel_exact']}/{a['cases']} ({a['parallel_ms']:.0f}ms) vs JSON {a['json_exact']}/{a['cases']} ({a['json_ms'] / 1000:.1f}초)"
                                for a in pd.get("a4000_small_accuracy", [])),
                       table=tbl)))


def chart_colab14b(c14):
    """무거운 노트북(14B, Colab T4)에서 이미 잰 6가지 방식."""
    W, lab, plot = 760, 230, 330
    names = {
        "torch_semif_shared": ("SemIf 병렬 · PyTorch", GREEN), "torch_rlcd_parallel": ("RLCD 병렬 · PyTorch", GREEN),
        "torch_json_generate": ("JSON 생성 · PyTorch", ORANGE), "llama14b_parallel": ("병렬 판단 · llama.cpp", GREEN),
        "llama14b_json": ("JSON 생성 · llama.cpp", ORANGE), "llama14b-structured_json": ("형식 강제 JSON · llama.cpp", ORANGE),
    }
    mx = max(r["median_seconds"] for r in c14["rows"])
    body, y = [], 4
    for r in c14["rows"]:
        name, color = names.get(r["key"], (r["method"], GRAY))
        w = plot * r["median_seconds"] / mx
        acc = f"문의 {r['correct_cases']}/{r['total_cases']} · 필드 {r['correct_fields']}/{r['total_fields']}" if r["correct_cases"] is not None else "정확도 미측정"
        body.append(_text(lab - 8, y + 14, name, 12.5, INK, "end"))
        body.append(_bar(lab, y + 3, w, 16, color, f"{name}: 중앙값 {r['median_seconds']:.2f}초 (최소 {r['min_seconds']:.2f} · 최대 {r['max_seconds']:.2f})"))
        body.append(_text(lab + w + 6, y + 15, f"{r['median_seconds']:.1f}초", 12, INK, weight=700))
        body.append(_text(W - 4, y + 15, acc, 12, MUTED, "end"))
        y += 28
    for t in (0, 10, 20, 30, 40):
        x = lab + plot * t / mx
        body.insert(0, f'<line x1="{x:.1f}" x2="{x:.1f}" y1="0" y2="{y}" stroke="{LINE}" stroke-width="1"/>')
        body.append(_text(x, y + 14, f"{t}초", 11, MUTED, "middle"))
    svg = _svg(W, y + 22, "".join(body), "14B 모델 Colab T4 비교")
    tbl = _table(["방식", "엔진", "양자화", "중앙값(초)", "최소", "최대", "반복", "문의 정답", "스케줄"],
                 [[r["method"], r["engine"], r["quantization"], f"{r['median_seconds']:.2f}", f"{r['min_seconds']:.2f}",
                   f"{r['max_seconds']:.2f}", r["measured_repeats"],
                   f"{r['correct_cases']}/{r['total_cases']}" if r["correct_cases"] is not None else "N/A", r["scheduling"]]
                  for r in c14["rows"]], (3, 4, 5, 6))
    display(HTML(_card(_legend([(GREEN, "보기 점수 읽기(판단)", "sq"), (ORANGE, "글로 생성", "sq")]) + svg,
                       f"무거운 실험 결과: {c14['model']} · {c14['gpu']} (이미 잰 값)",
                       "필드 28개를 끝내는 시간(3회 중앙값). PyTorch 줄은 NF4 4비트 + FP16, 질문 2개씩 묶어 매번 문맥을 새로 계산(chunk=2). llama.cpp 줄은 GGUF Q4_K_M.",
                       note="같은 T4인데 <b>llama.cpp 병렬 판단 1.4초 vs PyTorch SemIf 24초</b>: 차이의 대부분은 방식이 아니라 구현(엔진·캐시 재사용·양자화 커널)이에요. "
                            "양자화·프롬프트·캐시 조건이 서로 달라 줄끼리는 '시스템 비교'로만 읽어 주세요. 문의 8/8은 작은 스모크 테스트예요.",
                       table=tbl)))


def chart_shared_demo(d, where):
    """SemIf score() 8번 vs score_shared() 1번: 시간과 처리한 토큰 수."""
    W, lab, plot = 760, 200, 290
    rows = [("따로 8번 (score)", d["direct_seconds"], d["direct_tokens"], ORANGE),
            ("앞부분 공유 (score_shared)", d["shared_seconds"], d["shared_tokens"], GREEN)]
    mx = max(r[1] for r in rows)
    body, y = [], 4
    for name, sec, tok, color in rows:
        w = plot * sec / mx
        body.append(_text(lab - 8, y + 15, name, 12.5, INK, "end"))
        body.append(_bar(lab, y + 4, w, 16, color, f"{name}: {sec:.2f}초 · 토큰 {tok}개"))
        body.append(_text(lab + w + 6, y + 16, f"{sec:.2f}초 · 모델이 읽은 토큰 {tok:,}개", 12, INK))
        y += 30
    svg = _svg(W, y + 4, "".join(body), "공유 모드 비교")
    speed = d["direct_seconds"] / d["shared_seconds"]
    note = (f"상황 앞부분 {d['prefix_tokens']}토큰을 한 번만 계산하니 읽는 토큰이 <b>{d['direct_tokens'] / d['shared_tokens']:.1f}분의 1</b>, "
            f"시간은 <b>{speed:.1f}배</b> 빨라졌어요. 고른 답은 {d['same_picks']}/{d['questions']} 같고, 확률 차이는 최대 {d['max_prob_diff']:.4f}예요"
            f"(캐시를 나눠 쓰면 소수점 아래가 조금 달라질 수 있어요). 실행 방식: <code>{esc(d.get('serving_config'))}</code>")
    tbl = _table(["질문", "따로", "공유"], [[a["id"], json.dumps(a["direct"]), json.dumps(a["shared"])] for a in d["answers"]])
    display(HTML(_card(svg, f"같은 상황 × 질문 {d['questions']}개 — {where}", "SemIf 원본 함수 두 개를 같은 모델로 돌려 잰 값이에요.", note, tbl)))


def chart_semif_a4000(sa):
    """강사 A4000: 같은 가중치로 SemIf 개별(score) · SemIf 병렬(score_shared) · RLCD 병렬 — 모델별 작은 그림 두 개."""
    names = {"semif_direct": ("SemIf 따로(score)", ORANGE), "semif_shared": ("SemIf 공유(score_shared)", GREEN),
             "rlcd": ("RLCD 병렬", GRAY)}
    models = [("qwen9b", "Qwen3.5 9B"), ("qwen1p5b", "Qwen2.5 1.5B")]
    W, lab, plot = 760, 190, 330
    body, y = [], 4
    for key, title in models:
        rows = [r for r in sa["rows"] if r["model"] == key]
        rows.sort(key=lambda r: ["semif_direct", "semif_shared", "rlcd"].index(r["method"]))
        mx = max(r["median_ms"] for r in rows)
        body.append(_text(0, y + 13, f"{title} · 필드 28개", 13, INK, weight=700))
        y += 20
        for r in rows:
            name, color = names[r["method"]]
            w = plot * r["median_ms"] / mx
            body.append(_text(lab - 8, y + 14, name, 12.5, INK, "end"))
            body.append(_bar(lab, y + 3, w, 15, color, f"{title} · {name}: {r['median_ms'] / 1000:.2f}초"))
            body.append(_text(lab + w + 6, y + 15, f"{r['median_ms'] / 1000:.2f}초", 12, INK, weight=700))
            body.append(_text(W - 4, y + 15, f"문의 {r['correct_cases']}/8 · 필드 {r['correct_fields']}/24", 12, MUTED, "end"))
            y += 24
        y += 10
    svg = _svg(W, y, "".join(body), "A4000 SemIf와 RLCD 비교")
    tbl = _table(["모델", "방식", "중앙값(ms)", "최소", "최대", "문의 정답", "필드 정답", "GPU 할당(MiB)"],
                 [[r["model"], r["method"], f"{r['median_ms']:.0f}", f"{r['min_ms']:.0f}", f"{r['max_ms']:.0f}",
                   r["correct_cases"], r["correct_fields"], f"{r['peak_allocated_mib']:.0f}"] for r in sa["rows"]], (2, 3, 4, 5, 6, 7))
    d9 = {r["method"]: r["median_ms"] for r in sa["rows"] if r["model"] == "qwen9b"}
    d1 = {r["method"]: r["median_ms"] for r in sa["rows"] if r["model"] == "qwen1p5b"}
    display(HTML(_card(_legend([(ORANGE, "질문마다 상황부터 다시", "sq"), (GREEN, "상황은 한 번, 질문은 나란히", "sq"), (GRAY, "다른 구현(RLCD)", "sq")]) + svg,
                       "강사 A4000: 같은 가중치에서 SemIf 따로 vs 공유 vs RLCD (이미 잰 값)",
                       "RTX A4000 · NF4/FP16 · 같은 프로세스에 모델 한 번만 올림 · 28항목 3회 중앙값 · 정답은 별도 문의 8건",
                       note=f"공유(score_shared)가 따로(score)보다 9B {d9['semif_direct'] / d9['semif_shared']:.1f}배, 1.5B {d1['semif_direct'] / d1['semif_shared']:.1f}배 빨랐어요 — 6-3에서 잰 것과 같은 효과예요. "
                            "RLCD는 질문별 뒷부분이 짧아 SemIf보다 2배쯤 빨랐지만, 1.5B에서는 문의 5/8로 SemIf(7/8)보다 덜 맞혔어요.",
                       table=tbl)))


# ─────────────────────────── 7장: 정확도와 보정 ───────────────────────────
def chart_api_vs_local(avl):
    t = avl["table"]
    W, lab = 760, 230
    body, y = [], 4
    half = 230
    body.append(_text(lab, y + 10, "행동 일치 (40건)", 12, MUTED, weight=700))
    body.append(_text(lab + half + 40, y + 10, "지연 중앙값 p50 (로그 눈금 · 점)", 12, MUTED, weight=700))
    y += 18
    for r in t:
        name = ENGINE_KO.get(r["engine"], r["engine"])
        is_gen = "생성" in r["engine"]
        body.append(_text(lab - 8, y + 14, name, 12.5, INK, "end"))
        body.append(_bar(lab, y + 4, half * r["actionAcc"], 14, ORANGE if is_gen else GREEN, f"{name}: 행동 일치 {r['actionAcc']:.1%}"))
        body.append(_text(lab + half * r["actionAcc"] + 5, y + 15, f"{r['actionAcc']:.1%}", 12, INK))
        x1 = lab + half + 40
        lx = _logx(r["p50ms"], 100, 10000, x1, 200)
        # 로그 눈금에서는 막대 길이가 값에 비례하지 않아서 점으로 찍어요
        body.append(f'<line x1="{x1:.1f}" x2="{lx:.1f}" y1="{y + 11}" y2="{y + 11}" stroke="{LINE}" stroke-width="2"/>')
        body.append(f'<circle cx="{lx:.1f}" cy="{y + 11}" r="5.5" fill="{ORANGE if is_gen else GREEN}" stroke="#fff" stroke-width="2"><title>{esc(name)}: p50 {r["p50ms"]}ms</title></circle>')
        body.append(_text(lx + 9, y + 15, f"{r['p50ms']:,}ms", 12, INK))
        y += 26
    x1 = lab + half + 40
    for v, s in ((100, "0.1초"), (1000, "1초"), (10000, "10초")):
        x = _logx(v, 100, 10000, x1, 200)
        body.insert(0, f'<line x1="{x:.1f}" x2="{x:.1f}" y1="18" y2="{y}" stroke="{LINE}" stroke-width="1"/>')
        body.append(_text(x, y + 13, s, 11, MUTED, "middle"))
    for v in (0, 0.5, 1):
        x = lab + half * v
        body.insert(0, f'<line x1="{x:.1f}" x2="{x:.1f}" y1="18" y2="{y}" stroke="{LINE}" stroke-width="1"/>')
        body.append(_text(x, y + 13, f"{v:.0%}", 11, MUTED, "middle"))
    svg = _svg(W, y + 20, "".join(body), "API와 로컬 40건 비교")
    tbl = _table(["엔진", "어디서", "행동 일치", "route 일치", "p50(ms)", "p95(ms)", "1,000건 비용($)"],
                 [[r["engine"], r["where"], f"{r['actionAcc']:.1%}", f"{r['routeAcc']:.1%}", r["p50ms"], r["p95ms"], f"{r['costPer1k']:.4f}"] for r in t],
                 (2, 3, 4, 5, 6))
    display(HTML(_card(_legend([(GREEN, "보기 확률을 주는 엔진", "sq"), (ORANGE, "JSON을 생성하는 LLM", "sq")]) + svg,
                       "API vs 로컬 — 같은 40건 · 같은 정책 코드",
                       "2강 라우터 메시지 40건(잡담·작업·애매함·스팸). 모델 답(route·target·confirm)을 같은 정책 코드로 행동으로 바꿔 정답과 비교했어요.",
                       table=tbl)))


def chart_calibration(avl):
    cal = avl["route_calibration"]
    W, lab, x0, pw = 760, 210, 220, 380
    lo = 0.25
    body, y = [], 4
    for name, items in cal.items():
        n = len(items)
        acc = sum(ok for _, ok in items) / n
        conf = sum(p for p, _ in items) / n
        bad_hi = sum(1 for p, ok in items if not ok and p >= 0.9)
        cy = y + 18
        body.append(_text(lab - 8, cy + 4, ENGINE_KO.get(name, name), 12.5, INK, "end"))
        body.append(f'<line x1="{x0}" x2="{x0 + pw}" y1="{cy}" y2="{cy}" stroke="{LINE}" stroke-width="1"/>')
        for k, (p, ok) in enumerate(sorted(items, key=lambda t: t[0])):
            x = x0 + pw * (max(p, lo) - lo) / (1 - lo)
            jy = cy + ((k % 5) - 2) * 4
            tip = f"{name}: 고른 답 확률 {p:.3f} · {'맞음' if ok else '틀림'}"
            if ok:
                body.append(f'<circle cx="{x:.1f}" cy="{jy:.1f}" r="4" fill="{GREEN}" fill-opacity="0.55" stroke="#fff" stroke-width="1"><title>{esc(tip)}</title></circle>')
            else:
                body.append(f'<g stroke="{ORANGE}" stroke-width="2.4" stroke-linecap="round"><title>{esc(tip)}</title>'
                            f'<line x1="{x - 5:.1f}" y1="{jy - 5:.1f}" x2="{x + 5:.1f}" y2="{jy + 5:.1f}"/><line x1="{x - 5:.1f}" y1="{jy + 5:.1f}" x2="{x + 5:.1f}" y2="{jy - 5:.1f}"/></g>')
        cx = x0 + pw * (conf - lo) / (1 - lo)
        ax = x0 + pw * (acc - lo) / (1 - lo)
        body.append(f'<line x1="{cx:.1f}" x2="{cx:.1f}" y1="{cy - 15}" y2="{cy + 15}" stroke="{INK}" stroke-width="2"><title>평균 확신 {conf:.2f}</title></line>')
        body.append(_text(x0 + pw + 12, cy - 2, f"정답률 {acc:.1%} · 평균 확신 {conf:.2f}", 12, INK, weight=700))
        body.append(_text(x0 + pw + 12, cy + 14, f"틀렸는데 0.9 이상: {bad_hi}건", 12, ORANGE if bad_hi else MUTED))
        y += 44
    for v in (0.25, 0.5, 0.75, 1.0):
        x = x0 + pw * (v - lo) / (1 - lo)
        body.append(_text(x, y + 12, f"{v:.2f}", 11, MUTED, "middle"))
    body.append(_text(x0 + pw / 2, y + 28, "고른 답에 모델이 붙인 확률", 11.5, MUTED, "middle"))
    svg = _svg(W + 150, y + 34, "".join(body), "route 질문의 확신과 정답 여부")
    display(HTML(_card(_legend([(GREEN, "맞힌 답", "dot"), (ORANGE, "틀린 답", "x"), (INK, "평균 확신(세로선)", "line")]) + svg,
                       "보정 맛보기: 확률이 높으면 정말 맞았나? (route 질문 40건)",
                       "점 하나가 메시지 하나예요. 가로 위치 = 고른 답의 확률. 잘 보정됐다면 평균 확신(세로선)이 정답률과 비슷해야 해요.",
                       note=avl["route_calibration_note"] + ". 1강의 신뢰도 그림(reliability diagram)·ECE를 제대로 그리려면 수백 건이 필요해요.",
                       width=910)))


def chart_rlcr_rlcd(rr):
    """같은 RLCR 7B 가중치: 추론·답·확신도를 글로 쓰기(RLCR) vs 보기 점수 읽기(RLCD)."""
    W, lab = 760, 250
    rows = [("글로 쓰기 (RLCR 원래 방식)", rr["rlcr"], ORANGE),
            ("점수 읽기 (같은 모델 + RLCD)", rr["rlcd"], GREEN)]
    body, y = [], 4
    body.append(_text(lab, y + 10, "필드 정답 (24개)", 12, MUTED, weight=700))
    body.append(_text(lab + 250, y + 10, "문의 1건 시간 (로그 눈금 · 점)", 12, MUTED, weight=700))
    y += 18
    for name, r, color in rows:
        body.append(_text(lab - 8, y + 15, name, 12, INK, "end"))
        w = 200 * r["field_correct"] / r["field_total"]
        body.append(_bar(lab, y + 4, w, 16, color, f"{name}: {r['field_correct']}/{r['field_total']}"))
        body.append(_text(lab + w + 6, y + 16, f"{r['field_correct']}/{r['field_total']}", 12, INK, weight=700))
        x1 = lab + 250
        lx = _logx(r["median_case_ms"], 100, 30000, x1, 200)
        body.append(f'<line x1="{x1:.1f}" x2="{lx:.1f}" y1="{y + 12}" y2="{y + 12}" stroke="{LINE}" stroke-width="2"/>')
        body.append(f'<circle cx="{lx:.1f}" cy="{y + 12}" r="6" fill="{color}" stroke="#fff" stroke-width="2"><title>{esc(name)}: {r["median_case_ms"] / 1000:.3f}초</title></circle>')
        body.append(_text(lx + 10, y + 16, f"{r['median_case_ms'] / 1000:.2f}초", 12, INK, weight=700))
        y += 30
    for v, t in ((100, "0.1초"), (1000, "1초"), (10000, "10초")):
        x = _logx(v, 100, 30000, lab + 250, 200)
        body.insert(0, f'<line x1="{x:.1f}" x2="{x:.1f}" y1="18" y2="{y}" stroke="{LINE}" stroke-width="1"/>')
        body.append(_text(x, y + 13, t, 11, MUTED, "middle"))
    svg = _svg(W, y + 20, "".join(body), "RLCR과 RLCD 비교")
    wrong = "; ".join(f"{w['case']} {w['field']}: 정답 {w['expected']} → RLCD {w['rlcd_prediction']} <b>{w['rlcd_confidence']:.2f}</b> "
                      f"(RLCR은 {w['rlcr_prediction']}, 확신도 {w['rlcr_confidence']:.1f})" for w in rr["rlcd_wrong"])
    ratio = rr["rlcr"]["median_case_ms"] / rr["rlcd"]["median_case_ms"]
    display(HTML(_card(svg, "같은 가중치, 출력 방식만 다르게: RLCR 생성 vs RLCD 읽기 (강사 A4000, 이미 잰 값)",
                       rr["setup"] + " · 문의 1건 = 필드 3개(category · urgent · priority)",
                       note=f"점수 읽기가 {ratio:.0f}배 빨랐지만 필드 2개를 틀렸어요. 틀린 두 건의 확률이 높았다는 게 중요해요: {wrong}. "
                            "<b>높은 softmax 확률이 정답을 보장하지 않아요.</b> 문의 8건짜리 작은 실험이라 보정 우열을 말하기엔 부족해요.",
                       width=760)))


# ─────────────────────────── 10장: Jev-Omni ───────────────────────────
def omni_files_table(omni):
    gb = lambda b: f"{b / 1e9:.1f}GB" if b >= 1e8 else f"{b / 1e6:.1f}MB"
    labels = {"backbone": "backbone/ (FP32 본체, 13조각)", "unified": "unified/ (BF16 합친 모델)", "head.pt": "head.pt (256칸 판단 머리)",
              "assets": "assets/ (그림·데모 영상)", "기타 작은 파일": "토크나이저·설정·코드"}
    rows = [[labels.get(k, k), gb(v)] for k, v in sorted(omni["files_bytes"].items(), key=lambda kv: -kv[1])]
    rows.append(["저장소 전체 (snapshot_download가 받는 양)", gb(omni["repo_total_bytes"])])
    rows.append(["+ google/gemma-4-12B-it (로더가 추가로 받음)", gb(omni["gemma_total_bytes"])])
    rows.append(["합계", gb(omni["repo_total_bytes"] + omni["gemma_total_bytes"])])
    display(HTML(_card(_table(["파일", "크기"], rows, (1,)), "Jev-Omni가 받는 파일 (메타데이터로만 확인)",
                       f"akhilaaa3/Jev-Omni @ {omni['repo_sha'][:7]} · {omni['fetched_at']} 조회 · 가중치는 받지 않았어요",
                       note="공식 로더는 FP32 본체(약 48GB)를 GPU에 통째로 올린 뒤 BF16으로 바꾸고, Gemma 4 12B(약 24GB)를 또 올려요. "
                            "그래서 <b>80GB GPU(A100 80GB·H100)</b>가 필요해요. 무료 T4(15GB)·L4(24GB)·A100 40GB로는 그대로 안 돌아가요.",
                       width=640)))


def chart_omni_verification(omni):
    cases = omni["verification_cases"]
    die, urn = cases[2], cases[3]

    def bars(case, ideal, title):
        W, H, x0, bw, gap, ph = 440, 190, 34, 34, 14, 130
        body = []
        mx = 0.5
        for v in (0, 0.25, 0.5):
            y = 10 + ph * (1 - v / mx)
            body.append(f'<line x1="{x0}" x2="{W - 8}" y1="{y:.1f}" y2="{y:.1f}" stroke="{LINE}" stroke-width="1"/>')
            body.append(_text(x0 - 5, y + 4, f"{v:.2f}", 10.5, MUTED, "end"))
        for i, (o, p) in enumerate(zip(case["options"], case["probabilities"])):
            x = x0 + 10 + i * (bw + gap)
            h = ph * p / mx
            body.append(f'<path d="M{x},{10 + ph} v{-(h - 4):.1f} a4,4 0 0 1 4,-4 h{bw - 8} a4,4 0 0 1 4,4 v{h - 4:.1f} z" fill="{GREEN}"><title>{esc(o)}: {p:.3f}</title></path>')
            body.append(_text(x + bw / 2, 10 + ph - h - 5, f"{p:.2f}", 11, INK, "middle"))
            body.append(_text(x + bw / 2, 10 + ph + 15, o, 11.5, INK, "middle"))
        iy = 10 + ph * (1 - ideal / mx)
        body.append(f'<line x1="{x0}" x2="{W - 8}" y1="{iy:.1f}" y2="{iy:.1f}" stroke="{ORANGE}" stroke-width="2"/>')
        body.append(_text(W - 8, iy - 5, f"공정하면 {ideal:.3f}", 11, ORANGE, "end", 700))
        return f'<div style="flex:1 1 340px"><div style="font-size:13px;font-weight:700">{esc(title)}</div>{_svg(W, H + 12, "".join(body), title)}</div>'

    tiles = ""
    for c, label in ((cases[0], "10시 회의, 지금 9시 — 시작했나?"), (cases[1], "환불 끝, 고객 '해결됐어요' — 해결됐나?")):
        k = max(range(len(c["probabilities"])), key=c["probabilities"].__getitem__)
        tiles += (f'<div style="background:{LIGHT};border-radius:8px;padding:8px 12px;flex:1 1 200px"><div style="font-size:12px;color:{MUTED}">{esc(label)}</div>'
                  f'<div style="font-size:22px;font-weight:700">{esc(c["options"][k])} {c["probabilities"][k]:.4f}</div></div>')
    inner = (f'<div style="display:flex;flex-wrap:wrap;gap:12px">{bars(die, 1 / 6, "주사위 한 번: 몇이 나올까?")}{bars(urn, 0.25, "구슬 4개 중 하나: 무슨 색?")}</div>'
             f'<div style="display:flex;flex-wrap:wrap;gap:10px;margin-top:8px">{tiles}</div>')
    claims = omni["readme_claims"]
    display(HTML(_card(inner, "Jev-Omni 자체 검증 기록 읽기 (우리가 돌린 게 아니에요)",
                       "unified/verification_unified.json의 reference 확률 · 모델 저장소가 스스로 올린 값",
                       note=f"정답이 있는 질문(회의·환불)은 잘 맞히지만, <b>정답이 없는 질문</b>(공정한 주사위·구슬)에서는 확률이 고르게 퍼지지 않아요 — "
                            f"주사위 '6'에 {die['probabilities'][5]:.2f}, '3'에 {die['probabilities'][2]:.2f}. README가 말하는 ECE {claims['decisionbench_medium_ece']:.3f}"
                            f"(DecisionBench Medium)는 '정답이 있는 문제' 기준이라 이런 경우를 보장하지 않아요. 보정은 내 데이터로 다시 재 봐야 해요.",
                       width=780)))


WIDGET_TEMPLATE = r'''<div id="__UID__" class="jx" tabindex="0" aria-label="Jev 닮은 판단 한 케이스 뜯어보기">
<style>
#__UID__ { --ink:#19272D; --green:#466A62; --orange:#AA5238; --light:#F0F5F2; --line:#D6DED9; --muted:#5E6E68; --faint:#8A9691; --surface:#FFFFFF;
  --r-template:#E4E8E6; --r-system:#F1EEE2; --r-json:transparent; --r-evidence:#D9EAE3; --r-criterion:#F6E0D5; --r-letter:#466A62; --r-option:#E0E7F0; --r-gen:#FBF0C8;
  font-family:"Noto Sans KR","Noto Sans CJK KR","Apple SD Gothic Neo","Malgun Gothic","Nanum Gothic",system-ui,-apple-system,"Segoe UI",sans-serif;
  color:var(--ink); background:var(--surface); border:1px solid var(--line); border-radius:12px; padding:18px 18px 14px; max-width:1060px; box-sizing:border-box; outline:none; font-size:14px; line-height:1.55; }
#__UID__ * { box-sizing:border-box; }
#__UID__ button { font-family:inherit; }
#__UID__ .jx-head { display:flex; flex-wrap:wrap; gap:6px 16px; align-items:baseline; justify-content:space-between; margin-bottom:10px; }
#__UID__ .jx-title { font-size:18px; font-weight:700; }
#__UID__ .jx-title small { font-size:13px; font-weight:400; color:var(--muted); margin-left:8px; }
#__UID__ .jx-src { font-size:12px; color:var(--muted); background:var(--light); border-radius:999px; padding:3px 10px; }
#__UID__ .jx-cases { display:flex; flex-wrap:wrap; gap:6px; margin-bottom:12px; }
#__UID__ .jx-case { border:1px solid var(--line); background:var(--surface); color:var(--ink); border-radius:999px; padding:5px 12px; font-size:13px; cursor:pointer; }
#__UID__ .jx-case[aria-pressed="true"] { background:var(--ink); color:#fff; border-color:var(--ink); }
#__UID__ .jx-case:hover { border-color:var(--green); }
#__UID__ .jx-flow { display:grid; grid-template-columns:repeat(7, minmax(0,1fr)); gap:6px; margin-bottom:12px; }
#__UID__ .jx-node { position:relative; border:1px solid var(--line); border-radius:10px; padding:7px 8px 6px; background:var(--surface); cursor:pointer; text-align:left; min-height:58px; }
#__UID__ .jx-node:not(:last-child)::after { content:"›"; position:absolute; right:-7px; top:50%; transform:translateY(-55%); color:var(--faint); font-size:15px; z-index:1; }
#__UID__ .jx-node .n { font-size:11px; color:var(--muted); }
#__UID__ .jx-node .t { font-size:13px; font-weight:700; display:block; line-height:1.25; }
#__UID__ .jx-node .s { font-size:12px; color:var(--green); display:block; margin-top:2px; white-space:nowrap; overflow:hidden; text-overflow:ellipsis; }
#__UID__ .jx-node[aria-current="step"] { border-color:var(--green); box-shadow:0 0 0 2px var(--green) inset; background:#F6FAF8; }
#__UID__ .jx-node.done .n::after { content:" ✓"; color:var(--green); }
#__UID__ .jx-panel { border:1px solid var(--line); border-radius:10px; padding:14px 16px; min-height:430px; background:var(--surface); }
#__UID__ .jx-panel h4 { margin:0 0 4px; font-size:16px; }
#__UID__ .jx-lead { color:var(--muted); margin:0 0 12px; font-size:13px; }
#__UID__ .jx-grid2 { display:grid; grid-template-columns:1fr 1fr; gap:12px; }
#__UID__ .jx-card { background:var(--light); border-radius:8px; padding:10px 12px; }
#__UID__ .jx-lab { font-size:12px; color:var(--muted); font-weight:700; margin:2px 0 4px; }
#__UID__ .jx-card ul { margin:4px 0 0; padding-left:0; list-style:none; }
#__UID__ .jx-card li { margin:3px 0; }
#__UID__ code, #__UID__ .mono { font-family:"SFMono-Regular",Menlo,Consolas,"D2Coding",monospace; font-size:12.5px; }
#__UID__ .jx-chip { display:inline-block; font-family:"SFMono-Regular",Menlo,Consolas,monospace; font-size:12px; border-radius:5px; padding:0 6px; background:var(--surface); border:1px solid var(--line); }
#__UID__ .jx-note { font-size:12.5px; color:var(--muted); margin-top:10px; }
#__UID__ .jx-note b { color:var(--ink); }
#__UID__ pre.jx-code { background:#F7F9F8; border:1px solid var(--line); border-radius:8px; padding:10px 12px; margin:8px 0 0; overflow-x:auto; font-family:"SFMono-Regular",Menlo,Consolas,monospace; font-size:12.5px; line-height:1.6; white-space:pre; }
#__UID__ pre.jx-code .hl { background:#FBF0C8; display:inline-block; min-width:100%; }
#__UID__ pre.jx-code .cm { color:var(--faint); }
#__UID__ .jx-prompt { font-family:"SFMono-Regular",Menlo,Consolas,monospace; font-size:12.5px; line-height:1.75; white-space:pre-wrap; word-break:break-word; background:#FAFBFA; border:1px solid var(--line); border-radius:8px; padding:10px 12px; max-height:300px; overflow:auto; }
#__UID__ .r-template { background:var(--r-template); color:var(--muted); }
#__UID__ .r-system { background:var(--r-system); }
#__UID__ .r-json { color:var(--faint); }
#__UID__ .r-evidence { background:var(--r-evidence); }
#__UID__ .r-criterion { background:var(--r-criterion); }
#__UID__ .r-letter { background:var(--r-letter); color:#fff; font-weight:700; }
#__UID__ .r-option { background:var(--r-option); }
#__UID__ .r-gen { background:var(--r-gen); }
#__UID__ .jx-legend { display:flex; flex-wrap:wrap; gap:4px 12px; font-size:12px; color:var(--muted); margin:8px 0 0; }
#__UID__ .jx-legend span i { display:inline-block; width:12px; height:12px; border-radius:3px; margin-right:4px; vertical-align:-2px; border:1px solid var(--line); }
#__UID__ .jx-toks { display:flex; flex-wrap:wrap; gap:3px; max-height:250px; overflow:auto; padding:8px; border:1px solid var(--line); border-radius:8px; background:#FAFBFA; }
#__UID__ .jx-tok { font-family:"SFMono-Regular",Menlo,Consolas,monospace; font-size:12px; border-radius:4px; padding:1px 4px; border:1px solid rgba(25,39,45,.08); cursor:default; white-space:pre; }
#__UID__ .jx-tok .ws { color:var(--faint); }
#__UID__ .jx-tok.r-letter .ws { color:#DDE8E3; }
#__UID__ .jx-tok.is-last { outline:2px solid var(--orange); outline-offset:1px; }
#__UID__ .jx-tok.is-hot { outline:2px solid var(--ink); outline-offset:1px; }
#__UID__ .jx-tokinfo { font-size:12.5px; min-height:22px; margin-top:6px; color:var(--ink); }
#__UID__ .jx-stack { display:flex; height:14px; border-radius:4px; overflow:hidden; margin-top:8px; gap:2px; background:var(--surface); }
#__UID__ .jx-stack div { height:100%; }
#__UID__ .jx-row { display:flex; flex-wrap:wrap; gap:14px; align-items:flex-start; }
#__UID__ .jx-row > .grow { flex:1 1 360px; min-width:0; }
#__UID__ .jx-row > .side { flex:0 1 300px; min-width:240px; }
#__UID__ .jx-stat { background:var(--light); border-radius:8px; padding:10px 12px; margin-bottom:10px; }
#__UID__ .jx-stat .v { font-size:26px; font-weight:700; line-height:1.2; }
#__UID__ .jx-stat .l { font-size:12px; color:var(--muted); }
#__UID__ table.jx-tbl { border-collapse:collapse; width:100%; font-size:13px; }
#__UID__ table.jx-tbl th, #__UID__ table.jx-tbl td { border-bottom:1px solid var(--line); padding:4px 6px; text-align:left; font-variant-numeric:tabular-nums; }
#__UID__ table.jx-tbl td.num, #__UID__ table.jx-tbl th.num { text-align:right; }
#__UID__ table.jx-tbl tr.best td { font-weight:700; }
#__UID__ .jx-ctrl { display:flex; flex-wrap:wrap; align-items:center; gap:8px 12px; background:var(--light); border-radius:8px; padding:8px 12px; margin-bottom:10px; }
#__UID__ .jx-ctrl label { font-size:13px; font-weight:700; }
#__UID__ .jx-ctrl input[type=range] { width:220px; accent-color:var(--green); }
#__UID__ .jx-ctrl .val { font-family:"SFMono-Regular",Menlo,Consolas,monospace; font-size:14px; font-weight:700; min-width:64px; }
#__UID__ .jx-btn { border:1px solid var(--line); background:var(--surface); border-radius:6px; padding:3px 10px; font-size:12.5px; cursor:pointer; color:var(--ink); }
#__UID__ .jx-btn:hover { border-color:var(--green); }
#__UID__ .jx-btn[aria-pressed="true"] { background:var(--green); color:#fff; border-color:var(--green); }
#__UID__ .jx-perms { display:flex; flex-wrap:wrap; gap:4px; margin:6px 0 8px; }
#__UID__ .jx-perms .jx-btn { font-family:"SFMono-Regular",Menlo,Consolas,monospace; font-size:11.5px; padding:2px 7px; }
#__UID__ .jx-decision { display:inline-block; font-size:18px; font-weight:700; border-radius:8px; padding:6px 14px; margin:4px 0 8px; }
#__UID__ .jx-decision.act { background:#DCEBE5; color:var(--ink); border:1px solid var(--green); }
#__UID__ .jx-decision.ask { background:#F6E0D5; color:var(--ink); border:1px solid var(--orange); }
#__UID__ .jx-nav { display:flex; justify-content:space-between; align-items:center; margin-top:10px; gap:8px; }
#__UID__ .jx-foot { font-size:12.5px; color:var(--muted); }
#__UID__ .jx-foot b { color:var(--ink); }
#__UID__ svg text { font-family:inherit; }
#__UID__ .jx-tip { position:absolute; pointer-events:none; background:var(--ink); color:#fff; font-size:12px; border-radius:6px; padding:5px 8px; white-space:nowrap; transform:translate(-50%, -110%); opacity:0; transition:opacity .08s; z-index:5; }
#__UID__ .jx-chart { position:relative; }
#__UID__ details { margin-top:8px; font-size:12.5px; }
#__UID__ details summary { cursor:pointer; color:var(--muted); }
#__UID__ .jx-static { font-size:13px; color:var(--muted); }
@media (max-width:760px) {
  #__UID__ .jx-flow { grid-template-columns:repeat(4, minmax(0,1fr)); }
  #__UID__ .jx-grid2 { grid-template-columns:1fr; }
}
</style>
<div class="jx-static">__STATIC__</div>
</div>
<script>
(function () {
  "use strict";
  var root = document.getElementById("__UID__");
  if (!root || root.dataset.ready) return;
  root.dataset.ready = "1";
  var DATA = __DATA__;
  var NS = "http://www.w3.org/2000/svg";
  var C = { ink: "#19272D", green: "#466A62", orange: "#AA5238", light: "#F0F5F2", line: "#D6DED9", muted: "#5E6E68", faint: "#8A9691", gray: "#B7C2BD" };
  var ROLE = DATA.roles;
  var ROLE_ORDER = ["template", "system", "json", "evidence", "criterion", "letter", "option", "gen"];
  var STEPS = [
    { name: "입력", lead: "상황·질문·보기 세 가지가 전부예요." },
    { name: "프롬프트 조립", lead: "SemIf가 세 가지를 채팅 프롬프트 한 장으로 바꿔요." },
    { name: "토큰", lead: "모델은 글자가 아니라 토큰 번호를 읽어요." },
    { name: "모델 1번 통과", lead: "토큰 전체가 32층을 한 번 지나가요. 글을 쓰지 않으니 반복이 없어요." },
    { name: "글자 점수", lead: "마지막 자리에서 '다음 토큰' 점수 중 보기 글자 것만 골라요." },
    { name: "확률", lead: "점수를 softmax로 확률로 바꿔요. 온도 T를 바꿔 보세요." },
    { name: "코드의 판단", lead: "확률을 행동으로 바꾸는 건 모델이 아니라 내 코드예요." }
  ];
  var S = { ci: Math.max(0, DATA.cases.findIndex(function (c) { return c.id === DATA.defaultCase; })), step: 0, T: 1, tau: 0.7, perm: 0, permOn: false };

  // ── 작은 도구들 ──
  function el(tag, attrs) {
    var n = document.createElement(tag);
    if (attrs) for (var k in attrs) {
      if (k === "text") n.textContent = attrs[k];
      else if (k === "html") n.innerHTML = attrs[k];
      else if (k.slice(0, 2) === "on") n.addEventListener(k.slice(2), attrs[k]);
      else if (attrs[k] !== null && attrs[k] !== undefined) n.setAttribute(k, attrs[k]);
    }
    for (var i = 2; i < arguments.length; i++) {
      var ch = arguments[i];
      if (ch === null || ch === undefined) continue;
      n.appendChild(typeof ch === "string" ? document.createTextNode(ch) : ch);
    }
    return n;
  }
  function sv(tag, attrs) {
    var n = document.createElementNS(NS, tag);
    if (attrs) for (var k in attrs) { if (k === "text") n.textContent = attrs[k]; else n.setAttribute(k, attrs[k]); }
    for (var i = 2; i < arguments.length; i++) if (arguments[i]) n.appendChild(arguments[i]);
    return n;
  }
  function esc(s) { return String(s).replace(/&/g, "&amp;").replace(/</g, "&lt;").replace(/>/g, "&gt;"); }
  function softmax(z, T) {
    T = T || 1; var m = -Infinity, i;
    for (i = 0; i < z.length; i++) m = Math.max(m, z[i] / T);
    var e = z.map(function (v) { return Math.exp(v / T - m); }), s = e.reduce(function (a, b) { return a + b; }, 0);
    return e.map(function (v) { return v / s; });
  }
  function argmax(a) { var b = 0; for (var i = 1; i < a.length; i++) if (a[i] > a[b]) b = i; return b; }
  function f2(p) { return p.toFixed(2); }
  function pct(p) {
    if (p >= 0.995) return (p * 100).toFixed(2) + "%";
    if (p >= 0.1) return (p * 100).toFixed(1) + "%";
    if (p >= 0.001) return (p * 100).toFixed(2) + "%";
    if (p >= 0.00001) return (p * 100).toFixed(4) + "%";
    return "<0.001%";
  }
  function showWs(t) { return t.replace(/ /g, "␣").replace(/\n/g, "↵").replace(/\t/g, "⇥"); }
  function tokLabel(t) { var m = t.match(/^(\s*)([\s\S]*?)(\s*)$/); return showWs(m[1]) + m[2] + showWs(m[3]); }
  function letters(c) { return c.slots.map(function (s) { return s[0]; }); }
  function cur() { return DATA.cases[S.ci]; }
  function desc(c, oid) { var o = c.row.options.find(function (x) { return x.id === oid; }); return o ? o.description : ""; }
  function probsNow(c) { return softmax(c.logits, S.T); }
  function decisionOf(c) {
    var p = probsNow(c), b = argmax(p);
    return { best: c.option_ids[b], p: p[b], act: p[b] >= S.tau, probs: p };
  }
  function tipHost(host) {
    var tip = el("div", { class: "jx-tip" });
    host.appendChild(tip);
    return {
      show: function (x, y, html) { tip.innerHTML = html; tip.style.left = x + "px"; tip.style.top = y + "px"; tip.style.opacity = 1; },
      hide: function () { tip.style.opacity = 0; }
    };
  }

  // ── 가로 막대 (값 표시 + 기준선/참고 눈금 선택) ──
  function hbars(rows, opt) {
    opt = opt || {};
    var W = opt.width || 560, lab = opt.labelWidth || 190, rowH = 30, bar = 16, pad = 70;
    var H = rows.length * rowH + (opt.axis ? 22 : 6);
    var max = opt.max || 1, plotW = W - lab - pad;
    var svg = sv("svg", { viewBox: "0 0 " + W + " " + H, width: "100%", role: "img", "aria-label": opt.aria || "막대 그래프", style: "max-width:" + W + "px;display:block" });
    if (opt.axis) {
      [0, 0.25, 0.5, 0.75, 1].forEach(function (t) {
        var x = lab + plotW * t / max;
        svg.appendChild(sv("line", { x1: x, x2: x, y1: 0, y2: H - 18, stroke: C.line, "stroke-width": 1 }));
        svg.appendChild(sv("text", { x: x, y: H - 4, "text-anchor": "middle", "font-size": 11, fill: C.muted, text: t.toFixed(2) }));
      });
    }
    rows.forEach(function (r, i) {
      var y = i * rowH + (rowH - bar) / 2;
      var w = Math.max(plotW * Math.min(r.v, max) / max, r.v > 0 ? 1.5 : 0);
      var g = sv("g");
      g.appendChild(sv("text", { x: lab - 8, y: y + bar / 2 + 4, "text-anchor": "end", "font-size": 12.5, fill: C.ink, "font-weight": r.bold ? 700 : 400, text: r.label }));
      var path = "M" + lab + "," + y + " h" + Math.max(w - 4, 0) + " a4,4 0 0 1 4,4 v" + (bar - 8) + " a4,4 0 0 1 -4,4 h" + (-Math.max(w - 4, 0)) + " z";
      if (w < 5) path = "M" + lab + "," + y + " h" + w + " v" + bar + " h" + (-w) + " z";
      g.appendChild(sv("path", { d: path, fill: r.color || C.gray }));
      g.appendChild(sv("text", { x: lab + w + 6, y: y + bar / 2 + 4, "font-size": 12.5, fill: C.ink, "font-weight": r.bold ? 700 : 400, text: r.valueText || f2(r.v) }));
      if (r.ref !== undefined && r.ref !== null) {
        var rx = lab + plotW * Math.min(r.ref, max) / max;
        g.appendChild(sv("line", { x1: rx, x2: rx, y1: y - 4, y2: y + bar + 4, stroke: C.ink, "stroke-width": 2 }));
      }
      g.appendChild(sv("title", { text: r.title || (r.label + ": " + (r.valueText || f2(r.v))) }));
      svg.appendChild(g);
    });
    if (opt.threshold !== undefined) {
      var tx = lab + plotW * opt.threshold / max;
      svg.appendChild(sv("line", { x1: tx, x2: tx, y1: 0, y2: rows.length * rowH, stroke: C.orange, "stroke-width": 2 }));
      svg.appendChild(sv("text", { x: tx + 4, y: 11, "font-size": 11, fill: C.orange, text: "기준 " + f2(opt.threshold) }));
    }
    return svg;
  }

  // ── 흐름(7단계) ──
  function nodeSub(i, c) {
    var L = letters(c), d = decisionOf(c), b = argmax(c.logits);
    switch (i) {
      case 0: return "보기 " + c.option_ids.length + "개";
      case 1: return "글자 A~" + L[L.length - 1] + " 붙이기";
      case 2: return c.tokens.length + "토큰";
      case 3: return c.layer_types.length + "층 · 1번";
      case 4: return L[b] + " 점수 " + c.logits[b].toFixed(2);
      case 5: return d.best + " " + f2(d.p);
      case 6: return d.act ? d.best + "(으)로" : "되묻기";
    }
  }

  function renderCases() {
    var box = root.querySelector(".jx-cases"); box.innerHTML = "";
    DATA.cases.forEach(function (c, i) {
      box.appendChild(el("button", { class: "jx-case", "aria-pressed": i === S.ci ? "true" : "false", title: c.id,
        onclick: function () { S.ci = i; S.perm = 0; render(); } }, c.title));
    });
  }
  function renderFlow() {
    var box = root.querySelector(".jx-flow"); box.innerHTML = ""; var c = cur();
    STEPS.forEach(function (st, i) {
      var b = el("button", { class: "jx-node" + (i < S.step ? " done" : ""), "aria-current": i === S.step ? "step" : null,
        onclick: function () { S.step = i; render(); } },
        el("span", { class: "n", text: "①②③④⑤⑥⑦".charAt(i) }), el("span", { class: "t", text: st.name }), el("span", { class: "s", text: nodeSub(i, c) }));
      box.appendChild(b);
    });
  }

  // ① 입력
  function step0(c, P) {
    var g = el("div", { class: "jx-grid2" });
    g.appendChild(el("div", { class: "jx-card" }, el("div", { class: "jx-lab", text: "상황 (state)" }), el("div", { text: typeof c.row.state === "string" ? c.row.state : JSON.stringify(c.row.state) })));
    var ul = el("ul");
    c.row.options.forEach(function (o, i) { ul.appendChild(el("li", null, el("span", { class: "jx-chip", text: o.id }), " ", o.description)); });
    g.appendChild(el("div", { class: "jx-card" }, el("div", { class: "jx-lab", text: "질문 (question)" }), el("div", { text: c.row.question }), el("div", { class: "jx-lab", style: "margin-top:8px", text: "보기 (options) — id와 설명" }), ul));
    P.appendChild(g);
    var call = "row = {\n  \"id\": " + JSON.stringify(c.row.id) + ",\n  \"state\": \"…\",      # 위 왼쪽\n  \"question\": \"…\",   # 위 오른쪽\n  \"options\": [" + c.row.options.map(function (o) { return "{\"id\": " + JSON.stringify(o.id) + ", …}"; }).join(", ") + "],\n}\nscore(model, tokenizer, row, metadata)   # SemIf: 보기마다 확률을 돌려줘요";
    P.appendChild(el("pre", { class: "jx-code", text: call }));
    P.appendChild(el("div", { class: "jx-note", html: "Jev의 <b>CHOICE</b> 질문과 같은 모양이에요. 보기는 2~16개까지 되고, 모델이 새 글을 쓰지 않으니 답은 반드시 이 보기 중 하나예요." }));
  }

  // ② 프롬프트 조립
  function promptView(c, cls) {
    var pre = el("div", { class: cls || "jx-prompt" }), cursor = 0, prev = null, buf = "";
    function flush() { if (buf) { pre.appendChild(el("span", { class: "r-" + prev, title: ROLE[prev] }, buf)); buf = ""; } }
    c.tokens.forEach(function (t) {
      var s = t[1], e = t[2], role = t[3];
      if (e <= cursor) return;
      var text = c.prompt.slice(Math.max(s, cursor), e); cursor = e;
      if (role !== prev) { flush(); prev = role; }
      buf += text;
    });
    flush();
    return pre;
  }
  function legend(counts) {
    var lg = el("div", { class: "jx-legend" });
    ROLE_ORDER.forEach(function (r) {
      var sw = el("i", { class: "r-" + r }); if (r === "json") sw.style.background = "#fff";
      lg.appendChild(el("span", null, sw, ROLE[r] + (counts ? " " + (counts[r] || 0) : "")));
    });
    return lg;
  }
  function step1(c, P) {
    P.appendChild(promptView(c));
    P.appendChild(legend());
    var L = letters(c);
    var notes = el("div", { class: "jx-note" });
    notes.innerHTML = "• <b>지시문</b>: 보기 글자 하나로만 답하라고 해요. &nbsp;• <b>JSON</b>: 상황은 <code>evidence</code>, 질문은 <code>criterion</code>, 보기는 <code>letter</code>+<code>description</code>으로 들어가요.<br>" +
      "• 보기 id(<code>" + c.option_ids.map(esc).join("</code>, <code>") + "</code>)는 프롬프트에 없어요. 모델은 글자 <b>" + L.join(", ") + "</b>만 보고, 코드가 글자를 다시 id로 바꿔요.<br>" +
      "• 맨 끝 <span class='r-gen'>&lt;think&gt; &lt;/think&gt;</span>는 생각 모드를 끈 빈 칸이에요. 그 바로 뒤가 답(글자 하나)이 올 자리예요.";
    P.appendChild(notes);
    var sha = el("div", { class: "jx-note mono", style: "margin-top:6px" });
    sha.textContent = "프롬프트 지문(SHA-256) " + c.sha.slice(0, 16) + "…" + (c.ref && c.ref.sha ? (c.ref.sha === c.sha ? "  ✓ " + c.ref.label + "과 같은 프롬프트" : "  ✗ " + c.ref.label + "과 다름") : "");
    P.appendChild(sha);
  }

  // ③ 토큰
  function step2(c, P) {
    var counts = {}, info = el("div", { class: "jx-tokinfo", text: "토큰에 마우스를 올려 보세요." });
    var box = el("div", { class: "jx-toks" });
    var slotIds = {}; c.slots.forEach(function (s) { slotIds[s[1]] = s[0]; });
    c.tokens.forEach(function (t, i) {
      counts[t[3]] = (counts[t[3]] || 0) + 1;
      var raw = c.prompt.slice(t[1], t[2]);
      var chip = el("span", { class: "jx-tok r-" + t[3] + (i === c.tokens.length - 1 ? " is-last" : "") + (slotIds[t[0]] && t[3] === "letter" ? " is-hot" : ""), tabindex: "-1" });
      var shown = raw.length ? raw : "·";
      var m = shown.match(/^(\s*)([\s\S]*?)(\s*)$/);
      if (m[1]) chip.appendChild(el("span", { class: "ws", text: showWs(m[1]) }));
      if (m[2]) chip.appendChild(document.createTextNode(m[2]));
      if (m[3]) chip.appendChild(el("span", { class: "ws", text: showWs(m[3]) }));
      var msg = "#" + (i + 1) + " · 토큰 id " + t[0] + " · " + JSON.stringify(raw) + " · " + ROLE[t[3]] + (i === c.tokens.length - 1 ? " · ← 여기서 다음 토큰 점수를 읽어요" : "");
      chip.title = msg;
      chip.addEventListener("mouseenter", function () { info.textContent = msg; });
      box.appendChild(chip);
    });
    P.appendChild(box);
    P.appendChild(info);
    var stack = el("div", { class: "jx-stack", role: "img", "aria-label": "역할별 토큰 수" });
    ROLE_ORDER.forEach(function (r) {
      if (!counts[r]) return;
      var d = el("div", { class: "r-" + r, title: ROLE[r] + " " + counts[r] + "개" });
      d.style.flex = String(counts[r]); if (r === "json") d.style.background = "#E9EEEC";
      stack.appendChild(d);
    });
    P.appendChild(stack);
    P.appendChild(legend(counts));
    var L = c.slots.map(function (s) { return s[0] + " = id " + s[1]; }).join(", ");
    P.appendChild(el("div", { class: "jx-note", html: "총 <b>" + c.tokens.length + "토큰</b>. 보기 글자는 각각 토큰 하나예요(<code>" + L + "</code>). SemIf는 글자가 토큰 하나로 딱 떨어지는지 먼저 검사해요. 주황 테두리가 마지막 토큰 — 이 자리에서 '다음에 올 토큰'의 점수를 읽어요." }));
  }

  // ④ 모델 1번 통과: 층 띠 + 로짓 렌즈
  function step3(c, P) {
    var types = c.layer_types, n = types.length, W = 720, H = 250, l = 44, r = 150, t = 16, b = 34;
    var strip = sv("svg", { viewBox: "0 0 " + W + " 40", width: "100%", style: "max-width:" + W + "px;display:block", role: "img", "aria-label": "32개 층의 종류" });
    var cw = (W - l - r) / n, full = 0;
    types.forEach(function (ty, i) {
      var isFull = ty === "full_attention"; if (isFull) full++;
      var rect = sv("rect", { x: l + i * cw + 1, y: 6, width: cw - 2, height: 18, rx: 3, fill: isFull ? C.ink : "#CFE0D9" });
      rect.appendChild(sv("title", { text: (i + 1) + "층 · " + (isFull ? "전체 어텐션" : "선형 어텐션") }));
      strip.appendChild(rect);
      if ((i + 1) % 4 === 0 || i === 0) strip.appendChild(sv("text", { x: l + i * cw + cw / 2, y: 37, "text-anchor": "middle", "font-size": 10, fill: C.muted, text: String(i + 1) }));
    });
    strip.appendChild(sv("text", { x: l + n * cw + 8, y: 19, "font-size": 11.5, fill: C.muted, text: "▮ 전체 " + full + " · ▯ 선형 " + (n - full) }));
    P.appendChild(strip);

    var host = el("div", { class: "jx-chart" }), tip = tipHost(host);
    var svg = sv("svg", { viewBox: "0 0 " + W + " " + H, width: "100%", style: "max-width:" + W + "px;display:block", role: "img", "aria-label": "층마다 읽어 본 보기 확률" });
    var pw = W - l - r, ph = H - t - b;
    function X(i) { return l + (i - 1) * pw / (n - 1); }
    function Y(p) { return t + (1 - p) * ph; }
    [0, 0.5, 1].forEach(function (v) {
      svg.appendChild(sv("line", { x1: l, x2: l + pw, y1: Y(v), y2: Y(v), stroke: C.line, "stroke-width": 1 }));
      svg.appendChild(sv("text", { x: l - 6, y: Y(v) + 4, "text-anchor": "end", "font-size": 11, fill: C.muted, text: v.toFixed(1) }));
    });
    types.forEach(function (ty, i) { if (ty === "full_attention") svg.appendChild(sv("line", { x1: X(i + 1), x2: X(i + 1), y1: t + ph, y2: t + ph + 5, stroke: C.ink, "stroke-width": 1.5 })); });
    [1, 8, 16, 24, 32].forEach(function (k) { svg.appendChild(sv("text", { x: X(k), y: H - 12, "text-anchor": "middle", "font-size": 11, fill: C.muted, text: k + "층" })); });
    var lens = c.lens.slice(1).map(function (z) { return softmax(z, 1); });  // 0번(임베딩)은 뺐어요
    var fin = argmax(c.logits);
    var order = c.option_ids.map(function (_, j) { return j; }).sort(function (a, bb) { return (a === fin) - (bb === fin); });
    var ends = [];
    order.forEach(function (j) {
      var d = lens.map(function (p, i) { return (i ? "L" : "M") + X(i + 1).toFixed(1) + "," + Y(p[j]).toFixed(1); }).join(" ");
      svg.appendChild(sv("path", { d: d, fill: "none", stroke: j === fin ? C.green : C.gray, "stroke-width": j === fin ? 2.5 : 1.8, "stroke-linejoin": "round", "stroke-linecap": "round" }));
      var last = lens[lens.length - 1][j];
      svg.appendChild(sv("circle", { cx: X(n), cy: Y(last), r: 4, fill: j === fin ? C.green : C.gray, stroke: "#fff", "stroke-width": 2 }));
      ends.push({ j: j, y: Y(last) });
    });
    ends.sort(function (a, bb) { return a.y - bb.y; });
    for (var k = 1; k < ends.length; k++) if (ends[k].y - ends[k - 1].y < 14) ends[k].y = ends[k - 1].y + 14;
    ends.forEach(function (e) {
      svg.appendChild(sv("text", { x: X(n) + 10, y: e.y + 4, "font-size": 12, fill: C.ink, "font-weight": e.j === fin ? 700 : 400, text: letters(c)[e.j] + " " + c.option_ids[e.j] + " " + f2(lens[lens.length - 1][e.j]) }));
    });
    var cross = sv("line", { x1: 0, x2: 0, y1: t, y2: t + ph, stroke: C.ink, "stroke-width": 1, opacity: 0 });
    svg.appendChild(cross);
    var hit = sv("rect", { x: l, y: t, width: pw, height: ph, fill: "transparent" });
    hit.addEventListener("mousemove", function (ev) {
      var box = svg.getBoundingClientRect(), sx = (ev.clientX - box.left) * W / box.width;
      var i = Math.max(1, Math.min(n, Math.round((sx - l) / pw * (n - 1)) + 1));
      cross.setAttribute("x1", X(i)); cross.setAttribute("x2", X(i)); cross.setAttribute("opacity", 0.35);
      var p = lens[i - 1], html = "<b>" + i + "층</b> (" + (types[i - 1] === "full_attention" ? "전체 어텐션" : "선형 어텐션") + ")<br>" +
        c.option_ids.map(function (o, j) { return letters(c)[j] + " " + esc(o) + " " + f2(p[j]); }).join("<br>");
      tip.show((X(i) / W) * box.width, (Y(Math.max.apply(null, p)) / H) * box.height, html);
    });
    hit.addEventListener("mouseleave", function () { cross.setAttribute("opacity", 0); tip.hide(); });
    svg.appendChild(hit);
    host.appendChild(svg);
    P.appendChild(host);
    var ms = c.ms ? " 이 케이스 한 번 통과: <b>" + Math.round(c.ms) + "ms</b> (" + esc(DATA.deviceLabel) + ")." : "";
    P.appendChild(el("div", { class: "jx-note", html: "Qwen3.5-4B는 <b>선형 어텐션 층 3개 + 전체 어텐션 층 1개</b>를 8번 쌓았어요." + ms +
      "<br>선은 '로짓 렌즈'예요: 각 층을 지난 마지막 토큰 상태에 모델의 마지막 정규화와 출력층을 바로 붙여, 그 층에서 멈췄다면 보기 글자 확률이 어땠을지 본 <b>분석용 근사</b>예요. SemIf 판정은 32층 끝에서 한 번만 읽어요. 초록 선이 최종 선택이에요." }));
  }

  // ⑤ 글자 점수
  function step4(c, P) {
    var row = el("div", { class: "jx-row" }), left = el("div", { class: "grow" }), right = el("div", { class: "side" });
    var slotSet = {}; c.slots.forEach(function (s) { slotSet[s[1]] = s[0]; });
    left.appendChild(el("div", { class: "jx-lab", text: "마지막 자리에서 모델이 다음에 쓰고 싶은 토큰 상위 " + c.top.length + "개 (어휘 " + (DATA.vocabSize ? DATA.vocabSize.toLocaleString() : "전체") + "개 기준)" }));
    left.appendChild(hbars(c.top.map(function (t) {
      var isSlot = slotSet[t[0]] !== undefined;
      return { label: tokLabel(t[1]) + (isSlot ? "  (보기 " + slotSet[t[0]] + ")" : ""), v: t[2], valueText: pct(t[2]), color: isSlot ? C.green : C.gray, bold: isSlot };
    }), { width: 520, labelWidth: 170, aria: "전체 어휘 상위 토큰 확률" }));
    row.appendChild(left);
    right.appendChild(el("div", { class: "jx-stat" }, el("div", { class: "v", text: pct(c.mass) }), el("div", { class: "l", text: "보기 글자 " + letters(c).join("·") + "에 모인 확률 (나머지는 '<think>' 같은 다른 토큰)" })));
    var tbl = el("table", { class: "jx-tbl" }), b = argmax(c.logits);
    tbl.appendChild(el("tr", null, el("th", { text: "글자" }), el("th", { text: "보기 id" }), el("th", { class: "num", text: "점수(logit)" })));
    c.option_ids.forEach(function (o, j) {
      tbl.appendChild(el("tr", { class: j === b ? "best" : null }, el("td", { text: letters(c)[j] }), el("td", { text: o }), el("td", { class: "num", text: c.logits[j].toFixed(3) })));
    });
    right.appendChild(tbl);
    row.appendChild(right);
    P.appendChild(row);
    P.appendChild(el("div", { class: "jx-note", html: "SemIf는 전체 어휘 점수 중 <b>보기 글자 토큰의 점수만</b> 골라요. 그래서 보기 밖 답이 나올 수 없어요. 점수(logit)는 크기 비교용 숫자이고, 아직 확률이 아니에요." }));
  }

  // ⑥ 확률 + 온도 + 순서 바꾸기
  // 슬라이더를 끌 때는 아래 그림만 다시 그려요(슬라이더 자체를 새로 만들면 끌기가 끊겨요).
  function tempCtrl(P, redraw) {
    var ctrl = el("div", { class: "jx-ctrl" });
    var val = el("span", { class: "val", text: "T = " + S.T.toFixed(2) });
    var rng = el("input", { type: "range", min: "-2", max: "2", step: "0.05", value: String(Math.log2(S.T)), "aria-label": "온도 T" });
    rng.addEventListener("input", function () { S.T = Math.pow(2, parseFloat(rng.value)); val.textContent = "T = " + S.T.toFixed(2); redraw(); renderFlow(); renderFoot(); });
    ctrl.appendChild(el("label", { text: "온도" })); ctrl.appendChild(rng); ctrl.appendChild(val);
    ctrl.appendChild(el("button", { class: "jx-btn", text: "T=1 (원래 값)", onclick: function () {
      S.T = 1; rng.value = "0"; val.textContent = "T = 1.00"; redraw(); renderFlow(); renderFoot(); } }));
    ctrl.appendChild(el("span", { class: "jx-foot", text: "p = softmax(점수 ÷ T)" }));
    P.appendChild(ctrl);
  }
  function step5(c, P) {
    var body = el("div");
    function redraw() {
      body.innerHTML = "";
      var L = letters(c), rows;
      var pmode = S.permOn && c.perms && c.perms.length > 1;
      if (!pmode) {
        var p = probsNow(c), b = argmax(p);
        rows = c.option_ids.map(function (o, j) {
          return { label: L[j] + " · " + o, v: p[j], bold: j === b, color: j === b ? C.green : C.gray, ref: c.ref && c.ref.probs ? c.ref.probs[o] : null,
            title: L[j] + " · " + o + " (" + desc(c, o) + "): " + f2(p[j]) + (c.ref && c.ref.probs ? " · " + c.ref.label + " " + f2(c.ref.probs[o]) : "") };
        });
        body.appendChild(hbars(rows, { width: 640, labelWidth: 200, axis: true, aria: "보기 확률" }));
        if (c.ref && c.ref.probs) body.appendChild(el("div", { class: "jx-legend" },
          el("span", { html: "<i style='background:" + C.green + "'></i>이번 숫자 (T=" + S.T.toFixed(2) + ")" }),
          el("span", { html: "<b style='color:" + C.ink + "'>│</b> " + esc(c.ref.label) + " (T=1)" })));
      } else {
        var pm = c.perms[S.perm], z = pm.order.map(function (o) { return pm.logits[o]; }), pp = softmax(z, S.T), bb = argmax(pp);
        rows = pm.order.map(function (o, j) { return { label: L[j] + " · " + o, v: pp[j], bold: j === bb, color: j === bb ? C.green : C.gray }; });
        body.appendChild(el("div", { class: "jx-lab", text: "지금 순서: " + pm.order.map(function (o, j) { return L[j] + "=" + o; }).join(", ") }));
        body.appendChild(hbars(rows, { width: 640, labelWidth: 200, axis: true, aria: "순서를 바꾼 보기 확률" }));
      }
      if (c.perms && c.perms.length > 1) {
        var bar = el("div", { class: "jx-ctrl", style: "margin-top:10px" });
        bar.appendChild(el("button", { class: "jx-btn", "aria-pressed": S.permOn ? "true" : "false",
          onclick: function () { S.permOn = !S.permOn; redraw(); renderFoot(); },
          text: S.permOn ? "보기 순서 바꾸기 끄기" : "보기 순서 바꿔 보기 (" + c.perms.length + "가지)" }));
        bar.appendChild(el("span", { class: "jx-foot", text: "보기 내용은 그대로, 글자 A·B…만 다른 보기에 붙여 SemIf를 다시 돌린 결과예요." }));
        body.appendChild(bar);
        if (S.permOn) {
          var pb = el("div", { class: "jx-perms" });
          c.perms.forEach(function (pm2, k) {
            pb.appendChild(el("button", { class: "jx-btn", "aria-pressed": k === S.perm ? "true" : "false", title: pm2.order.join(" · "),
              onclick: function () { S.perm = k; redraw(); renderFoot(); }, text: pm2.order.map(function (o) { return o.slice(0, 6); }).join("·") }));
          });
          body.appendChild(pb);
        }
        body.appendChild(spread(c));
      }
    }
    tempCtrl(P, redraw);
    P.appendChild(body);
    redraw();
    P.appendChild(el("div", { class: "jx-note", html: "SemIf가 돌려주는 원래 확률은 <b>T=1</b>이에요. T를 키우면 고르게, 줄이면 뾰족해지지만 <b>1등은 바뀌지 않아요</b>. 내 데이터로 T를 맞추는 게 1강의 보정(온도 조절)이에요 — SemIf 저장소에도 작업별 온도 보정 기능이 있어요. 이 숫자는 '보기끼리 비교한 점수'이지, 맞을 확률로 보정된 값이 아니에요." }));
  }
  function spread(c) {
    var W = 640, lab = 200, rowH = 26, H = c.option_ids.length * rowH + 30, pw = W - lab - 70;
    var L = letters(c);
    var box = el("div", { style: "margin-top:4px" });
    box.appendChild(el("div", { class: "jx-lab", text: "순서에 따른 흔들림: 보기마다 " + c.perms.length + "가지 순서의 확률 범위 (T=1)" }));
    var svg = sv("svg", { viewBox: "0 0 " + W + " " + H, width: "100%", style: "max-width:" + W + "px;display:block", role: "img", "aria-label": "순서에 따른 확률 범위" });
    [0, 0.5, 1].forEach(function (v) {
      svg.appendChild(sv("line", { x1: lab + pw * v, x2: lab + pw * v, y1: 0, y2: H - 18, stroke: C.line, "stroke-width": 1 }));
      svg.appendChild(sv("text", { x: lab + pw * v, y: H - 4, "text-anchor": "middle", "font-size": 11, fill: C.muted, text: v.toFixed(1) }));
    });
    var words = [];
    c.option_ids.forEach(function (o, j) {
      var vals = c.perms.map(function (pm) { return softmax(pm.order.map(function (x) { return pm.logits[x]; }), 1)[pm.order.indexOf(o)]; });
      var kmin = 0, kmax = 0;
      vals.forEach(function (v, k) { if (v < vals[kmin]) kmin = k; if (v > vals[kmax]) kmax = k; });
      var lo = vals[kmin], hi = vals[kmax], y = j * rowH + 12;
      svg.appendChild(sv("text", { x: lab - 8, y: y + 4, "text-anchor": "end", "font-size": 12.5, fill: C.ink, text: o }));
      svg.appendChild(sv("line", { x1: lab + pw * lo, x2: lab + pw * hi, y1: y, y2: y, stroke: C.gray, "stroke-width": 6, "stroke-linecap": "round" }));
      vals.forEach(function (v, k) {
        var on = S.permOn && k === S.perm;
        var cc = sv("circle", { cx: lab + pw * v, cy: y, r: on ? 5 : 3.2, fill: on ? C.orange : C.green, stroke: "#fff", "stroke-width": 1.5 });
        cc.appendChild(sv("title", { text: c.perms[k].order.map(function (x, i) { return L[i] + "=" + x; }).join(", ") + " → " + o + " " + f2(v) }));
        svg.appendChild(cc);
      });
      svg.appendChild(sv("text", { x: lab + pw * hi + 10, y: y + 4, "font-size": 12, fill: C.ink, text: f2(lo) + " ~ " + f2(hi) }));
      words.push({ o: o, lo: lo, hi: hi, atHi: L[c.perms[kmax].order.indexOf(o)], atLo: L[c.perms[kmin].order.indexOf(o)] });
    });
    box.appendChild(svg);
    var w = words.sort(function (a, b) { return (b.hi - b.lo) - (a.hi - a.lo); })[0];
    var msg = "순서만 바꿨는데 <b>" + esc(w.o) + "</b>는 <b>" + f2(w.lo) + " ~ " + f2(w.hi) + "</b> 사이를 오가요 (" + w.atHi + " 자리일 때 " + f2(w.hi) + ", " + w.atLo + " 자리일 때 " + f2(w.lo) + ").";
    msg += (w.hi - w.lo) >= 0.1 ? " 보기 순서도 입력의 일부라서, 작은 모델은 자리(글자)에 따라 쏠려요 — 1강에서 본 '예시 순서·흔한 라벨 쪽 쏠림'과 같은 현상이에요." : " 이 케이스는 순서에 거의 흔들리지 않아요.";
    box.appendChild(el("div", { class: "jx-note", html: msg }));
    return box;
  }

  // ⑦ 코드의 판단
  function step6(c, P) {
    var body = el("div");
    function redraw() {
      body.innerHTML = "";
      var d = decisionOf(c);
      body.appendChild(el("div", { class: "jx-decision " + (d.act ? "act" : "ask"), text: d.act ? "→ " + d.best + " (으)로 진행  (" + f2(d.p) + " ≥ " + f2(S.tau) + ")" : "→ 사람에게 되묻기  (" + f2(d.p) + " < " + f2(S.tau) + ")" }));
      var pj = "{" + c.option_ids.map(function (o, j) { return JSON.stringify(o) + ": " + f2(d.probs[j]); }).join(", ") + "}";
      var lines = [
        ["p = " + pj, "# ⑥에서 나온 확률 (T=" + S.T.toFixed(2) + ")"],
        ["best = max(p, key=p.get)", "# " + JSON.stringify(d.best)],
        ["if p[best] >= " + f2(S.tau) + ":", "# 코드의 기준"],
        ["    act(best)", d.act ? "# ← 이번엔 여기" : ""],
        ["else:", ""],
        ["    ask_user()", d.act ? "" : "# ← 이번엔 여기"]
      ];
      var pre = el("pre", { class: "jx-code" });
      lines.forEach(function (ln, i) {
        var on = (d.act && i === 3) || (!d.act && i === 5);
        var span = el("span", { class: on ? "hl" : null });
        span.appendChild(document.createTextNode(ln[0]));
        if (ln[1]) { var pad = Math.max(2, 34 - ln[0].length); span.appendChild(el("span", { class: "cm", text: " ".repeat(pad) + ln[1] })); }
        pre.appendChild(span); pre.appendChild(document.createTextNode("\n"));
      });
      body.appendChild(pre);
      var L = letters(c), chart = el("div", { style: "margin-top:12px" });
      chart.appendChild(hbars(c.option_ids.map(function (o, j) { return { label: L[j] + " · " + o, v: d.probs[j], bold: o === d.best, color: o === d.best ? (d.act ? C.green : C.orange) : C.gray }; }), { width: 640, labelWidth: 200, axis: true, threshold: S.tau, aria: "기준선과 보기 확률" }));
      body.appendChild(chart);
    }
    var ctrl = el("div", { class: "jx-ctrl" });
    var val = el("span", { class: "val", text: f2(S.tau) });
    var rng = el("input", { type: "range", min: "0.5", max: "0.99", step: "0.01", value: String(S.tau), "aria-label": "기준" });
    rng.addEventListener("input", function () { S.tau = parseFloat(rng.value); val.textContent = f2(S.tau); redraw(); renderFlow(); renderFoot(); });
    ctrl.appendChild(el("label", { text: "코드의 기준 τ" })); ctrl.appendChild(rng); ctrl.appendChild(val);
    ctrl.appendChild(el("span", { class: "jx-foot", text: "온도 T = " + S.T.toFixed(2) + " (⑥에서 바꿀 수 있어요)" }));
    P.appendChild(ctrl);
    P.appendChild(body);
    redraw();
    P.appendChild(el("div", { class: "jx-note", html: "모델은 확률만 줘요. 얼마 이상이면 믿고 움직일지는 <b>코드가</b> 정해요. 이 기준은 7장의 보정 결과와 내 데이터로 정해야 해요 — 보정 전 점수에 0.9를 걸면 '0.9라더니 틀린' 일이 생겨요." }));
  }

  function renderPanel() {
    var P = root.querySelector(".jx-panel"); P.innerHTML = ""; var c = cur();
    P.appendChild(el("h4", { text: "①②③④⑤⑥⑦".charAt(S.step) + " " + STEPS[S.step].name }));
    P.appendChild(el("p", { class: "jx-lead", text: STEPS[S.step].lead }));
    [step0, step1, step2, step3, step4, step5, step6][S.step](c, P);
    var nav = el("div", { class: "jx-nav" });
    nav.appendChild(el("button", { class: "jx-btn", disabled: S.step === 0 ? "disabled" : null, onclick: function () { S.step = Math.max(0, S.step - 1); render(); }, text: "◀ 이전" }));
    nav.appendChild(el("span", { class: "jx-foot", text: "← → 키로도 움직여요" }));
    nav.appendChild(el("button", { class: "jx-btn", disabled: S.step === 6 ? "disabled" : null, onclick: function () { S.step = Math.min(6, S.step + 1); render(); }, text: "다음 ▶" }));
    P.appendChild(nav);
  }
  function renderFoot() {
    var c = cur(), d = decisionOf(c), f = root.querySelector(".jx-foot-line");
    f.innerHTML = "케이스 <b>" + esc(c.id) + "</b> · 온도 T <b>" + S.T.toFixed(2) + "</b> · 기준 τ <b>" + f2(S.tau) + "</b> · 1등 <b>" + esc(d.best) + " " + f2(d.p) + "</b> → <b>" + (d.act ? "진행" : "되묻기") + "</b>" + (S.permOn ? " · 순서 바꾸기 켜짐(⑥)" : "");
  }
  function render() { renderCases(); renderFlow(); renderPanel(); renderFoot(); }

  // 뼈대
  var stat = root.querySelector(".jx-static"); if (stat) stat.remove();
  root.appendChild(el("div", { class: "jx-head" },
    el("div", { class: "jx-title" }, "한 케이스 뜯어보기", el("small", { text: "Jev 닮은 판단(SemIf 방식)이 계산되는 길" })),
    el("div", { class: "jx-src", text: DATA.sourceLabel })));
  root.appendChild(el("div", { class: "jx-cases", role: "group", "aria-label": "케이스 고르기" }));
  root.appendChild(el("div", { class: "jx-flow" }));
  root.appendChild(el("div", { class: "jx-panel" }));
  root.appendChild(el("div", { class: "jx-foot jx-foot-line", style: "margin-top:8px" }));
  root.addEventListener("keydown", function (ev) {
    if (ev.target && ev.target.tagName === "INPUT") return;
    if (ev.key === "ArrowRight") { S.step = Math.min(6, S.step + 1); render(); ev.preventDefault(); }
    if (ev.key === "ArrowLeft") { S.step = Math.max(0, S.step - 1); render(); ev.preventDefault(); }
  });
  render();
  try { if (window.google && google.colab && google.colab.output && google.colab.output.setIframeHeight) google.colab.output.setIframeHeight(0, true, { maxHeight: 5000 }); } catch (e) {}
})();
</script>
'''
print('그림 도우미 준비 완료')

**② 판정 뜯어보기 도우미**예요. `explain_case()`는 SemIf `score()`를 그대로 부르고, 위젯에 필요한 토큰·층별 점수·보기 순서 바꾸기 결과를 더 모아요. `run_shared_demo()`는 6장의 공유 모드 실측이에요. 모델은 3장에서 올려요.

In [ ]:
#@title 🔧 ② 판정 뜯어보기 도우미 (SemIf 방식 그대로)
# 한 케이스를 SemIf와 똑같은 방식으로 판정하면서, 위젯에 보여 줄 중간값을 함께 모아요.
# - 판정 숫자(확률)는 SemIf 원본 함수 semif_phase1.direct.score()가 낸 값을 그대로 씁니다.
# - 토큰 색칠, 전체 어휘 상위 후보, 층별 점수(logit lens), 보기 순서 바꾸기는 "보여 주기용" 추가 계산이에요.
import itertools
import json
import time

ROLE_NAMES = {
    "template": "채팅 틀(특수 토큰)",
    "system": "지시문(system)",
    "json": "JSON 키·기호",
    "evidence": "상황(state)",
    "criterion": "질문(question)",
    "letter": "보기 글자",
    "option": "보기 설명",
    "gen": "답이 올 자리",
}


def _payload_spans(row, letters):
    """SemIf direct_messages()가 만드는 JSON 문자열을 똑같이 조립하면서 부분별 글자 위치를 기록해요."""
    parts = []

    def add(text, role, label=None):
        parts.append((text, role, label))

    add('{"evidence": ', "json")
    add(json.dumps(row["state"], ensure_ascii=False), "evidence")
    add(', "criterion": ', "json")
    add(json.dumps(row["question"], ensure_ascii=False), "criterion")
    add(', "options": [', "json")
    for index, option in enumerate(row["options"]):
        if index:
            add(", ", "json")
        add('{"letter": ', "json")
        add(json.dumps(letters[index], ensure_ascii=False), "letter", letters[index])
        add(', "description": ', "json")
        add(json.dumps(option["description"], ensure_ascii=False), "option", letters[index])
        add("}", "json")
    add("]}", "json")
    text, spans, cursor = "", [], 0
    for piece, role, label in parts:
        spans.append((cursor, cursor + len(piece), role, label))
        text += piece
        cursor += len(piece)
    return text, spans


def _role_spans(prompt, messages, row, letters):
    system_text = messages[0]["content"]
    user_text = messages[1]["content"]
    built, payload_spans = _payload_spans(row, letters)
    if built != user_text:
        raise ValueError("SemIf 프롬프트 조립 방식이 바뀌었어요. 색칠 규칙을 고쳐야 해요.")
    spans = []
    s0 = prompt.index(system_text)
    spans.append((s0, s0 + len(system_text), "system", None))
    u0 = prompt.index(user_text, s0 + len(system_text))
    for start, end, role, label in payload_spans:
        spans.append((u0 + start, u0 + end, role, label))
    spans.append((u0 + len(user_text), len(prompt), "gen_or_template", None))
    return spans, u0 + len(user_text)


def _token_role(spans, start, end, user_end, prompt):
    if end <= start:
        return "template", None
    best, best_len = ("template", None), 0
    for s, e, role, label in spans:
        overlap = min(end, e) - max(start, s)
        if overlap > best_len:
            best, best_len = (role, label), overlap
    if best[0] == "gen_or_template":
        # 사용자 턴이 끝난 뒤: assistant 머리말은 틀, 그 뒤 빈 생각 블록은 '답이 올 자리' 앞부분
        tail = prompt[user_end:]
        head = tail.find("assistant")
        cut = user_end + (head + len("assistant") if head >= 0 else 0)
        return ("gen", None) if start >= cut else ("template", None)
    if best[0] in ("evidence", "criterion", "letter", "option") and not prompt[start:end].strip(' {}[]":,'):
        return "json", None  # 따옴표·쉼표만 있는 토큰은 JSON 기호로 칠해요
    return best


def explain_case(model, tokenizer, row, metadata, *, topk=10, lens=True, permutations=True, max_perms=24):
    """SemIf 방식 판정 1건 + 위젯용 중간값. 반환값은 JSON으로 바로 저장할 수 있어요."""
    import torch
    from semif_phase1.core import LETTERS, direct_messages, validate_row
    from semif_phase1.direct import encode_prompt, score

    validate_row(row)
    official = score(model, tokenizer, row, metadata)  # ← SemIf 원본 판정 (확률은 이 값)
    messages = direct_messages(row)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids, slots, prompt_hash = encode_prompt(tokenizer, row, 4096)
    assert prompt_hash == official["prompt_sha256"]
    letters = LETTERS[: len(row["options"])]

    # ① 토큰과 역할(색칠용)
    enc = tokenizer(prompt, add_special_tokens=False, return_offsets_mapping=True)
    if list(enc["input_ids"]) != list(ids):
        raise ValueError("토큰화 결과가 SemIf와 달라요")
    spans, user_end = _role_spans(prompt, messages, row, letters)
    # 브라우저(JavaScript)는 글자 위치를 UTF-16 단위로 세요. 이모지 같은 글자가 있어도 어긋나지 않게 바꿔 둬요.
    u16 = [0]
    for ch in prompt:
        u16.append(u16[-1] + (2 if ord(ch) > 0xFFFF else 1))
    tokens = []
    for token_id, (start, end) in zip(ids, enc["offset_mapping"]):
        role, label = _token_role(spans, start, end, user_end, prompt)
        tokens.append([int(token_id), u16[start], u16[end], role, label])

    # ② 모델 한 번 통과: 마지막 위치의 전체 어휘 점수 + 층별 점수(logit lens)
    device = next(model.parameters()).device
    x = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.inference_mode():
        out = model(input_ids=x, attention_mask=torch.ones_like(x), use_cache=False,
                    output_hidden_states=lens, return_dict=True, logits_to_keep=1)
        vocab = out.logits[0, -1].float()
        logp = torch.log_softmax(vocab, dim=-1)
        top = torch.topk(logp, topk)
        slot_logits = vocab[slots]
        mass = float((slot_logits.logsumexp(-1) - vocab.logsumexp(-1)).exp())
        top_tokens = [[int(i), tokenizer.decode([int(i)]), float(p)]
                      for i, p in zip(top.indices.tolist(), top.values.exp().tolist())]
        lens_rows = []
        if lens:
            base = model.model
            states = out.hidden_states
            for index, state in enumerate(states):
                v = state[0, -1]
                if index < len(states) - 1:  # 마지막 항목은 이미 최종 정규화를 거친 값이에요
                    v = base.norm(v)
                lens_rows.append([round(float(z), 3) for z in model.lm_head(v).float()[slots].cpu().tolist()])
        del out
    same = max(abs(a - b) for a, b in zip(slot_logits.cpu().tolist(), official["option_logits"]))

    # ③ 보기 순서 바꾸기: 같은 보기, 순서만 바꿔 SemIf 판정을 다시 돌려요
    perms = []
    if permutations:
        n = len(row["options"])
        for k, order in enumerate(itertools.permutations(range(n))):
            if k >= max_perms:
                break
            shuffled = dict(row, id=f"{row['id']}#perm{k}", options=[row["options"][j] for j in order])
            r = score(model, tokenizer, shuffled, metadata)
            perms.append({
                "order": [row["options"][j]["id"] for j in order],
                "probs": {oid: round(p, 6) for oid, p in zip(r["option_ids"], r["probabilities"])},
                "logits": {oid: round(v, 4) for oid, v in zip(r["option_ids"], r["option_logits"])},
            })

    text_config = model.config
    return {
        "id": row["id"],
        "row": row,
        "system": messages[0]["content"],
        "payload": messages[1]["content"],
        "prompt": prompt,
        "prompt_sha256": prompt_hash,
        "tokens": tokens,
        "slots": [[letter, int(tid)] for letter, tid in zip(letters, slots)],
        "option_ids": official["option_ids"],
        "option_logits": official["option_logits"],
        "probabilities": official["probabilities"],
        "forward_seconds": official["forward_seconds"],
        "input_tokens": official["input_tokens"],
        "top_tokens": top_tokens,
        "allowed_mass": mass,
        "readout_check_max_abs_diff": same,
        "lens": lens_rows,
        "layer_types": list(getattr(text_config, "layer_types", []) or []),
        "perms": perms,
        "model": {k: metadata.get(k) for k in ("source", "revision", "dtype", "device", "torch_version", "transformers_version")},
        "measured_at": time.strftime("%Y-%m-%d %H:%M"),
    }


# ── 6장: 같은 상황에 질문 여러 개 → SemIf 공유 모드(score_shared)와 따로 호출(score) 비교 ──
SHARED_STATE = (
    "Support ticket #4821 from Acme Corp (enterprise plan, customer for 6 years, renewal due next month). "
    "Message: 'Since 09:10 this morning every API call from our checkout service returns HTTP 502. "
    "Our online store has been down for two hours and we are losing orders. We were already promised a fix "
    "last week. If this is not resolved today we will cancel the contract and ask our lawyers to review the SLA.' "
    "Status page: all systems operational. Recent change: API gateway config deployed at 09:05. "
    "Gateway log excerpt: 09:05 config v412 applied; 09:10 upstream checkout-api unhealthy (502) x 1,204; "
    "09:40 retries exhausted; 10:55 still failing. Previous tickets from this customer: 2 in the last 30 days, "
    "both about slow responses, both closed with a promise to investigate."
)
SHARED_QUESTIONS = [
    ("severity", "How severe is this incident?", [("sev0", "Critical: production outage for a paying customer"), ("sev1", "High: major feature degraded"), ("sev2", "Medium: minor issue"), ("sev3", "Low: question or request")]),
    ("team", "Which team should own this ticket?", [("sre", "Infrastructure / SRE"), ("billing", "Billing"), ("sales", "Sales / account management"), ("docs", "Documentation")]),
    ("churn", "Is the customer at risk of leaving?", [("yes", "Yes"), ("no", "No")]),
    ("refund", "Should a refund or credit be offered?", [("yes", "Yes"), ("no", "No")]),
    ("legal", "Does this need legal review?", [("yes", "Yes"), ("no", "No")]),
    ("status", "Should the public status page show an incident?", [("yes", "Yes"), ("no", "No")]),
    ("phone", "Should someone call the customer now?", [("yes", "Yes"), ("no", "No")]),
    ("cause", "What is the most likely root cause area?", [("gateway", "API gateway configuration"), ("db", "Database"), ("client", "Customer's own code"), ("unknown", "Unknown")]),
]


def shared_demo_rows():
    return [dict(id=q, state=SHARED_STATE, question=text, options=[dict(id=i, description=d) for i, d in opts])
            for q, text, opts in SHARED_QUESTIONS]


def run_shared_demo(model, tokenizer, metadata, repeats=2):
    """질문 8개를 ① 따로 8번(score) ② 상황 앞부분을 한 번만 계산(score_shared)으로 판정하고 시간을 재요."""
    from semif_phase1.direct import score
    from semif_phase1.shared import score_shared

    rows = shared_demo_rows()
    score(model, tokenizer, rows[0], metadata)          # 워밍업(버림)
    score_shared(model, tokenizer, rows[:2], metadata)  # 워밍업(버림)
    runs = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        direct = [score(model, tokenizer, r, metadata) for r in rows]
        t_direct = time.perf_counter() - t0
        t0 = time.perf_counter()
        shared, timing = score_shared(model, tokenizer, rows, metadata)
        t_shared = time.perf_counter() - t0
        runs.append((t_direct, t_shared, timing))
    t_direct, t_shared, timing = min(runs, key=lambda x: x[0] + x[1])
    pick = lambda r: r["option_ids"][max(range(len(r["probabilities"])), key=r["probabilities"].__getitem__)]
    return {
        "questions": len(rows),
        "direct_seconds": t_direct,
        "shared_seconds": t_shared,
        "direct_tokens": sum(r["input_tokens"] for r in direct),
        "shared_tokens": timing["prefix_tokens"] + timing["true_suffix_tokens"],
        "prefix_tokens": timing["prefix_tokens"],
        "serving_config": shared[0]["model"].get("serving_config"),
        "same_picks": sum(pick(a) == pick(b) for a, b in zip(direct, shared)),
        "max_prob_diff": max(abs(p - q) for a, b in zip(direct, shared) for p, q in zip(a["probabilities"], b["probabilities"])),
        "answers": [{"id": a["id"], "direct": dict(zip(a["option_ids"], [round(p, 4) for p in a["probabilities"]])),
                     "shared": dict(zip(b["option_ids"], [round(p, 4) for p in b["probabilities"]]))}
                    for a, b in zip(direct, shared)],
        "repeats": repeats,
    }


print('판정 뜯어보기 도우미 준비 완료')

**③ 미리 잰 숫자**예요. 강의 케이스와 A4000 기록(`CASES`), 차트용 숫자(`PRE`), GPU가 없을 때 쓸 위젯 숫자(`FALLBACK`)를 불러와요. 출력 한 줄에 무엇을 불러왔는지 나와요.

In [ ]:
#@title 📦 ③ 미리 잰 숫자 불러오기 (출처는 각 묶음의 source)
import json

# 강의 5케이스 입력 + 강사 A4000 기록(results_a4000.jsonl) + 8장 예시 질문
CASES = json.loads(r'''{"_source":{"rows":"~/jev/lecture-code/colab/3강_openjev_colab.ipynb 셀 8 (= ~/jev/lecture-code/3강-오픈소스/openjev/decisions.jsonl 앞 5줄)","a4000":"~/jev/lecture-code/3강-오픈소스/openjev/results_a4000.jsonl (강사 RTX A4000, bfloat16, SemIf 옛 이름 openjev_phase1 direct-options-v1)","my_row":"~/jev/lecture-code/colab/3강_openjev_colab.ipynb 셀 12 예시"},"a4000_model":{"source":"Qwen/Qwen3.5-4B","revision":"851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a","dtype":"bfloat16","torch_version":"2.14.0+cu126","transformers_version":"5.17.0"},"rows":[{"id":"route-gpu","state":"A dashboard routing request. Message: 'A4000 GPU server experiment - continue processing it.' Facts: [a4000] Windows GPU experiment worker, [main] general conversation session.","question":"Which session should handle the message?","options":[{"id":"a4000","description":"Windows GPU experiment worker session"},{"id":"main","description":"General conversation session"}]},{"id":"route-vague","state":"A dashboard routing request. Message: 'just handle this for me' (no destination indicated). Facts: [a4000] Windows GPU experiment worker, [main] general conversation session.","question":"Which session should handle the message?","options":[{"id":"a4000","description":"Windows GPU experiment worker session"},{"id":"main","description":"General conversation session"}]},{"id":"classify-fail","state":"A command failed while running. No error text or logs are available in the state.","question":"What is the failure classification?","options":[{"id":"transient","description":"Temporary error - retry once"},{"id":"code","description":"Code defect - needs a fix"},{"id":"permission","description":"Permission or auth issue"},{"id":"unknown","description":"Cannot determine - ask the user"}]},{"id":"route-mail","state":"A dashboard routing request. Message: 'check the mail and report urgent items only.' Facts: [a4000] Windows GPU experiment worker, [todobot] email-checking assistant session, [main] general conversation session.","question":"Which session should handle the message?","options":[{"id":"a4000","description":"Windows GPU experiment worker session"},{"id":"todobot","description":"Email checking assistant"},{"id":"main","description":"General conversation session"}]},{"id":"gate-danger","state":"An agent is about to run a command: delete all files in the project directory. The user asked only for a summary report.","question":"Should the command proceed without asking the user?","options":[{"id":"proceed","description":"Run it now without asking"},{"id":"ask","description":"Stop and confirm with the user first"}]}],"a4000":{"route-gpu":{"option_ids":["a4000","main"],"probabilities":[0.9947798743064417,0.005220125693558397],"option_logits":[26.125,20.875],"input_tokens":138,"forward_seconds":0.43067120003979653,"prompt_sha256":"37d0a88b8fc984e09d830ac8abfb8d61f5cb29e4d5656d0b629c26f44e081013"},"route-vague":{"option_ids":["a4000","main"],"probabilities":[0.37754066879814546,0.6224593312018546],"option_logits":[24.0,24.5],"input_tokens":136,"forward_seconds":0.12508429994340986,"prompt_sha256":"9ffac973d3eb178706c7d013bb27d80c513f2a817a1a1e552f4bffd791554df9"},"classify-fail":{"option_ids":["transient","code","permission","unknown"],"probabilities":[0.025806747739553104,0.003082176483845284,0.0027200112002125428,0.9683910645763891],"option_logits":[22.25,20.125,20.0,25.875],"input_tokens":147,"forward_seconds":0.11391439998988062,"prompt_sha256":"8e7c19726177d2431f1077f16b36ad347788c6e6c2b73a298ca73e70037fc77d"},"route-mail":{"option_ids":["a4000","todobot","main"],"probabilities":[0.0021820718644840038,0.9975226168216934,0.0002953113138225862],"option_logits":[20.875,27.0,18.875],"input_tokens":158,"forward_seconds":0.10945770004764199,"prompt_sha256":"b2d7f32784f9d997aa7d5472ae39fb4d4423f22163439ad522ad648094f3fd50"},"gate-danger":{"option_ids":["proceed","ask"],"probabilities":[0.0035936025814200896,0.9964063974185798],"option_logits":[21.875,27.5],"input_tokens":128,"forward_seconds":0.10106090002227575,"prompt_sha256":"d6d770811f5964314be48c1e91b534c2a20c099bb0aec60fb50c3abec9859151"}},"my_row":{"id":"my-test","state":"고객 메일: '지난주에 주문한 노트북이 아직 안 왔어요. 그냥 취소하고 환불받고 싶습니다.'","question":"이 메일을 어느 팀으로 보내야 하나요?","options":[{"id":"shipping","description":"배송 조회·지연 담당"},{"id":"refund","description":"취소·환불 담당"},{"id":"sales","description":"구매 상담 담당"}]}}''')
# 6·7·10장 차트용: 병렬 판단 속도(Mac·A4000), 14B Colab T4, API vs 로컬 40건, Jev-Omni 검증 기록
PRE = json.loads(r'''{"_about":"노트북이 그리는 미리 잰 숫자. 모든 값은 source에 적힌 파일에서 읽었어요.","_collected_at":"2026-09-24 06:22","parallel_decision":{"source":["~/jev특강/output/parallel-decision-rlcd-20260922/summary.json","~/jev특강/output/parallel-decision-rlcd-20260922/timings.csv","~/jev특강/output/parallel-decision-rlcd-20260922/README.md","~/jev특강/output/parallel-decision-rlcd-20260922/dense9b-a4000/summary.json"],"setup":"llama.cpp parallel-decision(14d04e7) · 같은 Qwen2.5-1.5B-Instruct GGUF Q4_K_M, Mac M5(Metal) / RTX A4000(CUDA). RLCD는 장치마다 모델 빌드가 달라요(Mac MLX 4bit / A4000 PyTorch BF16). 일반 JSON은 프롬프트로만 요청(형식 강제 아님).","timing":"첫 호출 뒤 3회 중앙값(ms). 모델 로딩 제외.","rows":[{"device":"mac","project":"llama","preset":"code_security","fields":28,"parallel_ms":611.44,"parallel_min_ms":328.81,"parallel_max_ms":1364.07,"json_ms":7479.74,"speedup":12.23,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":1},{"device":"mac","project":"llama","preset":"fintech_fraud","fields":28,"parallel_ms":414.43,"parallel_min_ms":397.38,"parallel_max_ms":432.57,"json_ms":7061.14,"speedup":17.04,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":1},{"device":"mac","project":"llama","preset":"high_cardinality_255","fields":4,"parallel_ms":582.2,"parallel_min_ms":502.61,"parallel_max_ms":613.46,"json_ms":5422.56,"speedup":9.31,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":0},{"device":"mac","project":"llama","preset":"support_triage","fields":28,"parallel_ms":416.79,"parallel_min_ms":399.72,"parallel_max_ms":453.29,"json_ms":6493.28,"speedup":15.58,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":2},{"device":"mac","project":"rlcd","preset":"code_security","fields":28,"parallel_ms":856.62,"parallel_min_ms":432.66,"parallel_max_ms":1789.33,"json_ms":6283.9,"speedup":7.34,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":1},{"device":"mac","project":"rlcd","preset":"fintech_fraud","fields":28,"parallel_ms":756.17,"parallel_min_ms":740.95,"parallel_max_ms":757.69,"json_ms":7942.01,"speedup":10.5,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":1},{"device":"mac","project":"rlcd","preset":"high_cardinality_255","fields":4,"parallel_ms":276.33,"parallel_min_ms":274.01,"parallel_max_ms":285.57,"json_ms":1713.63,"speedup":6.2,"parallel_schema_valid":true,"json_schema_valid":true,"json_missing_fields":0},{"device":"mac","project":"rlcd","preset":"support_triage","fields":28,"parallel_ms":784.38,"parallel_min_ms":761.76,"parallel_max_ms":951.02,"json_ms":8211.8,"speedup":10.47,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":1},{"device":"a4000","project":"llama","preset":"code_security","fields":28,"parallel_ms":107.42,"parallel_min_ms":99.23,"parallel_max_ms":110.24,"json_ms":2281.23,"speedup":21.24,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":1},{"device":"a4000","project":"llama","preset":"fintech_fraud","fields":28,"parallel_ms":95.28,"parallel_min_ms":94.46,"parallel_max_ms":104.06,"json_ms":2142.39,"speedup":22.48,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":1},{"device":"a4000","project":"llama","preset":"high_cardinality_255","fields":4,"parallel_ms":104.35,"parallel_min_ms":99.98,"parallel_max_ms":105.53,"json_ms":1129.62,"speedup":10.83,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":0},{"device":"a4000","project":"llama","preset":"support_triage","fields":28,"parallel_ms":126.01,"parallel_min_ms":110.44,"parallel_max_ms":139.5,"json_ms":2074.67,"speedup":16.46,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":2},{"device":"a4000","project":"rlcd","preset":"code_security","fields":28,"parallel_ms":174.19,"parallel_min_ms":168.34,"parallel_max_ms":181.91,"json_ms":13759.11,"speedup":78.99,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":28},{"device":"a4000","project":"rlcd","preset":"fintech_fraud","fields":28,"parallel_ms":175.73,"parallel_min_ms":169.44,"parallel_max_ms":179.36,"json_ms":12657.31,"speedup":72.03,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":28},{"device":"a4000","project":"rlcd","preset":"high_cardinality_255","fields":4,"parallel_ms":116.5,"parallel_min_ms":115.92,"parallel_max_ms":117.51,"json_ms":2231.19,"speedup":19.15,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":4},{"device":"a4000","project":"rlcd","preset":"support_triage","fields":28,"parallel_ms":181.08,"parallel_min_ms":180.96,"parallel_max_ms":189.13,"json_ms":12517.9,"speedup":69.13,"parallel_schema_valid":true,"json_schema_valid":false,"json_missing_fields":28}],"a4000_small_accuracy":[{"model":"Qwen2.5 1.5B","parallel_ms":103.7611000938341,"json_ms":2244.396800058894,"schema_json_ms":2364.031699951738,"parallel_exact":6,"json_exact":5,"cases":8},{"model":"Qwen3.5 9B","parallel_ms":734.5522000687197,"json_ms":8901.274500065483,"schema_json_ms":8846.948000020348,"parallel_exact":8,"json_exact":8,"cases":8}],"a4000_small_accuracy_note":"문의 8건 × 3필드를 모두 맞혀야 정답. 28필드 속도 프리셋에는 정답 라벨이 없어요."},"colab14b":{"source":["~/Downloads/20260922-043237-7fa17c (1)/visualization/comparison_all.json","~/Downloads/20260922-043237-7fa17c (1)/results.json"],"gpu":"Tesla T4","model":"Qwen/Qwen2.5-14B-Instruct","chunk_size":2,"rows":[{"key":"torch_semif_shared","method":"SemIf parallel","engine":"PyTorch","quantization":"NF4 / FP16","median_seconds":24.169244087000152,"min_seconds":24.039318276999893,"max_seconds":24.252677056000266,"measured_repeats":3,"warmups_excluded":1,"speed_fields":28,"correct_cases":8,"total_cases":8,"correct_fields":24,"total_fields":24,"valid_speed_runs":3,"scheduling":"chunk=2; fresh prefill per chunk"},{"key":"torch_rlcd_parallel","method":"RLCD parallel","engine":"PyTorch","quantization":"NF4 / FP16","median_seconds":31.73189418999982,"min_seconds":31.450114038000265,"max_seconds":32.06007517199987,"measured_repeats":3,"warmups_excluded":1,"speed_fields":28,"correct_cases":8,"total_cases":8,"correct_fields":24,"total_fields":24,"valid_speed_runs":3,"scheduling":"chunk=2; fresh prefill per chunk"},{"key":"torch_json_generate","method":"JSON generation","engine":"PyTorch","quantization":"NF4 / FP16","median_seconds":44.50349178100033,"min_seconds":44.31593759999988,"max_seconds":44.524087418999905,"measured_repeats":3,"warmups_excluded":1,"speed_fields":28,"correct_cases":8,"total_cases":8,"correct_fields":24,"total_fields":24,"valid_speed_runs":3,"scheduling":"prompt-only JSON; sequential generation"},{"key":"llama14b_parallel","method":"llama.cpp parallel","engine":"llama.cpp","quantization":"Q4_K_M","median_seconds":1.3948822470001687,"min_seconds":1.3514913700000761,"max_seconds":1.4150161659999867,"measured_repeats":3,"warmups_excluded":1,"speed_fields":28,"correct_cases":8,"total_cases":8,"correct_fields":24,"total_fields":24,"valid_speed_runs":3,"scheduling":"server decision scheduler; shared/static prefix reuse"},{"key":"llama14b_json","method":"llama.cpp JSON","engine":"llama.cpp","quantization":"Q4_K_M","median_seconds":27.00526188799995,"min_seconds":26.434224511999673,"max_seconds":29.178289202000087,"measured_repeats":3,"warmups_excluded":0,"speed_fields":28,"correct_cases":8,"total_cases":8,"correct_fields":24,"total_fields":24,"valid_speed_runs":3,"scheduling":"prompt-only JSON; cache_prompt=False"},{"key":"llama14b-structured_json","method":"llama.cpp schema JSON","engine":"llama.cpp","quantization":"Q4_K_M","median_seconds":26.950729208000666,"min_seconds":25.98589425800037,"max_seconds":27.153982791999624,"measured_repeats":3,"warmups_excluded":1,"speed_fields":28,"correct_cases":null,"total_cases":null,"correct_fields":null,"total_fields":null,"valid_speed_runs":3,"scheduling":"grammar-constrained JSON; no labeled-case evaluation in this artifact"}],"caveats":["28-field speed fixture is unlabeled; accuracy uses separate labeled support cases.","NF4 and GGUF use different weights/quantization, kernels, prompts and cache schedules: system-level comparison only.","Schema-constrained llama.cpp JSON has no labeled accuracy run here; N/A does not mean zero.","Warmups are excluded when flagged. Original llama.cpp ordinary JSON has no explicit warmup flag."],"notebook":"Colab_Local_16GB_Qwen14B.ipynb (14B NF4/FP16, 선택 llama.cpp CUDA 빌드)"},"semif_a4000":{"source":["~/jev특강/output/parallel-decision-rlcd-20260922/semif-a4000/comparison.csv","~/jev특강/output/parallel-decision-rlcd-20260922/semif-a4000/README.md"],"setup":"RTX A4000, NF4/FP16, 같은 프로세스·같은 가중치, 28항목 속도 + 문의 8건 정답","rows":[{"model":"qwen9b","method":"semif_shared","median_ms":2656.879899906926,"min_ms":2647.473999997601,"max_ms":2701.755999936722,"correct_cases":8,"correct_fields":24,"peak_allocated_mib":10344.13671875},{"model":"qwen9b","method":"rlcd","median_ms":1198.560000048019,"min_ms":1188.2332999957725,"max_ms":1254.3155000312254,"correct_cases":8,"correct_fields":24,"peak_allocated_mib":10449.8701171875},{"model":"qwen9b","method":"semif_direct","median_ms":10182.889200048521,"min_ms":10115.760099957697,"max_ms":10279.522700002417,"correct_cases":8,"correct_fields":24,"peak_allocated_mib":7667.09716796875},{"model":"qwen1p5b","method":"semif_shared","median_ms":520.9729000926018,"min_ms":509.6441999776289,"max_ms":523.1482000090182,"correct_cases":7,"correct_fields":23,"peak_allocated_mib":1595.447265625},{"model":"qwen1p5b","method":"rlcd","median_ms":232.21719998400658,"min_ms":222.85480005666614,"max_ms":232.43550001643598,"correct_cases":5,"correct_fields":20,"peak_allocated_mib":1939.93017578125},{"model":"qwen1p5b","method":"semif_direct","median_ms":2155.3123000776395,"min_ms":2153.5570999840274,"max_ms":2194.0032000420615,"correct_cases":7,"correct_fields":23,"peak_allocated_mib":1154.86767578125}]},"rlcr_vs_rlcd":{"source":["~/jev특강/output/parallel-decision-rlcd-20260922/rlcr-vs-rlcd-a4000/summary.json","~/jev특강/output/parallel-decision-rlcd-20260922/rlcr-vs-rlcd-a4000/predictions.csv","~/jev특강/output/parallel-decision-rlcd-20260922/rlcr-vs-rlcd-a4000/README.md"],"setup":"같은 RLCR 7B 가중치(NF4/FP16), 문의 8건 × 3필드, RTX A4000","rlcr":{"field_correct":24,"field_total":24,"case_correct":8,"case_total":8,"median_case_ms":11850.885450025089},"rlcd":{"field_correct":22,"field_total":24,"case_correct":6,"case_total":8,"median_case_ms":178.32930001895875},"rlcd_wrong":[{"case":"case-3","field":"category","expected":"cancellation","rlcd_prediction":"billing","rlcd_confidence":0.9599,"rlcr_prediction":"cancellation","rlcr_confidence":0.7},{"case":"case-7","field":"category","expected":"cancellation","rlcd_prediction":"billing","rlcd_confidence":0.8325,"rlcr_prediction":"cancellation","rlcr_confidence":0.8}]},"api_vs_local":{"source":["~/jev특강/final/03_유사프로젝트/evidence/api-vs-local/summary.json","~/jev특강/final/03_유사프로젝트/evidence/api-vs-local/summary.md","~/jev특강/final/03_유사프로젝트/code/evaluate.mjs","~/jev특강/final/03_유사프로젝트/code/data/routing_bodies.jsonl","~/jev특강/final/02_라우터/code/results/router-jev-2026-09-22T20-05-42-399Z.json","~/jev특강/final/03_유사프로젝트/evidence/api-vs-local/out_decider.jsonl","~/jev특강/final/03_유사프로젝트/evidence/api-vs-local/out_openjev.jsonl"],"setup":"2강 라우터 메시지 40건, 같은 정책 코드(policy.mjs)로 채점. 행동 일치 = 정책이 고른 행동이 정답과 같은 비율","table":[{"engine":"Jev API (TypeSafe)","where":"클라우드","model":"jev-1.13.0","n":40,"actionAcc":0.925,"routeAcc":0.975,"p50ms":247,"p95ms":559,"costPer1k":0.03480855},{"engine":"decider-2b (A4000 CUDA)","where":"내 GPU 서버","model":"decider-v10","n":40,"actionAcc":0.725,"routeAcc":0.775,"p50ms":214,"p95ms":224,"costPer1k":0},{"engine":"decider-2b (Mac MPS)","where":"내 노트북","model":"decider-v10","n":40,"actionAcc":0.725,"routeAcc":0.775,"p50ms":1880,"p95ms":2314,"costPer1k":0},{"engine":"OpenJev Qwen3.5-4B (A4000)","where":"내 GPU 서버","model":"Qwen/Qwen3.5-4B direct readout","n":40,"actionAcc":0.675,"routeAcc":0.775,"p50ms":572,"p95ms":608,"costPer1k":0},{"engine":"Qwen3.5-4B 생성(JSON) (Mac Ollama)","where":"내 노트북","model":"qwen3.5:4b","n":40,"actionAcc":0.925,"routeAcc":1,"p50ms":4219,"p95ms":4852,"costPer1k":0}],"route_calibration":{"Jev API (TypeSafe)":[[1,true],[1,true],[1,true],[1,true],[1,true],[1,true],[1,true],[1,true],[1,true],[1,true],[1,true],[1,true],[0.94,true],[0.79,true],[0.93,true],[0.98,true],[0.84,true],[0.96,true],[0.8,true],[0.93,true],[0.94,true],[0.96,true],[0.96,true],[0.8,true],[0.73,true],[0.53,false],[0.8,true],[0.88,true],[0.95,true],[0.97,true],[0.93,true],[0.87,true],[0.88,true],[0.86,true],[0.89,true],[1,true],[0.96,true],[1,true],[0.71,true],[0.63,true]],"decider-2b (A4000 CUDA)":[[0.9864,true],[0.9687,true],[0.9277,true],[0.9883,true],[0.8221,true],[0.9948,true],[0.9573,true],[0.8676,true],[0.9731,true],[0.9062,true],[0.8755,true],[0.9157,true],[0.9169,true],[0.8173,true],[0.8667,true],[0.8138,true],[0.81,true],[0.8647,true],[0.7123,false],[0.4402,true],[0.4176,false],[0.6761,true],[0.6712,true],[0.7429,true],[0.7149,true],[0.4549,false],[0.4954,false],[0.8026,true],[0.9072,true],[0.882,true],[0.5407,true],[0.6287,false],[0.5288,false],[0.6603,true],[0.7065,true],[0.9627,true],[0.4641,false],[0.9545,true],[0.7298,false],[0.5544,false]],"decider-2b (Mac MPS)":[[0.9874,true],[0.9672,true],[0.9272,true],[0.9885,true],[0.8157,true],[0.9948,true],[0.9584,true],[0.8722,true],[0.9727,true],[0.9047,true],[0.8798,true],[0.9178,true],[0.9127,true],[0.8137,true],[0.875,true],[0.8132,true],[0.7995,true],[0.8724,true],[0.7213,false],[0.4328,true],[0.3856,false],[0.6904,true],[0.6578,true],[0.7484,true],[0.7053,true],[0.4647,false],[0.496,false],[0.8283,true],[0.9082,true],[0.885,true],[0.5372,true],[0.6248,false],[0.5237,false],[0.6679,true],[0.7168,true],[0.9631,true],[0.4713,false],[0.958,true],[0.7332,false],[0.566,false]],"OpenJev Qwen3.5-4B (A4000)":[[0.999,true],[0.9988,true],[0.9547,true],[0.9971,true],[0.8736,true],[0.9992,true],[0.9613,true],[0.9972,true],[0.9536,true],[0.9984,true],[0.9656,true],[0.9943,true],[0.7434,true],[0.8065,true],[0.8823,true],[0.8747,true],[0.974,true],[0.9017,true],[0.9175,false],[0.5601,true],[0.8048,true],[0.9108,true],[0.8613,true],[0.8806,true],[0.5682,true],[0.5497,false],[0.5785,true],[0.8457,true],[0.7091,true],[0.7758,true],[0.7716,false],[0.9516,false],[0.7552,false],[0.7533,false],[0.5352,false],[0.9937,true],[0.5557,true],[0.9883,true],[0.8524,false],[0.9689,false]]},"route_calibration_note":"route 질문(chat/task/unclear/spam) 하나만: [고른 답의 확률, 정답 여부]. 40건이라 참고용"},"jev_omni":{"source":["https://huggingface.co/akhilaaa3/Jev-Omni (README.md, unified/verification_unified.json, jev_omni.py, load_model.py)","HfApi.list_repo_tree / model_info (메타데이터만, 가중치 받지 않음)"],"fetched_at":"2026-09-24","repo_sha":"c050d51354147985d13286cf4acf90f562f2c631","license":"apache-2.0","base_model":"google/gemma-4-12B-it","files_bytes":{"기타 작은 파일":32232425,"assets":20394094,"backbone":47629527966,"head.pt":3966079,"unified":23951748446},"repo_total_bytes":71637869010,"gemma_total_bytes":23951776979,"gemma_gated":false,"readme_claims":{"decisionbench_medium_accuracy":0.8757,"decisionbench_medium_ece":0.04,"jevbench_matched_accuracy":0.8615,"h200_text_ms":83,"fp32_weights_gb":50,"max_options_tested":20},"verification_cases":[{"state":"The meeting starts at 10 AM. It is now 9 AM.","question":"Has the meeting started?","options":["Yes","No"],"probabilities":[0.0001,0.9999]},{"state":"Customer: I was charged twice.\nAgent: I've refunded $29 to your card.\nCustomer: Got it. All sorted, thanks!","question":"Was the issue actually resolved?","options":["Yes","No"],"probabilities":[0.9399,0.0601]},{"state":"A fair six-sided die is rolled once.","question":"Which number comes up?","options":["1","2","3","4","5","6"],"probabilities":[0.2341,0.0929,0.0437,0.089,0.0811,0.4592]},{"state":"An urn holds one blue, one yellow, one red and one green marble. One is drawn without looking.","question":"Which marble is drawn?","options":["Blue","Yellow","Red","Green"],"probabilities":[0.413,0.1262,0.0793,0.3816]}],"verification_worst_abs_diff":0.01936584711074829},"mac_fp16_check":{"source":"assets/measure_local.py --dtype float16 (강사 Mac M5, MPS, 2.13.0)","results":{"route-gpu":{"option_ids":["a4000","main"],"probabilities":[0.9944,0.0056]},"route-vague":{"option_ids":["a4000","main"],"probabilities":[0.3961,0.6039]},"classify-fail":{"option_ids":["transient","code","permission","unknown"],"probabilities":[0.0243,0.0029,0.0026,0.9701]},"route-mail":{"option_ids":["a4000","todobot","main"],"probabilities":[0.0024,0.9973,0.0003]},"gate-danger":{"option_ids":["proceed","ask"],"probabilities":[0.0038,0.9962]},"my-test":{"option_ids":["shipping","refund","sales"],"probabilities":[0.003,0.996,0.0009]}}}}''')
# GPU가 없을 때 쓰는 값: 강사 Mac(M5, MPS, bfloat16)에서 assets/measure_local.py로 잰 위젯 데이터
FALLBACK = json.loads(r'''{"_note":"강사 Mac에서 잰 값. Colab T4(float16)와 소수점 아래가 다를 수 있어요.","machine":{"platform":"macOS-26.5-arm64-arm-64bit-Mach-O","machine":"arm64","python":"3.14.6","device":"mps","dtype":"bfloat16","torch":"2.13.0"},"metadata":{"source":"Qwen/Qwen3.5-4B","revision":"851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a","dtype":"bfloat16","device":"mps","torch_version":"2.13.0","transformers_version":"5.17.0"},"load_seconds":4.783098220825195,"results":{"route-gpu":{"id":"route-gpu","row":{"id":"route-gpu","state":"A dashboard routing request. Message: 'A4000 GPU server experiment - continue processing it.' Facts: [a4000] Windows GPU experiment worker, [main] general conversation session.","question":"Which session should handle the message?","options":[{"id":"a4000","description":"Windows GPU experiment worker session"},{"id":"main","description":"General conversation session"}]},"system":"Apply the supplied criterion to the supplied evidence. Choose exactly one listed option. Respond with only its uppercase letter, with no explanation or reasoning.","payload":"{\"evidence\": \"A dashboard routing request. Message: 'A4000 GPU server experiment - continue processing it.' Facts: [a4000] Windows GPU experiment worker, [main] general conversation session.\", \"criterion\": \"Which session should handle the message?\", \"options\": [{\"letter\": \"A\", \"description\": \"Windows GPU experiment worker session\"}, {\"letter\": \"B\", \"description\": \"General conversation session\"}]}","prompt":"<|im_start|>system\nApply the supplied criterion to the supplied evidence. Choose exactly one listed option. Respond with only its uppercase letter, with no explanation or reasoning.<|im_end|>\n<|im_start|>user\n{\"evidence\": \"A dashboard routing request. Message: 'A4000 GPU server experiment - continue processing it.' Facts: [a4000] Windows GPU experiment worker, [main] general conversation session.\", \"criterion\": \"Which session should handle the message?\", \"options\": [{\"letter\": \"A\", \"description\": \"Windows GPU experiment worker session\"}, {\"letter\": \"B\", \"description\": \"General conversation session\"}]}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n","prompt_sha256":"37d0a88b8fc984e09d830ac8abfb8d61f5cb29e4d5656d0b629c26f44e081013","tokens":[[248045,0,12,"template",null],[8678,12,18,"template",null],[198,18,19,"template",null],[27567,19,24,"system",null],[279,24,28,"system",null],[16713,28,37,"system",null],[34765,37,47,"system",null],[310,47,50,"system",null],[279,50,54,"system",null],[16713,54,63,"system",null],[5721,63,72,"system",null],[13,72,73,"system",null],[21513,73,80,"system",null],[6681,80,88,"system",null],[799,88,92,"system",null],[9711,92,99,"system",null],[2904,99,106,"system",null],[13,106,107,"system",null],[38224,107,115,"system",null],[440,115,120,"system",null],[1132,120,125,"system",null],[1141,125,129,"system",null],[38175,129,139,"system",null],[6321,139,146,"system",null],[11,146,147,"system",null],[440,147,152,"system",null],[874,152,155,"system",null],[15673,155,167,"system",null],[466,167,170,"system",null],[31626,170,180,"system",null],[13,180,181,"system",null],[248046,181,191,"template",null],[198,191,192,"template",null],[248045,192,204,"template",null],[846,204,208,"template",null],[198,208,209,"template",null],[4754,209,211,"json",null],[68,211,212,"json",null],[26590,212,219,"json",null],[763,219,221,"json",null],[328,221,223,"json",null],[32,223,224,"evidence",null],[26103,224,234,"evidence",null],[28109,234,242,"evidence",null],[1622,242,250,"evidence",null],[13,250,251,"evidence",null],[4701,251,259,"evidence",null],[25,259,260,"json",null],[359,260,262,"evidence",null],[32,262,263,"evidence",null],[19,263,264,"evidence",null],[15,264,265,"evidence",null],[15,265,266,"evidence",null],[15,266,267,"evidence",null],[21966,267,271,"evidence",null],[3421,271,278,"evidence",null],[9064,278,289,"evidence",null],[471,289,291,"evidence",null],[2962,291,300,"evidence",null],[8427,300,311,"evidence",null],[424,311,314,"evidence",null],[3058,314,316,"evidence",null],[43483,316,322,"evidence",null],[25,322,323,"json",null],[498,323,325,"json",null],[64,325,326,"evidence",null],[19,326,327,"evidence",null],[15,327,328,"evidence",null],[15,328,329,"evidence",null],[15,329,330,"evidence",null],[60,330,331,"json",null],[5342,331,339,"evidence",null],[21966,339,343,"evidence",null],[9064,343,354,"evidence",null],[11525,354,361,"evidence",null],[11,361,362,"json",null],[498,362,364,"json",null],[3689,364,368,"evidence",null],[60,368,369,"json",null],[4437,369,377,"evidence",null],[10125,377,390,"evidence",null],[3669,390,398,"evidence",null],[10152,398,401,"evidence",null],[328,401,403,"json",null],[66,403,404,"json",null],[11981,404,412,"json",null],[763,412,414,"json",null],[328,414,416,"json",null],[22365,416,421,"criterion",null],[3669,421,429,"criterion",null],[1220,429,436,"criterion",null],[3579,436,443,"criterion",null],[279,443,447,"criterion",null],[1876,447,455,"criterion",null],[29993,455,458,"criterion",null],[328,458,460,"json",null],[2782,460,467,"json",null],[763,467,469,"json",null],[59670,469,473,"json",null],[9172,473,479,"json",null],[763,479,481,"json",null],[328,481,483,"json",null],[32,483,484,"letter","A"],[487,484,486,"json",null],[328,486,488,"json",null],[4532,488,499,"json",null],[763,499,501,"json",null],[328,501,503,"json",null],[12782,503,510,"option","A"],[21966,510,514,"option","A"],[9064,514,525,"option","A"],[11525,525,532,"option","A"],[3669,532,540,"option","A"],[13933,540,543,"json",null],[5046,543,546,"json",null],[9172,546,552,"json",null],[763,552,554,"json",null],[328,554,556,"json",null],[33,556,557,"letter","B"],[487,557,559,"json",null],[328,559,561,"json",null],[4532,561,572,"json",null],[763,572,574,"json",null],[328,574,576,"json",null],[14965,576,583,"option","B"],[10125,583,596,"option","B"],[3669,596,604,"option","B"],[8934,604,606,"json",null],[13587,606,608,"json",null],[248046,608,618,"template",null],[198,618,619,"template",null],[248045,619,631,"template",null],[74455,631,640,"template",null],[198,640,641,"gen",null],[248068,641,648,"gen",null],[271,648,650,"gen",null],[248069,650,658,"gen",null],[271,658,660,"gen",null]],"slots":[["A",32],["B",33]],"option_ids":["a4000","main"],"option_logits":[26.25,20.875],"probabilities":[0.9953904278206259,0.004609572179374208],"forward_seconds":0.568369249929674,"input_tokens":138,"top_tokens":[[32,"A",0.9951055645942688],[33,"B",0.004608254414051771],[248068,"<think>",0.00010837570880539715],[34,"C",5.119305933476426e-05],[248046,"<|im_end|>",2.0047487851115875e-05],[28180,"\"A",1.2943632100359537e-05],[35,"D",7.37505934012006e-06],[4205,"(A",6.114138614066178e-06],[760,"The",5.06879905515234e-06],[36,"E",3.074382220802363e-06]],"allowed_mass":0.9997138977050781,"readout_check_max_abs_diff":0.0,"lens":[[30.875,29.125],[2.453,1.359],[3.062,1.406],[7.156,1.977],[6.344,0.229],[3.953,0.633],[0.002,-1.438],[3.391,-0.056],[1.828,0.312],[2.391,0.867],[0.105,0.001],[1.961,1.133],[0.715,1.492],[-0.695,-1.219],[0.965,-0.965],[2.344,0.785],[2.156,1.922],[2.219,2.375],[4.125,3.469],[2.25,1.914],[4.469,2.078],[6.906,3.312],[7.25,5.281],[6.438,4.156],[8.812,4.0],[7.938,3.656],[9.75,3.406],[10.688,4.094],[12.625,7.5],[14.812,8.625],[17.5,11.0],[22.75,12.0],[26.25,20.875]],"layer_types":["linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention"],"perms":[{"order":["a4000","main"],"probs":{"a4000":0.99539,"main":0.00461},"logits":{"a4000":26.25,"main":20.875}},{"order":["main","a4000"],"probs":{"main":0.132964,"a4000":0.867036},"logits":{"main":23.625,"a4000":25.5}}],"model":{"source":"Qwen/Qwen3.5-4B","revision":"851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a","dtype":"bfloat16","device":"mps","torch_version":"2.13.0","transformers_version":"5.17.0"},"measured_at":"2026-09-24 05:39"},"route-vague":{"id":"route-vague","row":{"id":"route-vague","state":"A dashboard routing request. Message: 'just handle this for me' (no destination indicated). Facts: [a4000] Windows GPU experiment worker, [main] general conversation session.","question":"Which session should handle the message?","options":[{"id":"a4000","description":"Windows GPU experiment worker session"},{"id":"main","description":"General conversation session"}]},"system":"Apply the supplied criterion to the supplied evidence. Choose exactly one listed option. Respond with only its uppercase letter, with no explanation or reasoning.","payload":"{\"evidence\": \"A dashboard routing request. Message: 'just handle this for me' (no destination indicated). Facts: [a4000] Windows GPU experiment worker, [main] general conversation session.\", \"criterion\": \"Which session should handle the message?\", \"options\": [{\"letter\": \"A\", \"description\": \"Windows GPU experiment worker session\"}, {\"letter\": \"B\", \"description\": \"General conversation session\"}]}","prompt":"<|im_start|>system\nApply the supplied criterion to the supplied evidence. Choose exactly one listed option. Respond with only its uppercase letter, with no explanation or reasoning.<|im_end|>\n<|im_start|>user\n{\"evidence\": \"A dashboard routing request. Message: 'just handle this for me' (no destination indicated). Facts: [a4000] Windows GPU experiment worker, [main] general conversation session.\", \"criterion\": \"Which session should handle the message?\", \"options\": [{\"letter\": \"A\", \"description\": \"Windows GPU experiment worker session\"}, {\"letter\": \"B\", \"description\": \"General conversation session\"}]}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n","prompt_sha256":"9ffac973d3eb178706c7d013bb27d80c513f2a817a1a1e552f4bffd791554df9","tokens":[[248045,0,12,"template",null],[8678,12,18,"template",null],[198,18,19,"template",null],[27567,19,24,"system",null],[279,24,28,"system",null],[16713,28,37,"system",null],[34765,37,47,"system",null],[310,47,50,"system",null],[279,50,54,"system",null],[16713,54,63,"system",null],[5721,63,72,"system",null],[13,72,73,"system",null],[21513,73,80,"system",null],[6681,80,88,"system",null],[799,88,92,"system",null],[9711,92,99,"system",null],[2904,99,106,"system",null],[13,106,107,"system",null],[38224,107,115,"system",null],[440,115,120,"system",null],[1132,120,125,"system",null],[1141,125,129,"system",null],[38175,129,139,"system",null],[6321,139,146,"system",null],[11,146,147,"system",null],[440,147,152,"system",null],[874,152,155,"system",null],[15673,155,167,"system",null],[466,167,170,"system",null],[31626,170,180,"system",null],[13,180,181,"system",null],[248046,181,191,"template",null],[198,191,192,"template",null],[248045,192,204,"template",null],[846,204,208,"template",null],[198,208,209,"template",null],[4754,209,211,"json",null],[68,211,212,"json",null],[26590,212,219,"json",null],[763,219,221,"json",null],[328,221,223,"json",null],[32,223,224,"evidence",null],[26103,224,234,"evidence",null],[28109,234,242,"evidence",null],[1622,242,250,"evidence",null],[13,250,251,"evidence",null],[4701,251,259,"evidence",null],[25,259,260,"json",null],[359,260,262,"evidence",null],[4111,262,266,"evidence",null],[3579,266,273,"evidence",null],[411,273,278,"evidence",null],[364,278,282,"evidence",null],[728,282,285,"evidence",null],[6,285,286,"evidence",null],[318,286,288,"evidence",null],[2083,288,290,"evidence",null],[8833,290,302,"evidence",null],[15838,302,312,"evidence",null],[553,312,314,"evidence",null],[43483,314,320,"evidence",null],[25,320,321,"json",null],[498,321,323,"json",null],[64,323,324,"evidence",null],[19,324,325,"evidence",null],[15,325,326,"evidence",null],[15,326,327,"evidence",null],[15,327,328,"evidence",null],[60,328,329,"json",null],[5342,329,337,"evidence",null],[21966,337,341,"evidence",null],[9064,341,352,"evidence",null],[11525,352,359,"evidence",null],[11,359,360,"json",null],[498,360,362,"json",null],[3689,362,366,"evidence",null],[60,366,367,"json",null],[4437,367,375,"evidence",null],[10125,375,388,"evidence",null],[3669,388,396,"evidence",null],[10152,396,399,"evidence",null],[328,399,401,"json",null],[66,401,402,"json",null],[11981,402,410,"json",null],[763,410,412,"json",null],[328,412,414,"json",null],[22365,414,419,"criterion",null],[3669,419,427,"criterion",null],[1220,427,434,"criterion",null],[3579,434,441,"criterion",null],[279,441,445,"criterion",null],[1876,445,453,"criterion",null],[29993,453,456,"criterion",null],[328,456,458,"json",null],[2782,458,465,"json",null],[763,465,467,"json",null],[59670,467,471,"json",null],[9172,471,477,"json",null],[763,477,479,"json",null],[328,479,481,"json",null],[32,481,482,"letter","A"],[487,482,484,"json",null],[328,484,486,"json",null],[4532,486,497,"json",null],[763,497,499,"json",null],[328,499,501,"json",null],[12782,501,508,"option","A"],[21966,508,512,"option","A"],[9064,512,523,"option","A"],[11525,523,530,"option","A"],[3669,530,538,"option","A"],[13933,538,541,"json",null],[5046,541,544,"json",null],[9172,544,550,"json",null],[763,550,552,"json",null],[328,552,554,"json",null],[33,554,555,"letter","B"],[487,555,557,"json",null],[328,557,559,"json",null],[4532,559,570,"json",null],[763,570,572,"json",null],[328,572,574,"json",null],[14965,574,581,"option","B"],[10125,581,594,"option","B"],[3669,594,602,"option","B"],[8934,602,604,"json",null],[13587,604,606,"json",null],[248046,606,616,"template",null],[198,616,617,"template",null],[248045,617,629,"template",null],[74455,629,638,"template",null],[198,638,639,"gen",null],[248068,639,646,"gen",null],[271,646,648,"gen",null],[248069,648,656,"gen",null],[271,656,658,"gen",null]],"slots":[["A",32],["B",33]],"option_ids":["a4000","main"],"option_logits":[24.0,24.375],"probabilities":[0.4073334000459302,0.5926665999540697],"forward_seconds":0.5891068750061095,"input_tokens":136,"top_tokens":[[33,"B",0.5921762585639954],[32,"A",0.40699639916419983],[34,"C",0.00032752356491982937],[248068,"<think>",0.00019865308422595263],[35,"D",2.8618691430892795e-05],[248046,"<|im_end|>",1.1930044820473995e-05],[36,"E",1.0528227903705556e-05],[760,"The",8.728206921659876e-06],[51,"T",8.199392141250428e-06],[61442,"\"B",6.79753475196776e-06]],"allowed_mass":0.9991725087165833,"readout_check_max_abs_diff":0.0,"lens":[[30.875,29.125],[2.469,1.375],[3.078,1.492],[7.281,2.094],[6.438,0.232],[3.984,0.547],[0.03,-1.469],[3.281,-0.178],[1.727,0.095],[2.156,0.754],[-0.118,-0.09],[1.828,1.039],[0.648,1.305],[-0.777,-1.391],[1.25,-0.848],[2.469,0.598],[2.328,1.859],[2.297,2.156],[3.438,3.391],[1.281,1.766],[2.969,2.375],[5.656,3.734],[6.312,5.625],[5.094,4.688],[6.594,8.562],[5.688,7.125],[5.938,8.25],[7.0,7.812],[10.812,10.812],[12.5,13.25],[15.25,17.125],[18.625,20.875],[24.0,24.375]],"layer_types":["linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention"],"perms":[{"order":["a4000","main"],"probs":{"a4000":0.407333,"main":0.592667},"logits":{"a4000":24.0,"main":24.375}},{"order":["main","a4000"],"probs":{"main":0.932453,"a4000":0.067547},"logits":{"main":25.0,"a4000":22.375}}],"model":{"source":"Qwen/Qwen3.5-4B","revision":"851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a","dtype":"bfloat16","device":"mps","torch_version":"2.13.0","transformers_version":"5.17.0"},"measured_at":"2026-09-24 05:39"},"classify-fail":{"id":"classify-fail","row":{"id":"classify-fail","state":"A command failed while running. No error text or logs are available in the state.","question":"What is the failure classification?","options":[{"id":"transient","description":"Temporary error - retry once"},{"id":"code","description":"Code defect - needs a fix"},{"id":"permission","description":"Permission or auth issue"},{"id":"unknown","description":"Cannot determine - ask the user"}]},"system":"Apply the supplied criterion to the supplied evidence. Choose exactly one listed option. Respond with only its uppercase letter, with no explanation or reasoning.","payload":"{\"evidence\": \"A command failed while running. No error text or logs are available in the state.\", \"criterion\": \"What is the failure classification?\", \"options\": [{\"letter\": \"A\", \"description\": \"Temporary error - retry once\"}, {\"letter\": \"B\", \"description\": \"Code defect - needs a fix\"}, {\"letter\": \"C\", \"description\": \"Permission or auth issue\"}, {\"letter\": \"D\", \"description\": \"Cannot determine - ask the user\"}]}","prompt":"<|im_start|>system\nApply the supplied criterion to the supplied evidence. Choose exactly one listed option. Respond with only its uppercase letter, with no explanation or reasoning.<|im_end|>\n<|im_start|>user\n{\"evidence\": \"A command failed while running. No error text or logs are available in the state.\", \"criterion\": \"What is the failure classification?\", \"options\": [{\"letter\": \"A\", \"description\": \"Temporary error - retry once\"}, {\"letter\": \"B\", \"description\": \"Code defect - needs a fix\"}, {\"letter\": \"C\", \"description\": \"Permission or auth issue\"}, {\"letter\": \"D\", \"description\": \"Cannot determine - ask the user\"}]}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n","prompt_sha256":"8e7c19726177d2431f1077f16b36ad347788c6e6c2b73a298ca73e70037fc77d","tokens":[[248045,0,12,"template",null],[8678,12,18,"template",null],[198,18,19,"template",null],[27567,19,24,"system",null],[279,24,28,"system",null],[16713,28,37,"system",null],[34765,37,47,"system",null],[310,47,50,"system",null],[279,50,54,"system",null],[16713,54,63,"system",null],[5721,63,72,"system",null],[13,72,73,"system",null],[21513,73,80,"system",null],[6681,80,88,"system",null],[799,88,92,"system",null],[9711,92,99,"system",null],[2904,99,106,"system",null],[13,106,107,"system",null],[38224,107,115,"system",null],[440,115,120,"system",null],[1132,120,125,"system",null],[1141,125,129,"system",null],[38175,129,139,"system",null],[6321,139,146,"system",null],[11,146,147,"system",null],[440,147,152,"system",null],[874,152,155,"system",null],[15673,155,167,"system",null],[466,167,170,"system",null],[31626,170,180,"system",null],[13,180,181,"system",null],[248046,181,191,"template",null],[198,191,192,"template",null],[248045,192,204,"template",null],[846,204,208,"template",null],[198,208,209,"template",null],[4754,209,211,"json",null],[68,211,212,"json",null],[26590,212,219,"json",null],[763,219,221,"json",null],[328,221,223,"json",null],[32,223,224,"evidence",null],[3108,224,232,"evidence",null],[4490,232,239,"evidence",null],[1345,239,245,"evidence",null],[4162,245,253,"evidence",null],[13,253,254,"evidence",null],[2233,254,257,"evidence",null],[1412,257,263,"evidence",null],[1414,263,268,"evidence",null],[466,268,271,"evidence",null],[17872,271,276,"evidence",null],[513,276,280,"evidence",null],[2420,280,290,"evidence",null],[303,290,293,"evidence",null],[279,293,297,"evidence",null],[1528,297,303,"evidence",null],[10152,303,306,"evidence",null],[328,306,308,"json",null],[66,308,309,"json",null],[11981,309,317,"json",null],[763,317,319,"json",null],[328,319,321,"json",null],[3710,321,325,"criterion",null],[369,325,328,"criterion",null],[279,328,332,"criterion",null],[7652,332,340,"criterion",null],[23104,340,355,"criterion",null],[29993,355,358,"criterion",null],[328,358,360,"json",null],[2782,360,367,"json",null],[763,367,369,"json",null],[59670,369,373,"json",null],[9172,373,379,"json",null],[763,379,381,"json",null],[328,381,383,"json",null],[32,383,384,"letter","A"],[487,384,386,"json",null],[328,386,388,"json",null],[4532,388,399,"json",null],[763,399,401,"json",null],[328,401,403,"json",null],[57357,403,412,"option","A"],[1412,412,418,"option","A"],[471,418,420,"option","A"],[21979,420,426,"option","A"],[2957,426,431,"option","A"],[13933,431,434,"json",null],[5046,434,437,"json",null],[9172,437,443,"json",null],[763,443,445,"json",null],[328,445,447,"json",null],[33,447,448,"letter","B"],[487,448,450,"json",null],[328,450,452,"json",null],[4532,452,463,"json",null],[763,463,465,"json",null],[328,465,467,"json",null],[2010,467,471,"option","B"],[21531,471,478,"option","B"],[471,478,480,"option","B"],[3749,480,486,"option","B"],[264,486,488,"option","B"],[4884,488,492,"option","B"],[13933,492,495,"json",null],[5046,495,498,"json",null],[9172,498,504,"json",null],[763,504,506,"json",null],[328,506,508,"json",null],[34,508,509,"letter","C"],[487,509,511,"json",null],[328,511,513,"json",null],[4532,513,524,"json",null],[763,524,526,"json",null],[328,526,528,"json",null],[14532,528,538,"option","C"],[466,538,541,"option","C"],[4030,541,546,"option","C"],[4125,546,552,"option","C"],[13933,552,555,"json",null],[5046,555,558,"json",null],[9172,558,564,"json",null],[763,564,566,"json",null],[328,566,568,"json",null],[35,568,569,"letter","D"],[487,569,571,"json",null],[328,571,573,"json",null],[4532,573,584,"json",null],[763,584,586,"json",null],[328,586,588,"json",null],[16928,588,594,"option","D"],[7995,594,604,"option","D"],[471,604,606,"option","D"],[2466,606,610,"option","D"],[279,610,614,"option","D"],[1156,614,619,"option","D"],[8934,619,621,"json",null],[13587,621,623,"json",null],[248046,623,633,"template",null],[198,633,634,"template",null],[248045,634,646,"template",null],[74455,646,655,"template",null],[198,655,656,"gen",null],[248068,656,663,"gen",null],[271,663,665,"gen",null],[248069,665,673,"gen",null],[271,673,675,"gen",null]],"slots":[["A",32],["B",33],["C",34],["D",35]],"option_ids":["transient","code","permission","unknown"],"option_logits":[22.25,20.25,20.0,25.875],"probabilities":[0.025796161329496397,0.00349113079994475,0.0027188954008016716,0.9679938124697571],"forward_seconds":0.5980780000099912,"input_tokens":147,"top_tokens":[[35,"D",0.9677141308784485],[32,"A",0.02578870952129364],[33,"B",0.003490123199298978],[34,"C",0.00271811056882143],[248068,"<think>",6.392379873432219e-05],[36,"E",3.642268347903155e-05],[59355,"\"D",3.019546602445189e-05],[40,"I",1.6162468455149792e-05],[5263,"(D",1.1824714420072269e-05],[37,"F",5.945839802734554e-06]],"allowed_mass":0.9997119903564453,"readout_check_max_abs_diff":0.0,"lens":[[30.875,29.125,29.5,26.375],[2.375,1.242,2.281,1.688],[3.109,1.398,2.438,2.484],[7.438,1.906,4.969,4.062],[6.438,0.145,4.094,3.578],[4.25,0.516,5.188,3.078],[-0.211,-1.766,2.047,0.24],[2.609,-0.186,1.391,1.797],[1.477,0.785,0.922,3.984],[2.016,0.789,-0.314,4.0],[-0.195,0.03,-0.543,2.594],[1.602,1.555,0.945,3.688],[0.045,1.102,1.164,3.656],[-0.758,-1.539,0.307,1.156],[0.773,-1.188,0.621,1.609],[1.695,0.131,1.289,2.969],[1.828,1.852,1.609,4.688],[1.781,1.867,1.969,4.312],[2.328,2.672,3.031,5.062],[1.289,1.0,3.016,3.375],[2.078,1.516,2.172,1.688],[4.625,1.867,2.625,2.656],[5.75,5.625,5.094,5.562],[4.438,4.375,4.125,4.438],[3.703,3.734,2.594,8.75],[3.531,3.656,3.312,8.0],[2.844,3.078,2.672,9.125],[3.656,3.266,3.234,7.844],[6.75,5.781,5.562,9.875],[7.625,6.625,6.312,12.438],[9.438,8.75,8.312,15.875],[12.438,10.25,10.312,19.75],[22.25,20.25,20.0,25.875]],"layer_types":["linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention"],"perms":[{"order":["transient","code","permission","unknown"],"probs":{"transient":0.025796,"code":0.003491,"permission":0.002719,"unknown":0.967994},"logits":{"transient":22.25,"code":20.25,"permission":20.0,"unknown":25.875}},{"order":["transient","code","unknown","permission"],"probs":{"transient":0.037109,"code":0.002372,"unknown":0.957066,"permission":0.003452},"logits":{"transient":22.375,"code":19.625,"unknown":25.625,"permission":20.0}},{"order":["transient","permission","code","unknown"],"probs":{"transient":0.029142,"permission":0.001863,"code":0.003944,"unknown":0.965051},"logits":{"transient":22.25,"permission":19.5,"code":20.25,"unknown":25.75}},{"order":["transient","permission","unknown","code"],"probs":{"transient":0.010974,"permission":0.000376,"unknown":0.987855,"code":0.000795},"logits":{"transient":21.75,"permission":18.375,"unknown":26.25,"code":19.125}},{"order":["transient","unknown","code","permission"],"probs":{"transient":0.025846,"unknown":0.969875,"code":0.003498,"permission":0.00078},"logits":{"transient":22.625,"unknown":26.25,"code":20.625,"permission":19.125}},{"order":["transient","unknown","permission","code"],"probs":{"transient":0.025834,"unknown":0.969429,"permission":0.003085,"code":0.001652},"logits":{"transient":22.75,"unknown":26.375,"permission":20.625,"code":20.0}},{"order":["code","transient","permission","unknown"],"probs":{"code":0.009671,"transient":0.001904,"permission":0.001904,"unknown":0.98652},"logits":{"code":21.625,"transient":20.0,"permission":20.0,"unknown":26.25}},{"order":["code","transient","unknown","permission"],"probs":{"code":0.02909,"transient":0.005728,"unknown":0.963322,"permission":0.00186},"logits":{"code":22.125,"transient":20.5,"unknown":25.625,"permission":19.375}},{"order":["code","permission","transient","unknown"],"probs":{"code":0.017777,"permission":0.002123,"transient":0.009515,"unknown":0.970585},"logits":{"code":21.875,"permission":19.75,"transient":21.25,"unknown":25.875}},{"order":["code","permission","unknown","transient"],"probs":{"code":0.01576,"permission":0.000785,"unknown":0.97502,"transient":0.008436},"logits":{"code":21.75,"permission":18.75,"unknown":25.875,"transient":21.125}},{"order":["code","unknown","transient","permission"],"probs":{"code":0.02845,"unknown":0.942127,"transient":0.02845,"permission":0.000973},"logits":{"code":22.375,"unknown":25.875,"transient":22.375,"permission":19.0}},{"order":["code","unknown","permission","transient"],"probs":{"code":0.037071,"unknown":0.843726,"permission":0.005017,"transient":0.114186},"logits":{"code":22.25,"unknown":25.375,"permission":20.25,"transient":23.375}},{"order":["permission","transient","code","unknown"],"probs":{"permission":0.012279,"transient":0.009563,"code":0.00274,"unknown":0.975419},"logits":{"permission":21.25,"transient":21.0,"code":19.75,"unknown":25.625}},{"order":["permission","transient","unknown","code"],"probs":{"permission":0.005198,"transient":0.003153,"unknown":0.99049,"code":0.00116},"logits":{"permission":21.0,"transient":20.5,"unknown":26.25,"code":19.5}},{"order":["permission","code","transient","unknown"],"probs":{"permission":0.015685,"code":0.007409,"transient":0.006538,"unknown":0.970368},"logits":{"permission":21.625,"code":20.875,"transient":20.75,"unknown":25.75}},{"order":["permission","code","unknown","transient"],"probs":{"permission":0.017467,"code":0.00344,"unknown":0.953679,"transient":0.025415},"logits":{"permission":21.625,"code":20.0,"unknown":25.625,"transient":22.0}},{"order":["permission","unknown","transient","code"],"probs":{"permission":0.005159,"unknown":0.983205,"transient":0.008506,"code":0.003129},"logits":{"permission":21.125,"unknown":26.375,"transient":21.625,"code":20.625}},{"order":["permission","unknown","code","transient"],"probs":{"permission":0.005131,"unknown":0.977809,"code":0.003112,"transient":0.013948},"logits":{"permission":20.875,"unknown":26.125,"code":20.375,"transient":21.875}},{"order":["unknown","transient","code","permission"],"probs":{"unknown":0.983611,"transient":0.009643,"code":0.005849,"permission":0.000897},"logits":{"unknown":25.875,"transient":21.25,"code":20.75,"permission":18.875}},{"order":["unknown","transient","permission","code"],"probs":{"unknown":0.965554,"transient":0.015607,"permission":0.005067,"code":0.013773},"logits":{"unknown":25.75,"transient":21.625,"permission":20.5,"code":21.5}},{"order":["unknown","code","transient","permission"],"probs":{"unknown":0.96253,"code":0.010693,"transient":0.025651,"permission":0.001127},"logits":{"unknown":25.5,"code":21.0,"transient":21.875,"permission":18.75}},{"order":["unknown","code","permission","transient"],"probs":{"unknown":0.938045,"code":0.015162,"permission":0.005578,"transient":0.041215},"logits":{"unknown":25.375,"code":21.25,"permission":20.25,"transient":22.25}},{"order":["unknown","permission","transient","code"],"probs":{"unknown":0.931669,"permission":0.008061,"transient":0.040935,"code":0.019336},"logits":{"unknown":25.5,"permission":20.75,"transient":22.375,"code":21.625}},{"order":["unknown","permission","code","transient"],"probs":{"unknown":0.893459,"permission":0.011247,"code":0.030572,"transient":0.064722},"logits":{"unknown":25.125,"permission":20.75,"code":21.75,"transient":22.5}}],"model":{"source":"Qwen/Qwen3.5-4B","revision":"851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a","dtype":"bfloat16","device":"mps","torch_version":"2.13.0","transformers_version":"5.17.0"},"measured_at":"2026-09-24 05:39"},"route-mail":{"id":"route-mail","row":{"id":"route-mail","state":"A dashboard routing request. Message: 'check the mail and report urgent items only.' Facts: [a4000] Windows GPU experiment worker, [todobot] email-checking assistant session, [main] general conversation session.","question":"Which session should handle the message?","options":[{"id":"a4000","description":"Windows GPU experiment worker session"},{"id":"todobot","description":"Email checking assistant"},{"id":"main","description":"General conversation session"}]},"system":"Apply the supplied criterion to the supplied evidence. Choose exactly one listed option. Respond with only its uppercase letter, with no explanation or reasoning.","payload":"{\"evidence\": \"A dashboard routing request. Message: 'check the mail and report urgent items only.' Facts: [a4000] Windows GPU experiment worker, [todobot] email-checking assistant session, [main] general conversation session.\", \"criterion\": \"Which session should handle the message?\", \"options\": [{\"letter\": \"A\", \"description\": \"Windows GPU experiment worker session\"}, {\"letter\": \"B\", \"description\": \"Email checking assistant\"}, {\"letter\": \"C\", \"description\": \"General conversation session\"}]}","prompt":"<|im_start|>system\nApply the supplied criterion to the supplied evidence. Choose exactly one listed option. Respond with only its uppercase letter, with no explanation or reasoning.<|im_end|>\n<|im_start|>user\n{\"evidence\": \"A dashboard routing request. Message: 'check the mail and report urgent items only.' Facts: [a4000] Windows GPU experiment worker, [todobot] email-checking assistant session, [main] general conversation session.\", \"criterion\": \"Which session should handle the message?\", \"options\": [{\"letter\": \"A\", \"description\": \"Windows GPU experiment worker session\"}, {\"letter\": \"B\", \"description\": \"Email checking assistant\"}, {\"letter\": \"C\", \"description\": \"General conversation session\"}]}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n","prompt_sha256":"b2d7f32784f9d997aa7d5472ae39fb4d4423f22163439ad522ad648094f3fd50","tokens":[[248045,0,12,"template",null],[8678,12,18,"template",null],[198,18,19,"template",null],[27567,19,24,"system",null],[279,24,28,"system",null],[16713,28,37,"system",null],[34765,37,47,"system",null],[310,47,50,"system",null],[279,50,54,"system",null],[16713,54,63,"system",null],[5721,63,72,"system",null],[13,72,73,"system",null],[21513,73,80,"system",null],[6681,80,88,"system",null],[799,88,92,"system",null],[9711,92,99,"system",null],[2904,99,106,"system",null],[13,106,107,"system",null],[38224,107,115,"system",null],[440,115,120,"system",null],[1132,120,125,"system",null],[1141,125,129,"system",null],[38175,129,139,"system",null],[6321,139,146,"system",null],[11,146,147,"system",null],[440,147,152,"system",null],[874,152,155,"system",null],[15673,155,167,"system",null],[466,167,170,"system",null],[31626,170,180,"system",null],[13,180,181,"system",null],[248046,181,191,"template",null],[198,191,192,"template",null],[248045,192,204,"template",null],[846,204,208,"template",null],[198,208,209,"template",null],[4754,209,211,"json",null],[68,211,212,"json",null],[26590,212,219,"json",null],[763,219,221,"json",null],[328,221,223,"json",null],[32,223,224,"evidence",null],[26103,224,234,"evidence",null],[28109,234,242,"evidence",null],[1622,242,250,"evidence",null],[13,250,251,"evidence",null],[4701,251,259,"evidence",null],[25,259,260,"json",null],[359,260,262,"evidence",null],[1960,262,267,"evidence",null],[279,267,271,"evidence",null],[7819,271,276,"evidence",null],[321,276,280,"evidence",null],[1830,280,287,"evidence",null],[32556,287,294,"evidence",null],[3470,294,300,"evidence",null],[1132,300,305,"evidence",null],[3058,305,307,"evidence",null],[43483,307,313,"evidence",null],[25,313,314,"json",null],[498,314,316,"json",null],[64,316,317,"evidence",null],[19,317,318,"evidence",null],[15,318,319,"evidence",null],[15,319,320,"evidence",null],[15,320,321,"evidence",null],[60,321,322,"json",null],[5342,322,330,"evidence",null],[21966,330,334,"evidence",null],[9064,334,345,"evidence",null],[11525,345,352,"evidence",null],[11,352,353,"json",null],[498,353,355,"json",null],[65597,355,358,"evidence",null],[43159,358,362,"evidence",null],[60,362,363,"json",null],[2469,363,369,"evidence",null],[15464,369,375,"evidence",null],[286,375,378,"evidence",null],[17313,378,388,"evidence",null],[3669,388,396,"evidence",null],[11,396,397,"json",null],[498,397,399,"json",null],[3689,399,403,"evidence",null],[60,403,404,"json",null],[4437,404,412,"evidence",null],[10125,412,425,"evidence",null],[3669,425,433,"evidence",null],[10152,433,436,"evidence",null],[328,436,438,"json",null],[66,438,439,"json",null],[11981,439,447,"json",null],[763,447,449,"json",null],[328,449,451,"json",null],[22365,451,456,"criterion",null],[3669,456,464,"criterion",null],[1220,464,471,"criterion",null],[3579,471,478,"criterion",null],[279,478,482,"criterion",null],[1876,482,490,"criterion",null],[29993,490,493,"criterion",null],[328,493,495,"json",null],[2782,495,502,"json",null],[763,502,504,"json",null],[59670,504,508,"json",null],[9172,508,514,"json",null],[763,514,516,"json",null],[328,516,518,"json",null],[32,518,519,"letter","A"],[487,519,521,"json",null],[328,521,523,"json",null],[4532,523,534,"json",null],[763,534,536,"json",null],[328,536,538,"json",null],[12782,538,545,"option","A"],[21966,545,549,"option","A"],[9064,549,560,"option","A"],[11525,560,567,"option","A"],[3669,567,575,"option","A"],[13933,575,578,"json",null],[5046,578,581,"json",null],[9172,581,587,"json",null],[763,587,589,"json",null],[328,589,591,"json",null],[33,591,592,"letter","B"],[487,592,594,"json",null],[328,594,596,"json",null],[4532,596,607,"json",null],[763,607,609,"json",null],[328,609,611,"json",null],[4628,611,616,"option","B"],[12910,616,625,"option","B"],[17313,625,635,"option","B"],[13933,635,638,"json",null],[5046,638,641,"json",null],[9172,641,647,"json",null],[763,647,649,"json",null],[328,649,651,"json",null],[34,651,652,"letter","C"],[487,652,654,"json",null],[328,654,656,"json",null],[4532,656,667,"json",null],[763,667,669,"json",null],[328,669,671,"json",null],[14965,671,678,"option","C"],[10125,678,691,"option","C"],[3669,691,699,"option","C"],[8934,699,701,"json",null],[13587,701,703,"json",null],[248046,703,713,"template",null],[198,713,714,"template",null],[248045,714,726,"template",null],[74455,726,735,"template",null],[198,735,736,"gen",null],[248068,736,743,"gen",null],[271,743,745,"gen",null],[248069,745,753,"gen",null],[271,753,755,"gen",null]],"slots":[["A",32],["B",33],["C",34]],"option_ids":["a4000","todobot","main"],"option_logits":[20.875,27.0,19.0],"probabilities":[0.0021819860682580675,0.9974833955305882,0.00033461840115381173],"forward_seconds":0.6208215000806376,"input_tokens":158,"top_tokens":[[33,"B",0.9972577691078186],[32,"A",0.0021814925130456686],[34,"C",0.0003345428267493844],[248068,"<think>",9.584813233232126e-05],[61442,"\"B",3.111733531113714e-05],[35,"D",8.915264515962917e-06],[36,"E",7.86769396654563e-06],[248046,"<|im_end|>",7.86769396654563e-06],[65,"b",5.079764832771616e-06],[760,"The",5.079764832771616e-06]],"allowed_mass":0.9997730851173401,"readout_check_max_abs_diff":0.0,"lens":[[30.875,29.125,29.5],[2.422,1.375,2.359],[3.062,1.445,2.672],[7.094,2.031,4.906],[6.344,0.2,4.156],[3.953,0.566,5.156],[-0.05,-1.594,2.391],[3.422,-0.078,2.047],[1.867,0.264,0.949],[2.25,0.824,0.01],[-0.139,-0.165,-0.22],[1.68,0.98,1.156],[0.447,1.039,1.539],[-0.797,-1.406,0.559],[1.195,-1.062,1.227],[2.469,0.703,2.156],[2.656,2.125,2.047],[3.172,3.016,2.672],[4.375,3.984,4.406],[2.25,2.312,4.594],[3.297,3.969,2.891],[6.0,5.625,3.938],[6.281,7.594,5.156],[5.375,6.25,4.344],[5.156,9.875,3.766],[4.875,9.562,4.125],[4.5,9.938,3.562],[4.844,9.375,4.312],[6.125,12.25,5.688],[7.5,15.188,6.344],[8.062,17.25,7.469],[10.25,21.5,9.312],[20.875,27.0,19.0]],"layer_types":["linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention"],"perms":[{"order":["a4000","todobot","main"],"probs":{"a4000":0.002182,"todobot":0.997483,"main":0.000335},"logits":{"a4000":20.875,"todobot":27.0,"main":19.0}},{"order":["a4000","main","todobot"],"probs":{"a4000":0.003577,"main":0.004593,"todobot":0.99183},"logits":{"a4000":20.375,"main":20.625,"todobot":26.0}},{"order":["todobot","a4000","main"],"probs":{"todobot":0.993984,"a4000":0.005216,"main":0.0008},"logits":{"todobot":26.125,"a4000":20.875,"main":19.0}},{"order":["todobot","main","a4000"],"probs":{"todobot":0.987093,"main":0.012426,"a4000":0.000482},"logits":{"todobot":26.0,"main":21.625,"a4000":18.375}},{"order":["main","a4000","todobot"],"probs":{"main":0.002175,"a4000":0.003586,"todobot":0.994239},"logits":{"main":19.875,"a4000":20.375,"todobot":26.0}},{"order":["main","todobot","a4000"],"probs":{"main":0.001325,"todobot":0.998472,"a4000":0.000203},"logits":{"main":20.375,"todobot":27.0,"a4000":18.5}}],"model":{"source":"Qwen/Qwen3.5-4B","revision":"851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a","dtype":"bfloat16","device":"mps","torch_version":"2.13.0","transformers_version":"5.17.0"},"measured_at":"2026-09-24 05:39"},"gate-danger":{"id":"gate-danger","row":{"id":"gate-danger","state":"An agent is about to run a command: delete all files in the project directory. The user asked only for a summary report.","question":"Should the command proceed without asking the user?","options":[{"id":"proceed","description":"Run it now without asking"},{"id":"ask","description":"Stop and confirm with the user first"}]},"system":"Apply the supplied criterion to the supplied evidence. Choose exactly one listed option. Respond with only its uppercase letter, with no explanation or reasoning.","payload":"{\"evidence\": \"An agent is about to run a command: delete all files in the project directory. The user asked only for a summary report.\", \"criterion\": \"Should the command proceed without asking the user?\", \"options\": [{\"letter\": \"A\", \"description\": \"Run it now without asking\"}, {\"letter\": \"B\", \"description\": \"Stop and confirm with the user first\"}]}","prompt":"<|im_start|>system\nApply the supplied criterion to the supplied evidence. Choose exactly one listed option. Respond with only its uppercase letter, with no explanation or reasoning.<|im_end|>\n<|im_start|>user\n{\"evidence\": \"An agent is about to run a command: delete all files in the project directory. The user asked only for a summary report.\", \"criterion\": \"Should the command proceed without asking the user?\", \"options\": [{\"letter\": \"A\", \"description\": \"Run it now without asking\"}, {\"letter\": \"B\", \"description\": \"Stop and confirm with the user first\"}]}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n","prompt_sha256":"d6d770811f5964314be48c1e91b534c2a20c099bb0aec60fb50c3abec9859151","tokens":[[248045,0,12,"template",null],[8678,12,18,"template",null],[198,18,19,"template",null],[27567,19,24,"system",null],[279,24,28,"system",null],[16713,28,37,"system",null],[34765,37,47,"system",null],[310,47,50,"system",null],[279,50,54,"system",null],[16713,54,63,"system",null],[5721,63,72,"system",null],[13,72,73,"system",null],[21513,73,80,"system",null],[6681,80,88,"system",null],[799,88,92,"system",null],[9711,92,99,"system",null],[2904,99,106,"system",null],[13,106,107,"system",null],[38224,107,115,"system",null],[440,115,120,"system",null],[1132,120,125,"system",null],[1141,125,129,"system",null],[38175,129,139,"system",null],[6321,139,146,"system",null],[11,146,147,"system",null],[440,147,152,"system",null],[874,152,155,"system",null],[15673,155,167,"system",null],[466,167,170,"system",null],[31626,170,180,"system",null],[13,180,181,"system",null],[248046,181,191,"template",null],[198,191,192,"template",null],[248045,192,204,"template",null],[846,204,208,"template",null],[198,208,209,"template",null],[4754,209,211,"json",null],[68,211,212,"json",null],[26590,212,219,"json",null],[763,219,221,"json",null],[328,221,223,"json",null],[2014,223,225,"evidence",null],[8057,225,231,"evidence",null],[369,231,234,"evidence",null],[883,234,240,"evidence",null],[310,240,243,"evidence",null],[1542,243,247,"evidence",null],[264,247,249,"evidence",null],[3108,249,257,"evidence",null],[25,257,258,"json",null],[3572,258,265,"evidence",null],[660,265,269,"evidence",null],[3425,269,275,"evidence",null],[303,275,278,"evidence",null],[279,278,282,"evidence",null],[2313,282,290,"evidence",null],[6025,290,300,"evidence",null],[13,300,301,"evidence",null],[561,301,305,"evidence",null],[1156,305,310,"evidence",null],[4439,310,316,"evidence",null],[1132,316,321,"evidence",null],[364,321,325,"evidence",null],[264,325,327,"evidence",null],[11782,327,335,"evidence",null],[1830,335,342,"evidence",null],[10152,342,345,"evidence",null],[328,345,347,"json",null],[66,347,348,"json",null],[11981,348,356,"json",null],[763,356,358,"json",null],[328,358,360,"json",null],[14562,360,366,"criterion",null],[279,366,370,"criterion",null],[3108,370,378,"criterion",null],[10046,378,386,"criterion",null],[1973,386,394,"criterion",null],[9859,394,401,"criterion",null],[279,401,405,"criterion",null],[1156,405,410,"criterion",null],[29993,410,413,"criterion",null],[328,413,415,"json",null],[2782,415,422,"json",null],[763,422,424,"json",null],[59670,424,428,"json",null],[9172,428,434,"json",null],[763,434,436,"json",null],[328,436,438,"json",null],[32,438,439,"letter","A"],[487,439,441,"json",null],[328,441,443,"json",null],[4532,443,454,"json",null],[763,454,456,"json",null],[328,456,458,"json",null],[6516,458,461,"option","A"],[424,461,464,"option","A"],[1381,464,468,"option","A"],[1973,468,476,"option","A"],[9859,476,483,"option","A"],[13933,483,486,"json",null],[5046,486,489,"json",null],[9172,489,495,"json",null],[763,495,497,"json",null],[328,497,499,"json",null],[33,499,500,"letter","B"],[487,500,502,"json",null],[328,502,504,"json",null],[4532,504,515,"json",null],[763,515,517,"json",null],[328,517,519,"json",null],[10358,519,523,"option","B"],[321,523,527,"option","B"],[7440,527,535,"option","B"],[440,535,540,"option","B"],[279,540,544,"option","B"],[1156,544,549,"option","B"],[1118,549,555,"option","B"],[8934,555,557,"json",null],[13587,557,559,"json",null],[248046,559,569,"template",null],[198,569,570,"template",null],[248045,570,582,"template",null],[74455,582,591,"template",null],[198,591,592,"gen",null],[248068,592,599,"gen",null],[271,599,601,"gen",null],[248069,601,609,"gen",null],[271,609,611,"gen",null]],"slots":[["A",32],["B",33]],"option_ids":["proceed","ask"],"option_logits":[21.875,27.5],"probabilities":[0.0035936025814200896,0.9964063974185798],"forward_seconds":0.5511322080856189,"input_tokens":128,"top_tokens":[[33,"B",0.9962387681007385],[32,"A",0.0035929977893829346],[34,"C",5.125138341099955e-05],[248068,"<think>",3.9914615626912564e-05],[61442,"\"B",2.4209439288824797e-05],[35,"D",8.366559086425696e-06],[1,"\"",3.952082352043362e-06],[760,"The",1.9872318262059707e-06],[36,"E",1.6474730273330351e-06],[5173,"Option",1.4538899222316104e-06]],"allowed_mass":0.9998302459716797,"readout_check_max_abs_diff":0.0,"lens":[[30.875,29.125],[2.359,1.281],[3.078,1.516],[7.312,2.016],[6.469,0.467],[4.156,0.527],[-0.175,-1.711],[2.781,-0.527],[1.859,0.902],[2.625,1.211],[-0.057,0.19],[1.883,1.297],[0.085,1.18],[-1.406,-1.672],[0.332,-1.055],[1.711,0.237],[2.188,2.094],[2.453,2.672],[3.047,3.141],[1.242,1.5],[2.25,3.406],[4.656,4.812],[4.938,7.438],[3.375,6.406],[4.156,9.688],[3.625,9.688],[3.406,10.438],[3.703,9.625],[6.562,12.062],[7.906,14.688],[9.375,17.25],[11.125,22.125],[21.875,27.5]],"layer_types":["linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention"],"perms":[{"order":["proceed","ask"],"probs":{"proceed":0.003594,"ask":0.996406},"logits":{"proceed":21.875,"ask":27.5}},{"order":["ask","proceed"],"probs":{"ask":0.999089,"proceed":0.000911},"logits":{"ask":27.5,"proceed":20.5}}],"model":{"source":"Qwen/Qwen3.5-4B","revision":"851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a","dtype":"bfloat16","device":"mps","torch_version":"2.13.0","transformers_version":"5.17.0"},"measured_at":"2026-09-24 05:39"},"my-test":{"id":"my-test","row":{"id":"my-test","state":"고객 메일: '지난주에 주문한 노트북이 아직 안 왔어요. 그냥 취소하고 환불받고 싶습니다.'","question":"이 메일을 어느 팀으로 보내야 하나요?","options":[{"id":"shipping","description":"배송 조회·지연 담당"},{"id":"refund","description":"취소·환불 담당"},{"id":"sales","description":"구매 상담 담당"}]},"system":"Apply the supplied criterion to the supplied evidence. Choose exactly one listed option. Respond with only its uppercase letter, with no explanation or reasoning.","payload":"{\"evidence\": \"고객 메일: '지난주에 주문한 노트북이 아직 안 왔어요. 그냥 취소하고 환불받고 싶습니다.'\", \"criterion\": \"이 메일을 어느 팀으로 보내야 하나요?\", \"options\": [{\"letter\": \"A\", \"description\": \"배송 조회·지연 담당\"}, {\"letter\": \"B\", \"description\": \"취소·환불 담당\"}, {\"letter\": \"C\", \"description\": \"구매 상담 담당\"}]}","prompt":"<|im_start|>system\nApply the supplied criterion to the supplied evidence. Choose exactly one listed option. Respond with only its uppercase letter, with no explanation or reasoning.<|im_end|>\n<|im_start|>user\n{\"evidence\": \"고객 메일: '지난주에 주문한 노트북이 아직 안 왔어요. 그냥 취소하고 환불받고 싶습니다.'\", \"criterion\": \"이 메일을 어느 팀으로 보내야 하나요?\", \"options\": [{\"letter\": \"A\", \"description\": \"배송 조회·지연 담당\"}, {\"letter\": \"B\", \"description\": \"취소·환불 담당\"}, {\"letter\": \"C\", \"description\": \"구매 상담 담당\"}]}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n","prompt_sha256":"c9964ef924ef36adfe3ea9b4a7032eb65146be08ba714881eda1b0788cef31f1","tokens":[[248045,0,12,"template",null],[8678,12,18,"template",null],[198,18,19,"template",null],[27567,19,24,"system",null],[279,24,28,"system",null],[16713,28,37,"system",null],[34765,37,47,"system",null],[310,47,50,"system",null],[279,50,54,"system",null],[16713,54,63,"system",null],[5721,63,72,"system",null],[13,72,73,"system",null],[21513,73,80,"system",null],[6681,80,88,"system",null],[799,88,92,"system",null],[9711,92,99,"system",null],[2904,99,106,"system",null],[13,106,107,"system",null],[38224,107,115,"system",null],[440,115,120,"system",null],[1132,120,125,"system",null],[1141,125,129,"system",null],[38175,129,139,"system",null],[6321,139,146,"system",null],[11,146,147,"system",null],[440,147,152,"system",null],[874,152,155,"system",null],[15673,155,167,"system",null],[466,167,170,"system",null],[31626,170,180,"system",null],[13,180,181,"system",null],[248046,181,191,"template",null],[198,191,192,"template",null],[248045,192,204,"template",null],[846,204,208,"template",null],[198,208,209,"template",null],[4754,209,211,"json",null],[68,211,212,"json",null],[26590,212,219,"json",null],[763,219,221,"json",null],[328,221,223,"json",null],[221791,223,225,"evidence",null],[49661,225,227,"evidence",null],[31020,227,228,"evidence",null],[25,228,229,"json",null],[359,229,231,"evidence",null],[213101,231,233,"evidence",null],[52506,233,234,"evidence",null],[18803,234,235,"evidence",null],[159950,235,238,"evidence",null],[22836,238,239,"evidence",null],[218333,239,243,"evidence",null],[12434,243,244,"evidence",null],[161010,244,247,"evidence",null],[91886,247,249,"evidence",null],[164352,249,251,"evidence",null],[155676,251,253,"evidence",null],[13,253,254,"evidence",null],[184051,254,257,"evidence",null],[182683,257,260,"evidence",null],[149789,260,262,"evidence",null],[188136,262,265,"evidence",null],[208238,265,267,"evidence",null],[237902,267,272,"evidence",null],[3058,272,274,"evidence",null],[487,274,276,"json",null],[328,276,278,"json",null],[66,278,279,"json",null],[11981,279,287,"json",null],[763,287,289,"json",null],[328,289,291,"json",null],[12434,291,292,"criterion",null],[49661,292,294,"criterion",null],[185664,294,296,"criterion",null],[175885,296,299,"criterion",null],[175209,299,301,"criterion",null],[40649,301,303,"criterion",null],[175920,303,306,"criterion",null],[86576,306,307,"criterion",null],[150675,307,310,"criterion",null],[34530,310,311,"criterion",null],[29993,311,314,"criterion",null],[328,314,316,"json",null],[2782,316,323,"json",null],[763,323,325,"json",null],[59670,325,329,"json",null],[9172,329,335,"json",null],[763,335,337,"json",null],[328,337,339,"json",null],[32,339,340,"letter","A"],[487,340,342,"json",null],[328,342,344,"json",null],[4532,344,355,"json",null],[763,355,357,"json",null],[328,357,359,"json",null],[180689,359,361,"option","A"],[93816,361,364,"option","A"],[13535,364,365,"option","A"],[20673,365,366,"option","A"],[148881,366,367,"option","A"],[181035,367,370,"option","A"],[13933,370,373,"json",null],[5046,373,376,"json",null],[9172,376,382,"json",null],[763,382,384,"json",null],[328,384,386,"json",null],[33,386,387,"letter","B"],[487,387,389,"json",null],[328,389,391,"json",null],[4532,391,402,"json",null],[763,402,404,"json",null],[328,404,406,"json",null],[158744,406,407,"option","B"],[42147,407,408,"option","B"],[13535,408,409,"option","B"],[63035,409,410,"option","B"],[150856,410,411,"option","B"],[181035,411,414,"option","B"],[13933,414,417,"json",null],[5046,417,420,"json",null],[9172,420,426,"json",null],[763,426,428,"json",null],[328,428,430,"json",null],[34,430,431,"letter","C"],[487,431,433,"json",null],[328,433,435,"json",null],[4532,435,446,"json",null],[763,446,448,"json",null],[328,448,450,"json",null],[219604,450,452,"option","C"],[181276,452,455,"option","C"],[181035,455,458,"option","C"],[8934,458,460,"json",null],[13587,460,462,"json",null],[248046,462,472,"template",null],[198,472,473,"template",null],[248045,473,485,"template",null],[74455,485,494,"template",null],[198,494,495,"gen",null],[248068,495,502,"gen",null],[271,502,504,"gen",null],[248069,504,512,"gen",null],[271,512,514,"gen",null]],"slots":[["A",32],["B",33],["C",34]],"option_ids":["shipping","refund","sales"],"option_logits":[20.625,26.375,19.375],"probabilities":[0.003169801528228894,0.9959220351288388,0.0009081633429323395],"forward_seconds":0.8448315840214491,"input_tokens":143,"top_tokens":[[33,"B",0.9955710172653198],[32,"A",0.003168684197589755],[34,"C",0.0009078432340174913],[248068,"<think>",0.00026010157307609916],[35,"D",1.3784878319711424e-05],[61442,"\"B",1.0735673640738241e-05],[36,"E",8.900186003302224e-06],[248046,"<|im_end|>",6.117002612882061e-06],[58297,"Б",5.398235771281179e-06],[1,"\"",3.4853628676501103e-06]],"allowed_mass":0.9996472001075745,"readout_check_max_abs_diff":0.0,"lens":[[30.875,29.125,29.5],[2.484,1.336,2.297],[2.578,0.918,2.281],[6.125,1.148,4.438],[5.875,-0.046,4.594],[3.688,0.586,4.719],[-0.005,-1.086,2.172],[3.062,0.188,2.281],[2.516,0.91,1.703],[2.562,1.031,0.793],[0.281,0.247,0.625],[2.609,1.227,1.422],[1.141,1.242,1.352],[-0.061,-0.516,0.629],[1.148,-0.459,1.148],[2.344,1.133,1.969],[2.875,2.109,2.062],[2.703,3.156,2.969],[3.734,4.125,4.5],[1.93,2.109,3.531],[2.906,4.406,2.172],[5.5,5.438,3.406],[5.938,7.0,4.688],[4.25,5.125,3.406],[3.594,9.5,2.531],[3.328,8.562,3.047],[3.156,9.125,2.484],[3.609,7.812,3.453],[6.281,11.438,5.75],[7.281,13.875,6.281],[8.625,16.625,8.25],[10.438,21.5,9.75],[20.625,26.375,19.375]],"layer_types":["linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention","linear_attention","linear_attention","linear_attention","full_attention"],"perms":[{"order":["shipping","refund","sales"],"probs":{"shipping":0.00317,"refund":0.995922,"sales":0.000908},"logits":{"shipping":20.625,"refund":26.375,"sales":19.375}},{"order":["shipping","sales","refund"],"probs":{"shipping":0.029292,"sales":0.000689,"refund":0.970019},"logits":{"shipping":21.375,"sales":17.625,"refund":24.875}},{"order":["refund","shipping","sales"],"probs":{"refund":0.998028,"shipping":0.001169,"sales":0.000803},"logits":{"refund":26.25,"shipping":19.5,"sales":19.125}},{"order":["refund","sales","shipping"],"probs":{"refund":0.994718,"sales":0.001695,"shipping":0.003588},"logits":{"refund":26.0,"sales":19.625,"shipping":20.375}},{"order":["sales","shipping","refund"],"probs":{"sales":0.010984,"shipping":0.000258,"refund":0.988758},"logits":{"sales":20.875,"shipping":17.125,"refund":25.375}},{"order":["sales","refund","shipping"],"probs":{"sales":0.015754,"refund":0.97469,"shipping":0.009556},"logits":{"sales":21.5,"refund":25.625,"shipping":21.0}}],"model":{"source":"Qwen/Qwen3.5-4B","revision":"851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a","dtype":"bfloat16","device":"mps","torch_version":"2.13.0","transformers_version":"5.17.0"},"measured_at":"2026-09-24 05:39"}},"shared_demo":{"questions":8,"direct_seconds":10.949793375097215,"shared_seconds":3.8561414170544595,"direct_tokens":2473,"shared_tokens":751,"prefix_tokens":246,"serving_config":"native-state-prefix-looped-v1","same_picks":8,"max_prob_diff":8.798744592364294e-05,"answers":[{"id":"severity","direct":{"sev0":0.9999,"sev1":0.0001,"sev2":0.0,"sev3":0.0},"shared":{"sev0":0.9999,"sev1":0.0001,"sev2":0.0,"sev3":0.0}},{"id":"team","direct":{"sre":0.9991,"billing":0.0006,"sales":0.0002,"docs":0.0001},"shared":{"sre":0.9992,"billing":0.0006,"sales":0.0002,"docs":0.0001}},{"id":"churn","direct":{"yes":0.9996,"no":0.0004},"shared":{"yes":0.9996,"no":0.0004}},{"id":"refund","direct":{"yes":0.9047,"no":0.0953},"shared":{"yes":0.9047,"no":0.0953}},{"id":"legal","direct":{"yes":0.9933,"no":0.0067},"shared":{"yes":0.9933,"no":0.0067}},{"id":"status","direct":{"yes":0.982,"no":0.018},"shared":{"yes":0.982,"no":0.018}},{"id":"phone","direct":{"yes":0.9978,"no":0.0022},"shared":{"yes":0.9978,"no":0.0022}},{"id":"cause","direct":{"gateway":0.9999,"db":0.0001,"client":0.0,"unknown":0.0},"shared":{"gateway":0.9999,"db":0.0001,"client":0.0,"unknown":0.0}}],"repeats":2}}''')

print(f"강의 케이스 {len(CASES['rows'])}개 + A4000 기록 · 위젯용 Mac 측정 {len(FALLBACK['results'])}케이스 · "
      f"병렬 판단 속도 {len(PRE['parallel_decision']['rows'])}줄 · 14B T4 {len(PRE['colab14b']['rows'])}가지 · "
      f"API vs 로컬 {len(PRE['api_vs_local']['table'])}엔진 · Jev-Omni 검증 {len(PRE['jev_omni']['verification_cases'])}건")

## 1. 원리 한눈에: 글로 답하기 vs 점수 읽기

보통 LLM에게 "어느 세션으로 보낼까?"를 물으면 `{"choice": "main"}` 같은 **글을 토큰 하나씩** 써요. 토큰 하나마다 모델을 한 번씩 통과해야 하고, 코드는 그 글을 다시 읽어서 if 문으로 바꿔요.

SemIf 같은 Jev 닮은 모델은 글을 쓰지 않아요. 보기마다 글자(A, B, …)를 붙여 프롬프트에 넣고, 모델을 **딱 한 번** 통과시킨 뒤 '다음에 올 토큰' 점수 중 **보기 글자 점수만** 읽어요. 그 점수를 softmax로 바꾸면 보기별 확률이 돼요.

아래 영상 ①(27초)은 같은 질문(강의의 `route-vague` 케이스)을 두 방식으로 처리하는 모습이에요. 점수 24.0 · 24.5는 강사 A4000에서 잰 실제 값이에요.

In [ ]:
#@title 🎬 영상 ① 글로 답하기 vs 점수 읽기 (27초) — 실행하지 않아도 돼요
show_clip("clip1_generate_vs_read.mp4", "영상 ① · Manim으로 그림 · 숫자: route-vague, 강사 A4000 실측")

**영상 읽는 법**
- **통과 횟수**: 생성 방식은 답 토큰 수만큼(여기선 7번) 모델을 통과하고, 점수 읽기는 1번이에요. 속도 차이는 여기서 나와요(6장).
- **보기 밖 답이 없어요**: 정해 둔 보기 글자 중에서만 고르니 형식이 깨지지 않아요.
- **0.62의 뜻**: 'main이 맞을 확률 62%'가 아니라 **보기끼리 비교한 점수**예요. 보정은 7장에서 봐요.
- JSON 조각(`{"`, `choice`, `":` …)은 Qwen3.5-4B 토크나이저로 실제로 자른 결과예요.

## 2. GPU 확인 · 설치 · 모델 받기

### 2-1. GPU 확인과 실행 모드
어떤 장비에서 돌리는지 확인하고, 모델을 올릴 숫자 형식(dtype)을 정해요.
- **T4는 bfloat16을 하드웨어로 못 해서 float16**을 써요. 파이토치가 T4에서도 bf16을 '지원'한다고 답할 때가 있는데, 흉내 내는 방식이라 느려요. 그래서 GPU 세대(compute capability 8 이상인지)로 정해요.
- 강사 A4000은 bfloat16이었어요. 확률이 소수점 아래에서 조금 다를 수 있지만, 고르는 보기는 같아야 해요.
- GPU가 없으면 **precomputed(미리 잰 숫자) 모드**가 돼요. `FORCE_PRECOMPUTED = True`로 바꾸면 GPU가 있어도 이 모드로 돌아요.

In [ ]:
import os, sys, shutil, subprocess, time

IN_COLAB = "google.colab" in sys.modules
FORCE_PRECOMPUTED = os.environ.get("JEV3_PRECOMPUTED") == "1"   # True로 바꾸면 모델 없이 미리 잰 숫자만 써요

try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
    HAS_MPS = (not HAS_CUDA) and torch.backends.mps.is_available()
except ImportError:
    HAS_CUDA = HAS_MPS = False

if FORCE_PRECOMPUTED or not (HAS_CUDA or HAS_MPS):
    MODE, DTYPE, GPU_NAME = "precomputed", None, None
elif HAS_CUDA:
    MODE = "cuda"
    major, minor = torch.cuda.get_device_capability()
    DTYPE = "bfloat16" if major >= 8 else "float16"      # T4 = 7.5 → float16
    GPU_NAME = torch.cuda.get_device_name()
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU {GPU_NAME} · 메모리 {total_gb:.1f}GB · compute capability {major}.{minor}")
else:
    MODE, DTYPE, GPU_NAME = "mps", "bfloat16", "Apple GPU (MPS)"   # 강사 Mac 같은 Apple Silicon

print(f"모드: {MODE}" + (f" · dtype {DTYPE}" if DTYPE else " · 모델 없이 미리 잰 숫자로 그려요"))
if MODE == "cuda":
    !nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

**읽는 법**: `모드: cuda`, `Tesla T4`, 메모리 약 15GB가 보이면 준비 끝이에요. `모드: precomputed`가 보이면 GPU가 없는 런타임이에요 — 그림은 다 볼 수 있고, 모델을 직접 돌리는 셀만 미리 잰 숫자로 대신해요.

### 2-2. SemIf 설치 (커밋 고정)
SemIf는 openjev에서 이름이 바뀐 저장소예요(`TheoLeeCJ/SemIf`). 저장소가 계속 바뀌니 **강의 때 쓴 커밋 `1f2dea3`으로 고정**해서 받아요(그 커밋만 얕게 받아서 몇 초면 돼요).
- `pip install --no-deps -e`: 저장소가 적어 둔 torch 2.10을 새로 깔지 않고, **Colab에 이미 있는 CUDA용 torch를 그대로** 써요.
- 나머지(transformers 5.17 등)는 SemIf가 고정한 버전으로 맞춰요. 30초~1분쯤 걸리고, 다른 패키지와 버전이 안 맞는다는 경고는 무시해도 돼요.

In [ ]:
SEMIF_REPO = "https://github.com/TheoLeeCJ/SemIf.git"
SEMIF_COMMIT = "1f2dea3e25379f9dfc98cb83c324f00ab5deda37"   # 2026-09-22, 강의·A4000 실측과 같은 코드

def run(cmd):
    subprocess.run(cmd, check=True)

if MODE == "precomputed":
    print("precomputed 모드라 설치를 건너뛰어요.")
elif IN_COLAB:
    if not os.path.isdir("SemIf/.git"):
        run(["git", "init", "-q", "SemIf"])
        run(["git", "-C", "SemIf", "remote", "add", "origin", SEMIF_REPO])
        run(["git", "-C", "SemIf", "fetch", "-q", "--depth", "1", "origin", SEMIF_COMMIT])
        run(["git", "-C", "SemIf", "checkout", "-q", "--detach", "FETCH_HEAD"])
    head = subprocess.check_output(["git", "-C", "SemIf", "rev-parse", "HEAD"], text=True).strip()
    assert head == SEMIF_COMMIT, f"SemIf 커밋이 달라요: {head}"
    run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "./SemIf"])
    run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.17.0", "accelerate==1.12.0",
         "huggingface-hub==1.31.0", "tokenizers==0.23.2", "safetensors==0.8.0"])
    sys.path.insert(0, os.path.abspath("SemIf/src"))   # 런타임 재시작 없이 바로 import
else:
    # Colab 밖(예: 강사 Mac): SemIf를 미리 설치했거나 SEMIF_SRC=/path/to/SemIf/src 로 알려 주세요
    if os.environ.get("SEMIF_SRC"):
        sys.path.insert(0, os.environ["SEMIF_SRC"])

if MODE != "precomputed":
    import semif_phase1, transformers
    print(f"SemIf {semif_phase1.__version__} · transformers {transformers.__version__} · torch {torch.__version__}")

### 2-3. `hf download`로 필요한 파일만 받기
`hf`는 Hugging Face 공식 명령줄 도구예요(`huggingface_hub` 패키지에 들어 있어요). 모델 저장소에는 README·라이선스·이미지 전처리 설정도 있는데, 우리는 **글자 판단에 필요한 9개 파일만** 받아요.
- `--revision 851bf6e…`: A4000 실측과 **같은 커밋**으로 고정해요.
- `--include`: 받을 파일 모양(가중치 `*.safetensors`, 설정 `*.json`, 채팅 틀 `*.jinja`, `merges.txt`). `--exclude`로 이미지 전처리 설정은 빼요.
- 먼저 `--dry-run`으로 **무엇을 얼마나 받을지**만 보고, 다음 셀에서 진짜로 받아요.
- 받은 파일은 Hugging Face 캐시(`~/.cache/huggingface/hub`)에 들어가고, 3장의 `from_pretrained`가 그대로 찾아 써요. 공개 모델이라 로그인(`hf auth`)은 필요 없어요.

In [ ]:
MODEL = "Qwen/Qwen3.5-4B"
REVISION = "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"   # 강사 A4000 실측과 같은 커밋
HF_ARGS = (f'{MODEL} --revision {REVISION} --include "*.safetensors" --include "*.json" '
           f'--include "*.jinja" --include "merges.txt" --exclude "*preprocessor_config.json"')

print("$ hf download " + HF_ARGS + " --dry-run")
if shutil.which("hf"):
    !hf download {HF_ARGS} --dry-run
else:
    print("hf 명령이 없어요. 2-2 설치 셀(pip install huggingface-hub)을 먼저 실행해 주세요.")

**읽는 법**: 파일 9개, 합계 약 9.3GB예요. 대부분이 `model.safetensors-0000X-of-00002` 두 조각(5.3GB + 4.0GB)이고, 이미 받은 파일은 '받을 것'에 들어가지 않아요.

다음 셀이 **진짜로 받아요**(precomputed 모드면 건너뛰어요). Colab에서는 보통 몇 분 걸리고, 진행 막대가 보여요. 중간에 끊겨도 다시 실행하면 이어 받아요.

In [ ]:
if MODE == "precomputed":
    print("precomputed 모드라 모델을 받지 않아요.")
else:
    t0 = time.time()
    !hf download {HF_ARGS}
    print(f"받기 끝 · {time.time() - t0:.0f}초")

## 3. 모델 올리기

SemIf의 `load_causal_model()`로 올려요. 강사 A4000 상주 서비스(`openjev_service.py`)가 쓰던 바로 그 함수예요.
- 모델 이름과 **40자리 커밋**을 함께 줘야 해요. 안 주면 SemIf가 거절해요(재현성을 위해서예요).
- Qwen3.5-4B는 원래 이미지도 보는 모델이라, SemIf는 **글자 부분(`Qwen3_5ForCausalLM`)만** 꺼내 올려요.
- T4에서는 1~2분, GPU 메모리는 9GB쯤 써요.

In [ ]:
model = tokenizer = metadata = None
if MODE == "precomputed":
    print("precomputed 모드라 모델을 올리지 않아요. 4장부터는 미리 잰 숫자로 그려요.")
else:
    if MODE == "mps":
        os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"   # Mac(MPS): 가중치를 여러 스레드로 올리면 멈춰서 한 줄로 올려요
    from semif_phase1.core import load_causal_model
    from semif_phase1.direct import score
    t0 = time.time()
    model, tokenizer, metadata = load_causal_model(MODEL, REVISION, device=MODE, dtype=DTYPE)
    msg = f"로드 완료 {time.time() - t0:.0f}초 · {metadata['dtype']} · {metadata['device']}"
    if MODE == "cuda":
        msg += f" · GPU 메모리 {torch.cuda.memory_allocated() / 1e9:.1f}GB"
    print(msg)

**읽는 법**: `로드 완료 … float16 · cuda:0 · GPU 메모리 …GB`가 나오면 성공이에요. `causal_conv1d`·`flash-linear-attention`이 없어 느린 참조 구현을 쓴다는 경고가 나올 수 있어요. Qwen3.5의 선형 어텐션 층을 빠르게 해 주는 커널이 없다는 뜻이고, 결과는 같아요(조금 느릴 뿐).

## 4. 한 케이스 뜯어보기 (Transformer Explainer처럼)

먼저 강의에서 쓴 케이스 5개를 정의해요. 한 케이스 = `state`(상황) + `question`(질문) + `options`(보기). Jev의 **CHOICE** 질문과 같은 모양이에요. 강사 A4000 상주 서비스에 보냈던 입력 그대로예요.

In [ ]:
ROWS = [
    {
        "id": "route-gpu",
        "state": "A dashboard routing request. Message: 'A4000 GPU server experiment - continue processing it.' Facts: [a4000] Windows GPU experiment worker, [main] general conversation session.",
        "question": "Which session should handle the message?",
        "options": [
            {"id": "a4000", "description": "Windows GPU experiment worker session"},
            {"id": "main", "description": "General conversation session"},
        ],
    },
    {
        "id": "route-vague",
        "state": "A dashboard routing request. Message: 'just handle this for me' (no destination indicated). Facts: [a4000] Windows GPU experiment worker, [main] general conversation session.",
        "question": "Which session should handle the message?",
        "options": [
            {"id": "a4000", "description": "Windows GPU experiment worker session"},
            {"id": "main", "description": "General conversation session"},
        ],
    },
    {
        "id": "classify-fail",
        "state": "A command failed while running. No error text or logs are available in the state.",
        "question": "What is the failure classification?",
        "options": [
            {"id": "transient", "description": "Temporary error - retry once"},
            {"id": "code", "description": "Code defect - needs a fix"},
            {"id": "permission", "description": "Permission or auth issue"},
            {"id": "unknown", "description": "Cannot determine - ask the user"},
        ],
    },
    {
        "id": "route-mail",
        "state": "A dashboard routing request. Message: 'check the mail and report urgent items only.' Facts: [a4000] Windows GPU experiment worker, [todobot] email-checking assistant session, [main] general conversation session.",
        "question": "Which session should handle the message?",
        "options": [
            {"id": "a4000", "description": "Windows GPU experiment worker session"},
            {"id": "todobot", "description": "Email checking assistant"},
            {"id": "main", "description": "General conversation session"},
        ],
    },
    {
        "id": "gate-danger",
        "state": "An agent is about to run a command: delete all files in the project directory. The user asked only for a summary report.",
        "question": "Should the command proceed without asking the user?",
        "options": [
            {"id": "proceed", "description": "Run it now without asking"},
            {"id": "ask", "description": "Stop and confirm with the user first"},
        ],
    },
]

# 8장에서 쓸 예시 질문(한국어) — 위젯에도 같이 넣어요
MY_ROW_EXAMPLE = {
    "id": "my-test",
    "state": "고객 메일: '지난주에 주문한 노트북이 아직 안 왔어요. 그냥 취소하고 환불받고 싶습니다.'",
    "question": "이 메일을 어느 팀으로 보내야 하나요?",
    "options": [
        {"id": "shipping", "description": "배송 조회·지연 담당"},
        {"id": "refund", "description": "취소·환불 담당"},
        {"id": "sales", "description": "구매 상담 담당"},
    ],
}

A4000 = CASES["a4000"]   # 강사 A4000(bfloat16) 기록: 보기별 확률 · 점수 · 프롬프트 지문
print(f"케이스 {len(ROWS)}개: " + ", ".join(r["id"] for r in ROWS))

### 4-1. 영상 ③: 점수가 확률이 되는 길 (26초)
위젯의 ⑤·⑥단계를 먼저 영상으로 봐요. `classify-fail` 케이스의 A4000 실측 점수(22.25 · 20.125 · 20.0 · 25.875)가 softmax를 거쳐 0.026 · 0.003 · 0.003 · 0.968이 되는 과정과, 온도 T를 바꾸면 무엇이 달라지는지 보여 줘요.

In [ ]:
#@title 🎬 영상 ③ 점수(logit) → 확률, 그리고 온도 (26초) — 실행하지 않아도 돼요
show_clip("clip3_logits_to_probs.mp4", "영상 ③ · Manim으로 그림 · 숫자: classify-fail, 강사 A4000 실측")

**영상 읽는 법**: 가장 큰 값을 빼도 막대 모양은 그대로예요. 확률은 **점수의 차이**로만 정해져요. 온도 T는 1등을 바꾸지 않고, 확률이 얼마나 뾰족한지만 바꿔요.

### 4-2. 7단계 위젯
아래 셀은 케이스 6개(강의 5개 + 8장 예시)를 **SemIf 원본 `score()`** 로 판정하면서 중간값을 모아 위젯으로 그려요. 보기 순서를 바꾼 판정까지 합쳐 40번 넘게 돌려요.

| 단계 | 보여 주는 것 | 해 볼 것 |
|---|---|---|
| ① 입력 | state · question · options | 위쪽 버튼으로 케이스 바꾸기 |
| ② 프롬프트 조립 | SemIf가 만든 채팅 프롬프트 전문(역할별 색) | 보기 id가 프롬프트에 **없다**는 점 |
| ③ 토큰 | 토큰 조각과 번호 | 마우스를 올려 id 보기 · 마지막 토큰(주황 테두리) |
| ④ 모델 1번 통과 | 32층 구조와 층마다 읽어 본 보기 확률 | 층을 지나며 선이 어떻게 움직이는지 |
| ⑤ 글자 점수 | 전체 어휘 상위 10개와 보기 글자 점수 | 보기 글자에 모인 확률 |
| ⑥ 확률 | softmax · 온도 T 슬라이더 · 보기 순서 바꾸기 | T 끌어 보기 · 순서 바꾸기 켜기 |
| ⑦ 코드의 판단 | 기준 τ 슬라이더와 if 문 | τ를 옮겨 행동이 바뀌는지 |

- **GPU 모드**: 방금 이 런타임에서 잰 숫자로 그려요.
- **precomputed 모드**: 강사 Mac(M5, MPS, bfloat16)에서 같은 코드로 잰 숫자로 그려요. 5케이스 모두 A4000과 **프롬프트 지문(SHA-256)이 같고 고른 보기도 같았어요**.
- **단순화한 곳**: ④의 층별 선은 '로짓 렌즈'라는 **분석용 근사**예요(SemIf는 맨 끝 층만 읽어요). ⑥의 온도와 ⑦의 기준은 SemIf 판정에는 없는 **내 코드 쪽 손잡이**예요.
- 위젯은 인터넷 없이 동작해요(외부 스크립트·글꼴을 받지 않아요). 키보드 ←/→로도 단계를 옮길 수 있어요.

In [ ]:
CASE_ORDER = [r["id"] for r in ROWS] + [MY_ROW_EXAMPLE["id"]]
if MODE == "precomputed":
    EXPLAIN = [FALLBACK["results"][k] for k in CASE_ORDER]
    SOURCE = f"숫자: 강사 Mac(M5) · MPS · bfloat16 미리 잰 값 ({EXPLAIN[0]['measured_at']})"
    DEVICE = "Mac MPS · bfloat16"
else:
    score(model, tokenizer, ROWS[0], metadata)            # 첫 호출은 GPU 준비(워밍업) — 버려요
    t0 = time.time()
    EXPLAIN = [explain_case(model, tokenizer, row, metadata) for row in ROWS + [MY_ROW_EXAMPLE]]
    n_perm = sum(len(e["perms"]) for e in EXPLAIN)
    print(f"케이스 {len(EXPLAIN)}개 + 보기 순서 바꾼 판정 {n_perm}번 · {time.time() - t0:.1f}초")
    SOURCE = f"숫자: 방금 이 런타임에서 잰 값 ({GPU_NAME} · {DTYPE})"
    DEVICE = f"{GPU_NAME} · {DTYPE}"
show_explainer(EXPLAIN, SOURCE, DEVICE, refs=A4000)

**읽는 법 — 이것부터 눌러 보세요** (숫자는 강사 Mac 기준이에요. T4에서는 소수점이 조금 달라요)
1. `목적지 없는 요청` → **④**: 초록 선(main)이 층마다 오르내리다 마지막에 0.59로 끝나요. 근거가 없는 질문이라 모델도 갈팡질팡해요. 다른 케이스는 20층대에서 답이 굳어요.
2. 같은 케이스 → **⑥ → 보기 순서 바꿔 보기**: 순서만 바꿨는데 main이 **A 자리일 때 0.93, B 자리일 때 0.59**예요. 작은 모델은 자리(글자)에 쏠려요. 보기 순서도 입력의 일부예요.
3. **⑦**: 기준 0.70이면 '되묻기', 0.55로 내리면 'main으로 진행'. 같은 확률이라도 행동은 코드가 정해요.
4. `요약만 부탁했는데 전체 삭제?` → **⑥**: ask 0.996. 순서를 바꿔도 거의 그대로예요 — 분명한 질문은 흔들리지 않아요.

## 5. 강의 5케이스 판정 + 강사 A4000과 비교

이번엔 위젯 없이, 실제 코드에서 쓰는 모양 그대로 판정해요. `decide(row)`는 SemIf의 `validate_row()`로 입력을 검사하고 `score()`로 보기별 확률을 받아요. 강사 A4000(bfloat16) 기록과 나란히 놓아요.

In [ ]:
if MODE != "precomputed":
    from semif_phase1.core import validate_row

    def decide(row):
        validate_row(row)
        return score(model, tokenizer, row, metadata)
else:
    def decide(row):
        r = FALLBACK["results"].get(row["id"])
        if r is None or r["row"] != row:
            raise RuntimeError("precomputed 모드에서는 미리 잰 케이스만 볼 수 있어요. GPU 런타임에서 다시 실행해 주세요.")
        return r
    print("(precomputed 모드: 아래 '이번' 숫자와 ms는 강사 Mac · MPS · bfloat16 기록이에요)\n")

NOW = {}
for row in ROWS:
    r = decide(row)
    NOW[row["id"]] = r
    probs = dict(zip(r["option_ids"], r["probabilities"]))
    ref = dict(zip(A4000[row["id"]]["option_ids"], A4000[row["id"]]["probabilities"]))
    pick, ref_pick = max(probs, key=probs.get), max(ref, key=ref.get)
    print(f"[{row['id']}] {row['question']}")
    for k in probs:
        print(f"   {k:<10} 이번 {probs[k]:.3f}   A4000 {ref[k]:.3f}")
    same = "✓ 같음" if pick == ref_pick else "✗ 다름"
    print(f"   → 선택 {pick} ({r['forward_seconds'] * 1000:.0f}ms) | A4000 {ref_pick}  {same}\n")

chart_cases(NOW, A4000, "이번 실행" if MODE != "precomputed" else "강사 Mac (bfloat16)")

**읽는 법**
- `✓ 같음`이 5개 모두 나오면 성공이에요. T4(float16)와 A4000(bfloat16)은 숫자 형식이 달라 소수점 둘째 자리쯤 달라질 수 있어요. 강사 Mac(bfloat16)에서는 `route-vague`의 차이가 가장 컸고, 그래도 0.03 정도였어요(a4000 0.407 vs 0.378).
- `route-vague`처럼 근거가 없는 질문에서 확률이 한쪽으로 쏠리지 않으면 정상이에요. 강의에서 본 TypeSafe Jev는 같은 질문에 main 0.84였어요.
- ms는 한 번 판정(생성 없이 1번 통과) 시간이에요. 강사 A4000 기록은 첫 호출 뒤 0.10~0.13초였어요. 첫 호출은 GPU 준비 때문에 느려서 한 번 버리고 시작해요.

## 6. 왜 빠른가: 공통 앞부분은 한 번, 질문은 나란히

판단을 여러 개 할 때 속도를 가르는 건 **같은 상황(state)을 몇 번 읽느냐**예요.
- 문의 1건에 물을 게 28개면, 생성 방식은 28개 필드를 **한 토큰씩 차례로** 써요.
- 병렬 판단은 긴 상황을 **한 번만** 계산해 문맥 캐시(KV 캐시)에 두고, 질문+보기처럼 짧은 뒷부분만 **나란히** 계산해요. SemIf에는 `score_shared()`로, llama.cpp에는 `parallel-decision` 포크로 들어 있어요.

영상 ②(22초)는 이 구조와 강사 장비의 실측 숫자를 보여 줘요.

In [ ]:
#@title 🎬 영상 ② 공통 앞부분은 한 번, 질문은 나란히 (22초) — 실행하지 않아도 돼요
show_clip("clip2_shared_prefix.mp4", "영상 ② · Manim으로 그림 · 숫자: llama.cpp parallel-decision, 강사 Mac M5 · RTX A4000 실측")

**영상 읽는 법**: 초록 칸(질문+보기)은 서로 기다리지 않고 한꺼번에 계산돼요. 아래 주황 띠(JSON 생성)는 필드를 한 줄로 써 내려가다 이 실행에선 필드 2개를 빠뜨렸어요. 숫자는 같은 1.5B 모델 파일로 Mac과 A4000에서 잰 값이에요.

### 6-1. 미리 잰 결과 ①: 병렬 판단 vs JSON 생성 (Mac M5 · RTX A4000)
2026-09-22에 강사가 llama.cpp `parallel-decision` 포크로 잰 기록이에요. 같은 Qwen2.5-1.5B GGUF 파일을 두 장비에서 돌렸어요. 모델을 다시 돌리지 않고 기록(JSON)만 읽어 그려요.

In [ ]:
chart_parallel(PRE["parallel_decision"])

**읽는 법**: 초록 점(병렬 판단)과 주황 점(JSON 생성) 사이 거리가 배율이에요. 가로축이 로그 눈금이라 눈금 한 칸이 약 3배예요. A4000은 필드 28개를 0.1초 안팎에, Mac도 0.4~0.6초에 끝냈어요. 그림 아래 설명에 **빠르다고 다 맞는 건 아니라는** 작은 정답 테스트 결과도 있어요(1.5B 6/8 · 9B 8/8).

### 6-2. 미리 잰 결과 ②: 14B 모델을 Colab T4에서 (예전 무거운 노트북)
예전 노트북(`Colab_Local_16GB_Qwen14B.ipynb`)이 오래 걸린 이유가 이 실험이에요. Qwen2.5-14B 원본(약 30GB)을 받아 4비트로 올리고, SemIf·RLCD·JSON 생성을 반복해 재고, 선택으로 GGUF(약 9GB)를 받아 llama.cpp를 T4용 CUDA로 직접 빌드했어요. 그 결과 파일(`comparison_all.json`)을 그대로 그려요.

> 여기서 **RLCD**는 공개 데모 코드(`harshatheg/Qwen-2.5-1B-RLCD`)의 병렬 읽기 구현이에요. TypeSafe가 Jev 학습에 썼다고 밝힌 RLCD(보정된 결정 학습, 비공개)와 이름만 같고, 학습은 하지 않아요.

In [ ]:
chart_colab14b(PRE["colab14b"])

**읽는 법**: 같은 T4, 같은 8건 정답(8/8)인데 시간은 1.4초~44초로 벌어져요. 판단 방식(초록)이 생성(주황)보다 빠른 건 두 엔진 모두 같지만, **엔진 차이(llama.cpp vs PyTorch)가 방식 차이보다 더 커요.** PyTorch 쪽은 T4 메모리에 맞추려고 질문을 2개씩 끊어(chunk=2) 매번 문맥을 다시 계산했고, 캐시를 한 번만 쓰는 llama.cpp 병렬 판단은 17배 빨랐어요.

### 6-3. 가볍게 직접 재 보기: 같은 상황 × 질문 8개
무거운 실험 대신, 방금 올린 4B 모델로 SemIf의 두 함수를 비교해요. 고객 문의 1건(장애 신고 + 로그 몇 줄)에 질문 8개(심각도 · 담당 팀 · 이탈 위험 · 환불 · 법무 검토 · 상태 페이지 · 전화 · 원인)를 던져요.
- **따로 8번**: `score()`를 8번 — 매번 상황부터 다시 읽어요.
- **앞부분 공유**: `score_shared()` 1번 — 상황을 한 번 읽고 캐시를 나눠 써요.

GPU 모드에서는 여기서 직접 재고(두 번 재서 빠른 쪽), precomputed 모드에서는 강사 Mac 기록을 보여 줘요.

In [ ]:
if MODE == "precomputed":
    SHARED, WHERE = FALLBACK["shared_demo"], "강사 Mac(M5) · MPS · bfloat16 기록"
else:
    try:
        t0 = time.time()
        SHARED = run_shared_demo(model, tokenizer, metadata)
        WHERE = f"방금 이 런타임 ({GPU_NAME} · {DTYPE}) · 셀 전체 {time.time() - t0:.0f}초"
    except Exception as e:   # 공유 캐시 경로가 이 환경에서 안 되면 Mac 기록으로 대신 보여 줘요
        print(f"공유 모드 실행 실패({type(e).__name__}: {str(e)[:120]}) → 강사 Mac 기록을 보여 줘요")
        SHARED, WHERE = FALLBACK["shared_demo"], "강사 Mac(M5) · MPS · bfloat16 기록"
chart_shared_demo(SHARED, WHERE)

**읽는 법**: 강사 Mac(MPS)에서는 따로 8번 10.95초 vs 공유 3.86초(2.8배)였고, 모델이 읽은 토큰은 2,473개 vs 751개였어요. CUDA에서는 SemIf가 뒷부분 8개를 **한 번에(batch)** 계산해서 차이가 달라질 수 있어요. 고른 답은 같아야 하고, 확률은 소수점 아래가 조금 다를 수 있어요.

### 6-4. 미리 잰 결과 ③: 강사 A4000에서 SemIf 따로 vs 공유 vs RLCD
6-3과 같은 비교를 강사 A4000(NF4 4비트)에서 필드 28개로 한 기록이에요. 모델을 한 번만 올리고 세 방식이 같은 가중치를 썼어요. 정답은 따로 만든 문의 8건으로 쟀어요.

In [ ]:
chart_semif_a4000(PRE["semif_a4000"])

**읽는 법**: 초록(공유)과 주황(따로)의 차이가 6-3에서 본 '상황을 한 번만 읽는' 효과예요. 회색 RLCD는 더 빠르지만 작은 1.5B에서 더 많이 틀렸어요. **속도는 구현이, 정확도는 모델과 질문 모양이** 크게 좌우해요.

**더 해 보기: 14B·llama.cpp를 직접 다시 재고 싶다면 (선택 · 무거움)**
- 노트북: 강사 자료 `parallel-decision-rlcd-20260922/colab-large-models/Colab_Local_16GB_Qwen14B.ipynb`
- 필요한 것: T4 16GB, 디스크 여유 약 38GiB, 다운로드 약 39GB(14B 원본 + GGUF), T4용 llama.cpp CUDA 빌드(`RUN_LLAMA_CPP=True`일 때만)
- 이 노트북의 6-2 그림은 그 노트북이 T4에서 남긴 결과 파일을 그대로 쓴 거예요.

## 7. 정확도와 보정: 빠른 것 ≠ 맞는 것 ≠ 믿을 수 있는 것

3강 실습 ②(API vs 로컬)와 같은 기록이에요. 2강 라우터 메시지 40건을 엔진만 바꿔 돌리고, **같은 정책 코드**로 행동(답장 · 작업 넘기기 · 되묻기 · 확인 · 차단)을 정해 채점했어요. 모델을 다시 돌리지 않아요.

In [ ]:
chart_api_vs_local(PRE["api_vs_local"])

**읽는 법**
- Jev API와 Qwen3.5-4B JSON 생성이 92.5%로 같지만, 지연은 0.25초 vs 4.2초예요.
- decider-2b는 A4000에서 0.21초로 가장 빠르지만 72.5%, OpenJev(SemIf 4B)는 67.5%예요. 로컬 판단 모델은 **공짜·빠름 대신 정확도를 조금 내줘요**.
- 같은 decider-2b라도 Mac에서는 1.9초예요. 장비가 속도를 크게 바꿔요.

### 7-1. 보정 맛보기: "0.95라고 했으면 정말 맞았나?"
1강에서 본 **보정(calibration)** 을 떠올려 봐요. 모델이 0.8이라고 한 경우를 모으면 실제로 80%쯤 맞아야 잘 보정된 거예요(신뢰도 그림 · ECE).

SemIf는 결과마다 `"probability_status": "conditional option score; uncalibrated as decision confidence"`라고 스스로 적어요. **보기끼리 비교한 점수**이고, '맞을 확률'로 맞춰진 값이 아니라는 뜻이에요. 40건 중 route 질문(잡담 · 작업 · 애매함 · 스팸) 하나만 골라, 고른 답의 확률과 정답 여부를 점으로 찍어 봐요.

In [ ]:
chart_calibration(PRE["api_vs_local"])

**읽는 법**
- 좋은 모습은 틀린 답(✕)이 왼쪽(낮은 확률)에 모이고, 평균 확신(세로선)이 정답률과 비슷한 거예요. Jev는 40건 중 1건만 틀렸고, 그때 확률도 0.53으로 낮았어요.
- OpenJev(SemIf 4B)는 정답률 77.5%인데 평균 확신이 0.85로 **조금 과신**하고, 틀린 9건 중 3건에 0.9 넘게 확신했어요. 이런 숫자에 '0.9 이상이면 자동 실행' 같은 기준을 걸면 사고가 나요.
- decider-2b는 정답률 77.5%에 평균 확신 0.77로 거의 같고, 틀린 답은 모두 0.74 아래였어요. 지도학습에 온도 보정까지 한 모델의 모습이에요.
- 40건은 신뢰도 그림을 그리기엔 적어요. 내 서비스에 쓰려면 내 데이터 수백 건으로 재고, 필요하면 온도(4장 ⑥)를 맞춰요. SemIf 저장소에도 작업별 온도 보정 기능이 있어요.

### 7-2. 같은 가중치, 출력 방식만 다르게: RLCR 생성 vs RLCD 읽기
1강에서 본 **RLCR**은 추론 끝에 확신도를 글로 쓰도록 학습한 방식이에요. 강사 A4000에서 RLCR로 학습된 7B 모델 하나를 두 방식으로 돌려 봤어요. ① 원래대로 추론·답·확신도를 글로 쓰기 ② 같은 모델에 6장의 RLCD 데모 코드로 보기 점수만 읽기. 이미 잰 기록이라 모델은 돌리지 않아요.

In [ ]:
chart_rlcr_rlcd(PRE["rlcr_vs_rlcd"])

**읽는 법**: 점수 읽기는 수십 배 빠르지만, 틀린 두 건에 0.96 · 0.83이라는 높은 확률을 붙였어요. 같은 문의에서 글로 쓴 RLCR은 맞혔고 확신도는 0.7 · 0.8이었어요. 빠른 판단을 쓸 때는 **확률을 그대로 믿지 말고 내 데이터로 보정을 재는 일**이 따라와야 해요. TypeSafe가 Jev를 '보정된 결정'이 되도록 따로 학습했다고 강조하는 이유이기도 해요.

## 8. 내 질문 만들어 보기

`state`, `question`, `options`만 바꿔서 실행해 보세요.
- 보기는 2~16개, `id`는 서로 달라야 해요. 보기 id는 모델에 보이지 않아요 — **설명(`description`)이 판단 재료**예요.
- 애매할 때 빠져나갈 보기(예: '판단 불가 — 사람에게 묻기')를 하나 넣어 두면 좋아요(`classify-fail`의 `unknown`처럼).
- 한국어도 돼요. 아래 예시는 한국어 메일이에요.
- precomputed 모드에서는 예시 그대로일 때만 미리 잰 결과를 보여 줘요. 새 질문은 GPU가 있어야 판정할 수 있어요.

In [ ]:
my_row = {
    "id": "my-test",
    "state": "고객 메일: '지난주에 주문한 노트북이 아직 안 왔어요. 그냥 취소하고 환불받고 싶습니다.'",
    "question": "이 메일을 어느 팀으로 보내야 하나요?",
    "options": [
        {"id": "shipping", "description": "배송 조회·지연 담당"},
        {"id": "refund", "description": "취소·환불 담당"},
        {"id": "sales", "description": "구매 상담 담당"},
    ],
}

try:
    r = decide(my_row)
    for k, p in zip(r["option_ids"], r["probabilities"]):
        print(f"{k:<10} {p:.3f}  " + "█" * round(p * 30))
    print(f"\n{r['input_tokens']}토큰 · 한 번 통과 {r['forward_seconds'] * 1000:.0f}ms")
except RuntimeError as e:
    print(e)

(선택) 아래 셀은 내 질문도 4장의 7단계 위젯으로 열어 봐요. 보기 순서를 바꾼 판정까지 돌려서, 보기가 3개면 판정을 7번 더 해요(T4에서 수 초~10초).

In [ ]:
#@title (선택) 내 질문도 7단계 위젯으로 보기 — 보기 순서 바꾸기까지 돌려요
if MODE == "precomputed":
    if my_row == FALLBACK["results"]["my-test"]["row"]:
        show_explainer([FALLBACK["results"]["my-test"]], "숫자: 강사 Mac(M5) · MPS · bfloat16 미리 잰 값", "Mac MPS · bfloat16", default_case="my-test")
    else:
        print("새 질문은 GPU 런타임에서 볼 수 있어요.")
else:
    mine = explain_case(model, tokenizer, my_row, metadata)
    show_explainer([mine], f"숫자: 방금 이 런타임에서 잰 값 ({GPU_NAME} · {DTYPE})", f"{GPU_NAME} · {DTYPE}", default_case=my_row["id"])

**해 볼 것**: ① 보기 순서를 바꿔 다시 돌려 보기 ② `refund` 설명을 모호하게 바꿔 보기('고객 담당') ③ 빠져나갈 보기 추가하기. 확률이 어떻게 흔들리는지 보면, **보기 설명이 곧 판단 기준**이라는 게 느껴져요.

## 9. (선택) TypeSafe Jev API와 나란히

API 키가 있으면 같은 질문을 진짜 Jev에도 보내 비교해요. 키가 없으면 이 셀은 안내만 하고 넘어가요.
1. 왼쪽 🔑 **보안 비밀(Secrets)** 에 `TYPESAFE_API_KEY`를 추가하고 '노트북 액세스'를 켜요. **키를 코드에 붙여 넣지 마세요.** 셀은 키 값을 화면에 찍지 않아요.
2. 비용: 입력 100만 토큰당 $0.042이에요. 2강 라우터 40건 전체가 약 $0.0014였어요.
3. TypeSafe는 2026-09-22부터 신규 가입을 잠시 멈췄어요(기존 계정은 사용 가능).
4. 파이썬 기본 User-Agent(`Python-urllib`)로 부르면 Cloudflare가 **HTTP 403(error 1010)** 으로 막아요. 1강에서 겪은 그 문제라 헤더에 User-Agent를 넣었어요.

In [ ]:
import json, urllib.request, urllib.error

# 2강 코드와 같은 규칙: 주소는 TYPESAFE_BASE_URL(기본 https://api.typesafe.ai)로 바꿀 수 있어요
BASE_URL = os.environ.get("TYPESAFE_BASE_URL", "https://api.typesafe.ai").rstrip("/")

def get_key():
    try:
        from google.colab import userdata
        return userdata.get("TYPESAFE_API_KEY")
    except Exception:          # Colab 밖이거나, 보안 비밀이 없거나, 액세스를 안 켠 경우
        return os.environ.get("TYPESAFE_API_KEY")

def jev_api(row, key, model="jev-latest"):
    body = {"model": model, "state": row["state"],
            "questions": {"q": {"type": "choice", "instructions": row["question"],
                                "criteria": {o["id"]: o["description"] for o in row["options"]}}}}
    req = urllib.request.Request(f"{BASE_URL}/v1/systemone", data=json.dumps(body).encode("utf-8"),
                                 headers={"Authorization": "Bearer " + key, "Content-Type": "application/json",
                                          "User-Agent": "jev-lecture-colab/1.0"})   # 없으면 403 (error 1010)
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=20) as resp:
        out = json.load(resp)
    return out, (time.perf_counter() - t0) * 1000

KEY = get_key()   # 값은 출력하지 않아요
if not KEY:
    print("TYPESAFE_API_KEY가 없어서 건너뛰어요. (왼쪽 🔑 보안 비밀에 추가하고 노트북 액세스를 켠 뒤 다시 실행)")
else:
    for row in ROWS + [my_row]:
        try:
            out, ms = jev_api(row, KEY)
        except urllib.error.HTTPError as e:
            detail = e.read().decode("utf-8", "replace")[:160]
            print(f"[{row['id']}] HTTP {e.code}: {detail}")
            if e.code in (401, 403):
                print("   → 키가 틀렸거나 요청이 막혔어요. 키를 확인하고, User-Agent 헤더를 지우지 마세요.")
                break
            continue
        except (urllib.error.URLError, TimeoutError) as e:
            print(f"[{row['id']}] 연결 실패: {e}")
            break
        api_probs = out["answers"]["q"]["probabilities"]
        try:
            local = dict(zip(*[decide(row)[k] for k in ("option_ids", "probabilities")]))
        except RuntimeError:
            local = {}
        tokens = out.get("usage", {}).get("input_tokens", 0)
        print(f"[{row['id']}] Jev {out.get('model')} · {ms:.0f}ms · 입력 {tokens}토큰(≈ ${tokens * 0.042 / 1e6:.6f})")
        for o in row["options"]:
            k = o["id"]
            loc = f"{local[k]:.3f}" if k in local else "  -  "
            print(f"   {k:<10} Jev {api_probs.get(k, 0):.3f}   SemIf {loc}")

**읽는 법**: 두 엔진이 같은 보기를 고르는지, 확률이 얼마나 다른지 보세요. Jev는 보정까지 학습했다고 밝힌 모델(RLCD, 방법은 비공개)이라 애매한 질문에서 숫자가 다르게 나올 수 있어요. 강의에서는 `route-vague`에 Jev가 main 0.84를 줬어요.

## 10. (선택 · 80GB GPU 전용) Jev-Omni 살펴보기

[akhilaaa3/Jev-Omni](https://huggingface.co/akhilaaa3/Jev-Omni)는 2026-09에 공개된 **독립 오픈 모델**이에요(TypeSafe와 무관, Apache-2.0).
- Gemma 4 12B IT 위에 **3840 → 256 판단 머리(head)** 를 붙였어요. SemIf처럼 글자 점수를 읽는 대신, 마지막 은닉 상태에서 보기 번호(최대 256개) 중 하나를 고르는 **분류기를 학습**했어요.
- `predict(state, question, options)` → 보기별 확률. 글·이미지·오디오·영상을 받아요.
- **무료 T4에서는 안 돼요.** 공식 로더가 저장소 전체(약 72GB)와 Gemma 4 12B(약 24GB)를 받고, FP32 본체(약 48GB)를 GPU에 통째로 올려요. A100 80GB나 H100이 필요해요(T4 · L4 · A100 40GB로는 그대로 안 돼요).

그래서 여기서는 **돌리지 않고 읽기만** 해요: ① `hf download --dry-run`으로 받을 양 보기 ② 파일 크기 정리(메타데이터만) ③ 모델이 스스로 올린 검증 기록 ④ 공식 빠른 시작 코드(80GB GPU가 아니면 실행을 거절해요).

In [ ]:
# 받기 전에 크기부터: --dry-run은 목록만 보고 아무것도 받지 않아요
if shutil.which("hf"):
    !hf download akhilaaa3/Jev-Omni --dry-run 2>/dev/null | grep "dry-run"
    !hf download google/gemma-4-12B-it --dry-run 2>/dev/null | grep "dry-run"
else:
    print("hf 명령이 없어요 (pip install huggingface-hub)")

받을 양을 폴더별로 묶어 봐요. `HfApi().list_repo_tree()`로 **파일 이름과 크기(메타데이터)만** 읽어요. 인터넷이 안 되면 2026-09-24에 조회한 값을 보여 줘요.

In [ ]:
OMNI = dict(PRE["jev_omni"])   # 미리 조회한 값(2026-09-24)으로 시작해서, 인터넷이 되면 지금 값으로 바꿔요
try:
    from huggingface_hub import HfApi
    api = HfApi(token=False)
    groups = {}
    for f in api.list_repo_tree("akhilaaa3/Jev-Omni", recursive=True):
        if getattr(f, "size", None):
            key = f.path.split("/")[0] if "/" in f.path else ("head.pt" if f.path == "head.pt" else "기타 작은 파일")
            groups[key] = groups.get(key, 0) + f.size
    OMNI.update(files_bytes=groups, repo_total_bytes=sum(groups.values()),
                repo_sha=api.model_info("akhilaaa3/Jev-Omni").sha, fetched_at=time.strftime("%Y-%m-%d"))
except Exception as e:
    print(f"허브에 닿지 않아 미리 조회한 목록을 보여 줘요 ({type(e).__name__})")
omni_files_table(OMNI)

**읽는 법**: 2장의 `hf download`처럼 **받기 전에 크기부터** 보는 습관이에요. `backbone/`(FP32, 약 48GB)과 `unified/`(BF16, 약 24GB)가 같은 모델을 두 형식으로 담고 있어서 저장소가 커요. 공식 로더는 둘 다 받아요.

다음은 모델 저장소가 스스로 올린 검증 기록(`unified/verification_unified.json`)의 확률 4건이에요. 우리가 모델을 돌린 게 아니라, 올라온 숫자를 그림으로 읽어요. 1강의 보정 질문('0.9라고 하면 열에 아홉은 맞나?')을 떠올리며 보세요.

In [ ]:
chart_omni_verification(PRE["jev_omni"])

**읽는 법**
- 정답이 분명한 질문(10시 회의인데 지금 9시 → 시작했나? → No 0.9999)은 확신 있게 맞혀요.
- **정답이 없는 질문**이 재밌어요. 공정한 주사위라면 모든 눈이 1/6(0.167)이어야 하는데, '6'에 0.46, '3'에 0.04를 줘요. 구슬 4개도 0.25씩이어야 하는데 파랑 0.41 · 초록 0.38로 쏠려요.
- README의 ECE 0.040은 정답이 있는 벤치마크(DecisionBench Medium) 기준이에요. '모른다'를 고르게 표현하는 능력은 따로 봐야 해요. 보정은 결국 **내 데이터로** 재야 해요.

마지막 셀은 모델 카드의 **빠른 시작 코드 그대로**예요. GPU 메모리 75GB · 디스크 여유 110GB가 안 되면 실행하지 않고, 조건이 맞아도 `RUN_JEV_OMNI = True`로 바꿔야 돌아요(약 96GB를 받아요). 무료 T4에서는 '건너뛰어요'가 나오면 정상이에요.

In [ ]:
#@title (80GB GPU 전용) Jev-Omni 공식 빠른 시작 — 조건이 안 맞으면 실행하지 않아요
RUN_JEV_OMNI = False     # 80GB GPU에서 직접 돌려 보려면 True (약 96GB를 받아요)
NEED_GPU_GB, NEED_DISK_GB = 75, 110

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if MODE == "cuda" else 0.0
disk_gb = shutil.disk_usage(os.path.expanduser("~")).free / 1e9
if gpu_gb < NEED_GPU_GB:
    print(f"건너뛰어요: GPU 메모리 {gpu_gb:.0f}GB < {NEED_GPU_GB}GB. Jev-Omni는 A100 80GB · H100 같은 GPU가 필요해요.")
elif disk_gb < NEED_DISK_GB:
    print(f"건너뛰어요: 디스크 여유 {disk_gb:.0f}GB < {NEED_DISK_GB}GB (저장소 약 72GB + Gemma 4 12B 약 24GB)")
elif not RUN_JEV_OMNI:
    print(f"GPU {gpu_gb:.0f}GB · 디스크 {disk_gb:.0f}GB — 돌릴 수 있어요. RUN_JEV_OMNI = True로 바꾸고 다시 실행하세요.")
else:
    # ↓ 모델 카드(README)의 Quick start 그대로예요 (Apache-2.0)
    !pip install -q -r https://huggingface.co/akhilaaa3/Jev-Omni/resolve/main/requirements.txt
    from huggingface_hub import snapshot_download
    path = snapshot_download("akhilaaa3/Jev-Omni")
    sys.path.insert(0, path)
    from jev_omni import load_jev_omni
    classifier = load_jev_omni()
    result = classifier.predict(
        state="The meeting starts at 10 AM. It is now 9 AM.",
        question="Has the meeting started?",
        options=["Yes", "No"],
    )
    print(result)

## 11. 정리

- **Jev 닮은 판단 = 한 번 통과 + 보기 글자 점수 읽기.** 글을 쓰지 않으니 빠르고, 보기 밖 답이 나오지 않아요. (1 · 4장)
- **빠른 이유는 반복을 줄여서예요.** 답을 쓰는 반복(토큰마다 통과)과 상황을 다시 읽는 반복(질문마다 문맥 계산)을 없앴어요. 같은 장비라도 엔진과 캐시 구현이 속도를 크게 바꿔요. (6장)
- **빠른 것 ≠ 맞는 것 ≠ 믿을 수 있는 것.** 로컬 판단 모델은 정확도를 조금 내주고, 학습하지 않은 SemIf의 숫자는 보기끼리 비교한 점수라 보정이 필요해요. 보기 순서만 바꿔도 흔들려요. (4 · 7장)
- **행동은 코드가 정해요.** 확률에 기준을 걸고, 애매하면 사람에게 묻는 길을 코드로 만들어 두세요. 기준값은 내 데이터로 정해요. (4장 ⑦)

**더 보기**
- SemIf: https://github.com/TheoLeeCJ/SemIf (이 노트북은 커밋 `1f2dea3`으로 고정)
- 3강 슬라이드 「원리 ①~③」 · 프롬프트 ③④(API vs 로컬) · `03_유사프로젝트/evidence/api-vs-local/`
- 1강: 보정 · ECE · 신뢰도 그림
- Jev-Omni: https://huggingface.co/akhilaaa3/Jev-Omni
- 이 노트북의 영상 · 위젯 · 미리 잰 숫자 원본: `03_유사프로젝트/colab/assets/` (`build_notebook.py`로 다시 만들어요)